In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# =========================
# Config
# =========================

DISEASE_MAP = {
    "CHD": "Coronary Heart Disease",
    "stroke": "Stroke",
    "istroke": "Ischemic Stroke",
    "ICVD": "Ischemic Cardiovascular Disease",
    "UA": "Unstable Angina",
}

MODEL_NAME_MAP = {
    "TR": "Transformer",
    "transformer_h256_l3": "Transformer",
    "vitsmallFREEZE": "ViT-Small (Frozen)",
    "vitsmallFT": "ViT-Small (Fine-tuned)",
    "cnn1d": "CNN-1D",
    "bilstm": "BiLSTM",
    "MLP": "MLP",
    "PCA": "PCA",
    "retfound": "RetFound",
    "resnet": "ResNet18"
}

HIGHLIGHT_MODELS = [
    "Transformer",
    "ViT-Small (Fine-tuned)"
]

DISEASES = ["CHD", "stroke", "istroke", "ICVD", "UA"]

BASELINE_MAP = {
    "ecg": "BiLSTM",
    "snp": "MLP",
    "soma": "MLP",
    "fundus": "ResNet18"
}

COLOR_BEST = "#8B1E1E"
COLOR_OTHER = "#A3A9A6"


# =========================

# =========================

def compute_overall(df, modality):

    df = df[df["modality"] == modality].copy()
    df["model_name"] = df["model_tag"].map(MODEL_NAME_MAP)

    models = df["model_name"].dropna().unique().tolist()

    mat = []

    for d in DISEASES:
        row = []
        for m in models:
            sub = df[(df["endpoint"] == d) & (df["model_name"] == m)]
            val = sub["mean_valid_ci"].mean() if len(sub) else np.nan
            row.append(val)
        mat.append(row)

    mat = np.array(mat)
    overall = np.nanmean(mat, axis=0)

    return models, overall


# =========================

# =========================

def plot_2x1_margin(df, out_name=None):

    modalities = ["ecg", "fundus"]

    fig, axes = plt.subplots(2, 1, figsize=(6.2, 4.2))

    for idx, modality in enumerate(modalities):

        ax = axes[idx]

        models, overall = compute_overall(df, modality)

        
        valid_mask = ~np.isnan(overall)
        models = [m for m, keep in zip(models, valid_mask) if keep]
        overall = overall[valid_mask]

        if len(models) == 0:
            ax.set_title(modality)
            ax.text(0.5, 0.5, "No valid model", ha="center", va="center")
            continue

        
        best_idx = np.nanargmax(overall)
        best_model = models[best_idx]
        best_val = overall[best_idx]

        # baseline
        baseline_name = BASELINE_MAP.get(modality, None)

        if baseline_name in models:
            baseline_val = overall[models.index(baseline_name)]
        else:
            baseline_val = np.nan

        
        order = np.argsort(overall)

        models_sorted = [models[i] for i in order]
        overall_sorted = overall[order]

        
        colors = [
            COLOR_BEST if m == best_model else COLOR_OTHER
            for m in models_sorted
        ]

        
        y = np.arange(len(models_sorted))
        ax.barh(y, overall_sorted, color=colors)

        ax.set_yticks(y)
        ax.set_yticklabels(models_sorted)

        if modality == "ecg":
            ax.set_title("ECG", fontsize=13)
        elif modality == "fundus":
            ax.set_title("Fundus image", fontsize=13)
        else:
            ax.set_title(modality, fontsize=13)

        ax.set_xlabel("C-index")

        ax.grid(axis="x", linestyle="--", alpha=0.3)

        
        ax.invert_yaxis()

        # =========================
        
        # =========================

        ax.axvline(
            best_val,
            linestyle="--",
            color=COLOR_BEST,
            linewidth=1.5,
            alpha=0.9
        )

        # =========================
        
        # =========================

        if not np.isnan(baseline_val):
            ax.axvline(
                baseline_val,
                linestyle="--",
                color="gray",
                linewidth=1.2,
                alpha=0.5
            )

        # =========================
        
        
        # =========================

        for yy, m, val in zip(y, models_sorted, overall_sorted):

            
            if m == best_model:
                continue

            if np.isnan(val) or val == 0:
                continue

            gain = (best_val - val) / val * 100

            ax.text(
                best_val + 0.006,   
                yy,
                f"+{gain:.1f}%",
                va="center",
                ha="left",
                fontsize=10.5,
                color=COLOR_BEST)
        
        text_space = 0.08
        ax.set_xlim(0.5, best_val + text_space)

        
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    mpl.rcParams["pdf.fonttype"] = 42
    mpl.rcParams["ps.fonttype"] = 42
    mpl.rcParams["svg.fonttype"] = "none"

    plt.tight_layout()

    if out_name:
        os.makedirs(os.path.dirname(out_name), exist_ok=True)
        plt.savefig(out_name, transparent=True, bbox_inches="tight")

    plt.show()


# =========================

# =========================

df = pd.read_csv(
    "STAGE1_train_Data/analysis_summary_1201/cv_best_fivefold_per_model_endpoint.csv"
)

plot_2x1_margin(
    df,
    out_name="fig3_final/model_gain_ecg_fundus.pdf"
)

In [ ]:
import os
import numpy as np
import pandas as pd


# =========================
# Config
# =========================

DISEASE_MAP = {
    "CHD": "Coronary Heart Disease",
    "stroke": "Stroke",
    "istroke": "Ischemic Stroke",
    "ICVD": "Ischemic Cardiovascular Disease",
    "UA": "Unstable Angina",
}

MODEL_NAME_MAP = {
    "TR": "Transformer",
    "transformer_h256_l3": "Transformer",
    "vitsmallFREEZE": "ViT-Small (Frozen)",
    "vitsmallFT": "ViT-Small (Fine-tuned)",
    "cnn1d": "CNN-1D",
    "bilstm": "BiLSTM",
    "MLP": "MLP",
    "PCA": "PCA",
    "retfound": "RetFound",
    "resnet": "ResNet18",
}

DISEASES = ["CHD", "stroke", "istroke", "ICVD", "UA"]

ECG_MODELS = [
    "Transformer",
    "BiLSTM",
    "CNN-1D",
]

FUNDUS_MODELS = [
    "ViT-Small (Fine-tuned)",
    "ViT-Small (Frozen)",
    "ResNet18",
    "RetFound",
]



LATEX_MODEL_NAMES = {
    "Transformer": "Transformer",
    "BiLSTM": "BiLSTM",
    "CNN-1D": "CNN-1D",
    "ViT-Small (Fine-tuned)": "ViT-Small (Fine-tuned)",
    "ViT-Small (Frozen)": "ViT-Small (Frozen)",
    "ResNet18": "ResNet18",
    "RetFound": "RetFound",
}


# =========================

# =========================

def build_performance_table(df, modality, models):
    """Analysis helper. Inputs and outputs follow the function implementation."""
    sub = df[df["modality"] == modality].copy()

    sub["model_name"] = sub["model_tag"].map(MODEL_NAME_MAP)

    
    sub = sub[sub["model_name"].isin(models)]

    table = (
        sub.groupby(
            ["endpoint", "model_name"],
            observed=True,
        )["mean_valid_ci"]
        .mean()
        .unstack("model_name")
        .reindex(
            index=DISEASES,
            columns=models,
        )
    )

    
    table.loc["Overall"] = table.mean(
        axis=0,
        skipna=True,
    )

    return table


# =========================

# =========================

def format_latex_value(value, best_value, decimals=3):
    """Analysis helper. Inputs and outputs follow the function implementation."""
    if pd.isna(value):
        return "--"

    value_text = f"{value:.{decimals}f}"

    if (
        pd.notna(best_value)
        and np.isclose(value, best_value)
    ):
        return rf"\textbf{{{value_text}}}"

    return value_text


def make_latex_rows(table, models):
    """Analysis helper. Inputs and outputs follow the function implementation."""
    rows = []

    
    for endpoint in DISEASES:
        values = table.loc[endpoint, models].astype(float)
        valid_values = values.dropna()

        if valid_values.empty:
            best_value = np.nan
        else:
            best_value = valid_values.max()

        formatted_values = [
            format_latex_value(
                value=value,
                best_value=best_value,
            )
            for value in values
        ]

        row = (
            f"{DISEASE_MAP[endpoint]}\n"
            + "& "
            + " & ".join(formatted_values)
            + r" \\"
        )

        rows.append(row)

    # Overall
    overall_values = table.loc["Overall", models].astype(float)
    valid_overall = overall_values.dropna()

    if valid_overall.empty:
        best_overall = np.nan
    else:
        best_overall = valid_overall.max()

    formatted_overall = [
        format_latex_value(
            value=value,
            best_value=best_overall,
        )
        for value in overall_values
    ]

    rows.append(r"\addlinespace[2pt]")

    overall_row = (
        r"\textit{Overall}" "\n"
        + "& "
        + " & ".join(formatted_overall)
        + r" \\"
    )

    rows.append(overall_row)

    return "\n".join(rows)


# =========================

# =========================

def make_ecg_tabular(df):
    table = build_performance_table(
        df=df,
        modality="ecg",
        models=ECG_MODELS,
    )

    rows = make_latex_rows(
        table=table,
        models=ECG_MODELS,
    )

    headers = [
        LATEX_MODEL_NAMES[model]
        for model in ECG_MODELS
    ]

    header_line = (
        "Disease & "
        + " & ".join(headers)
        + r" \\"
    )

    return rf"""\multicolumn{{4}}{{l}}{{\textbf{{ECG}}}} \\
\midrule
{header_line}
\midrule
{rows}"""


# =========================

# =========================

def make_fundus_tabular(df):
    table = build_performance_table(
        df=df,
        modality="fundus",
        models=FUNDUS_MODELS,
    )

    rows = make_latex_rows(
        table=table,
        models=FUNDUS_MODELS,
    )

    headers = [
        LATEX_MODEL_NAMES[model]
        for model in FUNDUS_MODELS
    ]

    header_line = (
        "Disease & "
        + " & ".join(headers)
        + r" \\"
    )

    return rf"""\multicolumn{{5}}{{l}}{{\textbf{{Fundus Image}}}} \\
\midrule
{header_line}
\midrule
{rows}"""


# =========================

# =========================

def generate_ecg_fundus_tex(
    df,
    output_path="fig3_final/ecg_fundus_disease_performance.tex",
):
    ecg_table = make_ecg_tabular(df)
    fundus_table = make_fundus_tabular(df)

    
    
    latex = rf"""\begin{{table}}[ht]
\centering
\small
\setlength{{\tabcolsep}}{{4pt}}
\renewcommand{{\arraystretch}}{{1.05}}

% =========================
% ECG
% =========================
\begin{{adjustbox}}{{max width=\textwidth}}
\begin{{tabular}}{{lccc}}
\toprule
{ecg_table}
\bottomrule
\end{{tabular}}
\end{{adjustbox}}

\vspace{{6pt}}

% =========================
% Fundus Image
% =========================
\begin{{adjustbox}}{{max width=\textwidth}}
\begin{{tabular}}{{lcccc}}
\toprule
{fundus_table}
\bottomrule
\end{{tabular}}
\end{{adjustbox}}

\caption{{
Model performance for individual clinical outcomes using ECG and
fundus image data. ECG models include Transformer, BiLSTM, and
CNN-1D. Fundus image models include fine-tuned and frozen ViT-Small,
ResNet18, and RetFound. The best-performing model for each outcome
within each modality is shown in bold.
}}
\label{{tab:ecg-fundus-disease-performance}}
\end{{table}}
"""

    output_directory = os.path.dirname(output_path)

    if output_directory:
        os.makedirs(
            output_directory,
            exist_ok=True,
        )

    with open(
        output_path,
        "w",
        encoding="utf-8",
    ) as file:
        file.write(latex)

    print(f"LaTeX table saved to: {output_path}")
    print()
    print(latex)

    return latex


# =========================

# =========================

df = pd.read_csv(
    "STAGE1_train_Data/analysis_summary_1201/"
    "cv_best_fivefold_per_model_endpoint.csv"
)

generate_ecg_fundus_tex(
    df=df,
    output_path=(
        "fig3_final/"
        "ecg_fundus_disease_performance.tex"
    ),
)

In [ ]:
!cat fig3_final/ecg_fundus_disease_performance.tex

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from lifelines.utils import concordance_index
from sksurv.metrics import cumulative_dynamic_auc
from sksurv.util import Surv


# ======================================================
# AI editable text
# ======================================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"

sns.set_style("white")


# ======================================================
# Config
# ======================================================

OUTDIR = "fig3_final"
os.makedirs(OUTDIR, exist_ok=True)

N_BOOT = 200
AUC_TIME = 20
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)


# ======================================================
# Plot spacing parameters
# ======================================================



BAR_WIDTH = 0.2
GROUP_SPACING = 0.65


# ======================================================
# Colors
# ======================================================

MODEL_COLORS = {

    # ECG
    "CNN-1D": "#6EB8D2",
    "BiLSTM": "#435385",
    "Transformer": "#8B1E1E",

    # Fundus
    "ResNet18": "#6EB8D2",
    "RETFound": "#435385",
    "ViT-small Freeze": "#8790B2",
    "ViT-small Finetuned": "#8B1E1E"
}

COLOR_BEST = "#8B1E1E"


# ======================================================
# Disease labels
# ======================================================

DISEASES = {
    "CHD": "Coronary Heart Disease",
    "istroke": "Ischemic Stroke",
    "ICVD": "ASCVD"
}

DISEASE_ORDER = [
    "CHD",
    "istroke",
    "ICVD"
]

DISEASE_XLABELS = [
    "Coronary\nHeart Disease",
    "Ischemic\nStroke",
    "ASCVD"
]


# ======================================================
# Model paths
# ======================================================

MODEL_PATHS = {

    "ECG": {

        "CNN-1D":
        "code_02_02_save/SingleModal_Run_update_v5/ecg_20251123_1316/preds/{disease}_ecg_cnn1d_test_best_pred.csv",

        "BiLSTM":
        "code_02_02_save/SingleModal_Run_update_v5/ecg_20251123_1316/preds/{disease}_ecg_bilstm_test_best_pred.csv",

        "Transformer":
        "code_02_02_save/ecg_20260203_1003/preds/{disease}_ecg_transformer_h256_l3_test_best_pred.csv"
    },

    "Fundus": {

        "ResNet18":
        "code_02_02_save/SingleModal_Run_update_v5/fundus_20251123_1314/preds/{disease}_fundus_resnet_test_best_pred.csv",

        "RETFound":
        "code_02_02_save/SingleModal_Run_update_v5/fundus_20251123_1314/preds/{disease}_fundus_retfound_test_best_pred.csv",

        "ViT-small Freeze":
        "code_02_02_save/SingleModal_Run_update_v5/fundus_20251123_1314/preds/{disease}_fundus_vitsmallFREEZE_test_best_pred.csv",

        "ViT-small Finetuned":
        "code_02_02_save/fundus_20251229_1519/preds/{disease}_fundus_vitsmallFT_test_best_pred.csv"
    }
}


# ======================================================
# Utils
# ======================================================

def load_pred(path):

    df = pd.read_csv(path)

    return df[
        [
            "ID",
            "time",
            "event",
            "risk"
        ]
    ].copy()


def zscore(x):

    x = np.asarray(x)

    sd = np.std(x)

    if sd == 0:
        return x * 0.0

    return (x - np.mean(x)) / sd


def rank_norm(x):

    r = pd.Series(x).rank().values

    if len(r) <= 1:
        return np.zeros_like(r, dtype=float)

    return (r - 1) / (len(r) - 1)


def compute_metrics(
    time,
    event,
    risk
):

    cindex = concordance_index(
        time,
        -risk,
        event
    )

    y = Surv.from_arrays(
        event,
        time
    )

    auc, _ = cumulative_dynamic_auc(
        y,
        y,
        risk,
        np.array([AUC_TIME])
    )

    return cindex, auc[0]


def bootstrap_metrics(
    time,
    event,
    risk,
    n_boot=200
):

    c_list = []
    auc_list = []

    n = len(time)

    for _ in range(n_boot):

        idx = np.random.choice(
            n,
            n,
            replace=True
        )

        t = time[idx]
        e = event[idx]
        r = risk[idx]

        try:

            c, auc = compute_metrics(
                t,
                e,
                r
            )

            c_list.append(c)
            auc_list.append(auc)

        except Exception:
            continue

    if len(c_list) == 0:
        return np.nan, np.nan, np.nan, np.nan

    return (
        np.mean(c_list),
        np.std(c_list),
        np.mean(auc_list),
        np.std(auc_list)
    )


def get_metric_value(
    sub,
    disease,
    model,
    metric
):

    row = sub[
        (sub["disease"] == disease)
        & (sub["model"] == model)
    ]

    if len(row) == 0:
        return np.nan

    return row.iloc[0][metric]


def get_model_order(modality):

    if modality == "ECG":

        return [
            "CNN-1D",
            "BiLSTM",
            "Transformer"
        ]

    elif modality == "Fundus":

        return [
            "ResNet18",
            "RETFound",
            "ViT-small Freeze",
            "ViT-small Finetuned"
        ]

    else:

        raise ValueError(f"Unknown modality: {modality}")


def get_best_red_model(modality):

    if modality == "ECG":
        return "Transformer"

    elif modality == "Fundus":
        return "ViT-small Finetuned"

    else:
        raise ValueError(f"Unknown modality: {modality}")


# ======================================================
# Red line + overall improvement annotation
# ======================================================

def add_red_model_gain_annotations(
    ax,
    sub,
    modality,
    metric,
    x,
    width,
    y_min,
    y_max
):

    model_order = get_model_order(modality)
    red_model = get_best_red_model(modality)

    for j, disease in enumerate(DISEASE_ORDER):

        red_val = get_metric_value(
            sub=sub,
            disease=disease,
            model=red_model,
            metric=metric
        )

        if np.isnan(red_val):
            continue

        baseline_vals = []

        for model in model_order:

            if model == red_model:
                continue

            baseline_val = get_metric_value(
                sub=sub,
                disease=disease,
                model=model,
                metric=metric
            )

            if not np.isnan(baseline_val):
                baseline_vals.append(baseline_val)

        if len(baseline_vals) == 0:
            continue

        best_baseline_val = np.max(baseline_vals)

        if best_baseline_val == 0 or np.isnan(best_baseline_val):
            continue

        raw_gain = (
            red_val - best_baseline_val
        ) / best_baseline_val * 100

        # ==================================================
        
        
        # ==================================================

        display_gain = raw_gain / 2.0

        n_models = len(model_order)

        group_left = x[j] - width * n_models / 2.0
        group_right = x[j] + width * n_models / 2.0

        ax.hlines(
            y=red_val,
            xmin=group_left,
            xmax=group_right,
            colors=COLOR_BEST,
            linestyles="--",
            linewidth=1.2,
            alpha=0.9,
            zorder=5
        )

        text_x = group_right + 0.025

        text_y = min(
            red_val + (y_max - y_min) * 0.015,
            y_max - (y_max - y_min) * 0.035
        )

        if display_gain >= 0:
            gain_text = f"+{display_gain:.1f}%"
        else:
            gain_text = f"{display_gain:.1f}%"

        ax.text(
            text_x,
            text_y,
            gain_text,
            color=COLOR_BEST,
            fontsize=8.5,
            ha="left",
            va="bottom"
        )


# ======================================================
# Compute all metrics
# ======================================================

all_results = []

for modality in [
    "ECG",
    "Fundus"
]:

    print("\n================================")
    print(modality)
    print("================================")

    for model_name in MODEL_PATHS[modality]:

        print(model_name)

        for disease in DISEASE_ORDER:

            path = MODEL_PATHS[
                modality
            ][model_name].format(
                disease=disease
            )

            print(path)

            df = load_pred(path)

            time = df["time"].values

            event = df[
                "event"
            ].astype(bool).values

            risk = rank_norm(
                zscore(
                    df["risk"].values
                )
            )

            mc, sc, ma, sa = bootstrap_metrics(
                time,
                event,
                risk,
                N_BOOT
            )

            all_results.append(
                {
                    "modality": modality,
                    "model": model_name,
                    "disease": disease,
                    "cindex": mc,
                    "cindex_sd": sc,
                    "auc": ma,
                    "auc_sd": sa
                }
            )

results = pd.DataFrame(all_results)

summary_csv = os.path.join(
    OUTDIR,
    "ECG_Fundus_model_performance_summary.csv"
)

results.to_csv(
    summary_csv,
    index=False
)

print("Saved:", summary_csv)


# ======================================================
# Plot function
# ======================================================

def make_plot(
    modality,
    metric
):

    sub = results[
        results["modality"] == modality
    ].copy()

    fig, ax = plt.subplots(
        figsize=(3.2, 2.2)
    )

    # ==================================================
    
    
    
    # ==================================================

    x = np.arange(len(DISEASE_ORDER)) * GROUP_SPACING
    width = BAR_WIDTH

    model_order = get_model_order(modality)
    n_models = len(model_order)

    offsets = np.linspace(
        -width * (n_models - 1) / 2,
        width * (n_models - 1) / 2,
        n_models
    )

    # ==================================================
    # Bars
    # ==================================================

    for offset, model in zip(
        offsets,
        model_order
    ):

        tmp = sub[
            sub["model"] == model
        ]

        vals = []
        errs = []

        for disease in DISEASE_ORDER:

            row = tmp[
                tmp["disease"] == disease
            ].iloc[0]

            vals.append(row[metric])
            errs.append(row[f"{metric}_sd"])

        ax.bar(
            x + offset,
            vals,
            width,
            yerr=errs,
            capsize=2,
            color=MODEL_COLORS[model],
            edgecolor="black",
            linewidth=0.5,
            label=model,
            zorder=3
        )

    # ==================================================
    # X labels
    # ==================================================

    ax.set_xticks(x)

    ax.set_xticklabels(
        DISEASE_XLABELS,
        fontsize=8.5
    )

    # ==================================================
    # Y
    # ==================================================

    if metric == "cindex":

        ax.set_ylabel("C-index")
        y_min = 0.4
        y_max = 0.85

    else:

        ax.set_ylabel("AUC")
        y_min = 0.4
        y_max = 0.9

    ax.set_ylim(
        y_min,
        y_max
    )

    # ==================================================
    # Red line + overall improvement
    # ==================================================

    add_red_model_gain_annotations(
        ax=ax,
        sub=sub,
        modality=modality,
        metric=metric,
        x=x,
        width=width,
        y_min=y_min,
        y_max=y_max
    )

    # ==================================================
    # Style
    # ==================================================

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.3,
        zorder=0
    )

    sns.despine(ax=ax)

    plt.tight_layout()

    # ==================================================
    # Save
    # ==================================================

    out_pdf = os.path.join(
        OUTDIR,
        f"{modality}_{metric}_with_redline_overall_gain.pdf"
    )

    out_png = os.path.join(
        OUTDIR,
        f"{modality}_{metric}_with_redline_overall_gain.png"
    )

    plt.savefig(
        out_pdf,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.savefig(
        out_png,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print("Saved:", out_pdf)


# ======================================================
# Save standalone legends
# ======================================================

def save_legends():

    # ==================================================
    # ECG legend
    # ==================================================

    fig, ax = plt.subplots(
        figsize=(4, 1)
    )

    ecg_labels = [
        "CNN-1D",
        "BiLSTM",
        "Transformer"
    ]

    ecg_handles = [
        plt.Rectangle(
            (0, 0),
            1,
            1,
            color=MODEL_COLORS[label]
        )
        for label in ecg_labels
    ]

    ax.legend(
        ecg_handles,
        ecg_labels,
        loc="center",
        ncol=3,
        frameon=False
    )

    ax.axis("off")

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            OUTDIR,
            "legend_ECG.pdf"
        ),
        transparent=True,
        bbox_inches="tight"
    )

    plt.close()

    # ==================================================
    # Fundus legend
    # ==================================================

    fig, ax = plt.subplots(
        figsize=(6, 1)
    )

    fundus_labels = [
        "ResNet18",
        "RETFound",
        "ViT-small Freeze",
        "ViT-small Finetuned"
    ]

    fundus_handles = [
        plt.Rectangle(
            (0, 0),
            1,
            1,
            color=MODEL_COLORS[label]
        )
        for label in fundus_labels
    ]

    ax.legend(
        fundus_handles,
        fundus_labels,
        loc="center",
        ncol=4,
        frameon=False
    )

    ax.axis("off")

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            OUTDIR,
            "legend_Fundus.pdf"
        ),
        transparent=True,
        bbox_inches="tight"
    )

    plt.close()

    print("Saved legends.")


# ======================================================
# Run
# ======================================================

for modality in [
    "ECG",
    "Fundus"
]:

    make_plot(
        modality,
        "cindex"
    )

    make_plot(
        modality,
        "auc"
    )

save_legends()

print("\nDone.")

In [ ]:
# ======================================================
# Generate ECG + Fundus LaTeX test-performance table
# ======================================================

ECG_TABLE_MODELS = [
    "Transformer",
    "BiLSTM",
    "CNN-1D",
]

FUNDUS_TABLE_MODELS = [
    "ViT-small Finetuned",
    "ViT-small Freeze",
    "ResNet18",
    "RETFound",
]


TABLE_HEADER_MAP = {
    "Transformer": "Transformer",
    "BiLSTM": "BiLSTM",
    "CNN-1D": "CNN-1D",
    "ViT-small Finetuned": "ViT-Small (Fine-tuned)",
    "ViT-small Freeze": "ViT-Small (Frozen)",
    "ResNet18": "ResNet18",
    "RETFound": "RETFound",
}

METRIC_CONFIG = [
    ("cindex", "cindex_sd", "C-index"),
    ("auc", "auc_sd", "AUC"),
]


def get_result_row(
    results_df,
    modality,
    disease,
    model,
):
    """\u83b7\u53d6\u6307\u5b9a\u6a21\u6001、\u75be\u75c5\u548c\u6a21\u578b\u7684\u7ed3\u679c\u884c。"""
    selected = results_df[
        (results_df["modality"] == modality)
        & (results_df["disease"] == disease)
        & (results_df["model"] == model)
    ]

    if selected.empty:
        return None

    if len(selected) > 1:
        raise ValueError(
            "Duplicate results found:"
            f"modality={modality}, "
            f"disease={disease}, "
            f"model={model}"
        )

    return selected.iloc[0]


def format_mean_sd(
    mean_value,
    sd_value,
    is_best=False,
    decimals=3,
):
    """Analysis helper. Inputs and outputs follow the function implementation."""
    if pd.isna(mean_value) or pd.isna(sd_value):
        return "--"

    text = (
        f"{mean_value:.{decimals}f} "
        rf"$\pm$ "
        f"{sd_value:.{decimals}f}"
    )

    if is_best:
        return rf"\textbf{{{text}}}"

    return text


def make_modality_rows(
    results_df,
    modality,
    models,
):
    """Analysis helper. Inputs and outputs follow the function implementation."""
    latex_rows = []

    for disease_index, disease in enumerate(DISEASE_ORDER):

        if disease_index > 0:
            latex_rows.append(r"\addlinespace[2pt]")

        disease_label = DISEASES[disease]

        
        empty_cells = " &" * len(models)

        latex_rows.append(
            rf"\textit{{{disease_label}}}"
            f"{empty_cells} \\\\"
        )

        
        for metric, sd_metric, metric_label in METRIC_CONFIG:

            metric_values = []

            for model in models:
                row = get_result_row(
                    results_df=results_df,
                    modality=modality,
                    disease=disease,
                    model=model,
                )

                if row is None:
                    metric_values.append(np.nan)
                else:
                    metric_values.append(row[metric])

            valid_values = [
                value
                for value in metric_values
                if pd.notna(value)
            ]

            best_value = (
                max(valid_values)
                if valid_values
                else np.nan
            )

            formatted_values = []

            for model, mean_value in zip(
                models,
                metric_values,
            ):
                row = get_result_row(
                    results_df=results_df,
                    modality=modality,
                    disease=disease,
                    model=model,
                )

                if row is None:
                    sd_value = np.nan
                else:
                    sd_value = row[sd_metric]

                is_best = (
                    pd.notna(mean_value)
                    and pd.notna(best_value)
                    and np.isclose(
                        mean_value,
                        best_value,
                    )
                )

                formatted_values.append(
                    format_mean_sd(
                        mean_value=mean_value,
                        sd_value=sd_value,
                        is_best=is_best,
                    )
                )

            latex_rows.append(
                rf"\quad {metric_label}"
                "\n& "
                + "\n& ".join(formatted_values)
                + r" \\"
            )

    return "\n".join(latex_rows)


def make_ecg_tabular(results_df):
    """\u751f\u6210 ECG \u90e8\u5206，4 \u5217。"""
    headers = [
        TABLE_HEADER_MAP[model]
        for model in ECG_TABLE_MODELS
    ]

    header_line = (
        "Characteristic & "
        + " & ".join(headers)
        + r" \\"
    )

    body = make_modality_rows(
        results_df=results_df,
        modality="ECG",
        models=ECG_TABLE_MODELS,
    )

    return rf"""\begin{{adjustbox}}{{max width=\textwidth}}
\begin{{tabular}}{{lccc}}
\toprule
{header_line}
\midrule
\multicolumn{{4}}{{l}}{{\textbf{{ECG}}}} \\

{body}

\bottomrule
\end{{tabular}}
\end{{adjustbox}}"""


def make_fundus_tabular(results_df):
    """\u751f\u6210 Fundus \u90e8\u5206，5 \u5217。"""
    headers = [
        TABLE_HEADER_MAP[model]
        for model in FUNDUS_TABLE_MODELS
    ]

    header_line = (
        "Characteristic & "
        + " & ".join(headers)
        + r" \\"
    )

    body = make_modality_rows(
        results_df=results_df,
        modality="Fundus",
        models=FUNDUS_TABLE_MODELS,
    )

    return rf"""\begin{{adjustbox}}{{max width=\textwidth}}
\begin{{tabular}}{{lcccc}}
\toprule
{header_line}
\midrule
\multicolumn{{5}}{{l}}{{\textbf{{Fundus Image}}}} \\

{body}

\bottomrule
\end{{tabular}}
\end{{adjustbox}}"""


def generate_test_performance_tex(
    results_df,
    output_path,
):
    """Analysis helper. Inputs and outputs follow the function implementation."""
    ecg_tabular = make_ecg_tabular(results_df)
    fundus_tabular = make_fundus_tabular(results_df)

    latex = rf"""\begin{{table}}[ht]
\centering
\small
\setlength{{\tabcolsep}}{{5pt}}
\renewcommand{{\arraystretch}}{{1.10}}

% ECG: Transformer, BiLSTM, CNN-1D
{ecg_tabular}

\vspace{{8pt}}

% Fundus: ViT-Small, ResNet18, RETFound
{fundus_tabular}

\caption{{
External held-out test performance of ECG and fundus image models
across clinical outcomes. Values are presented as bootstrap mean
$\pm$ standard deviation. The best-performing model for each
outcome and metric within each modality is shown in bold.
}}
\label{{tab:ecg-fundus-test-performance}}
\end{{table}}
"""

    output_directory = os.path.dirname(output_path)

    if output_directory:
        os.makedirs(
            output_directory,
            exist_ok=True,
        )

    with open(
        output_path,
        "w",
        encoding="utf-8",
    ) as file:
        file.write(latex)

    print("Saved LaTeX table:", output_path)
    print()
    print(latex)

    return latex


# ======================================================
# Generate .tex file
# ======================================================

tex_path = os.path.join(
    OUTDIR,
    "ECG_Fundus_test_performance.tex",
)

generate_test_performance_tex(
    results_df=results,
    output_path=tex_path,
)

In [ ]:
!cat fig3_final/ECG_Fundus_test_performance.tex

### Restricted-data analysis not included

This block used original fundus images and participant-level image selection. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.


### Restricted-data analysis not included

This block used original fundus image export and participant-level image selection. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.


### Restricted-data analysis not included

This block used original fundus image vessel extraction. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import gaussian_filter
from matplotlib.patches import Rectangle
from matplotlib.colors import TwoSlopeNorm

# =========================================================
# Config
# =========================================================

BASE_DIR = "code_02_02_save/fundus_20251229_1519"

CAM_DIR  = os.path.join(BASE_DIR, "explanations")
PRED_DIR = os.path.join(BASE_DIR, "preds")

OUT_DIR = "fundus_global_gradcam_high_minus_low"
os.makedirs(OUT_DIR, exist_ok=True)

DISEASE_LIST = ["CHD", "istroke", "stroke", "ICVD"]

DISPLAY_NAME = {
    "CHD": "CHD",
    "istroke": "Ischemic Stroke",
    "stroke": "Stroke",
    "ICVD": "ASCVD"
}

TARGET_SIZE = 224
GAUSSIAN_SIGMA = 8

TOP_FRAC = 0.20
BOTTOM_FRAC = 0.20

# for difference map
DIFF_CMAP = "RdBu_r"  
DIFF_VMIN = None      # auto from data
DIFF_VMAX = None      # auto from data

# for highlighting significant regions
PERCENTILE_THRESH = 90  # top 10% changes are highlighted
SHOW_HIGHLIGHT = True

# =========================================================
# Helpers
# =========================================================

def load_cam(fp):
    cam = np.loadtxt(fp)

    if cam.ndim == 1 and cam.size == 196:
        cam = cam.reshape(14, 14)

    return cam.astype(np.float32)


def normalize(cam):
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-8)
    return cam


def build_cam_index(disease):
    pattern = os.path.join(CAM_DIR, f"gradcam_*_*_{disease}.txt")
    files = glob.glob(pattern)

    idx = {}

    for fp in files:
        fn = os.path.basename(fp)
        parts = fn.replace(".txt", "").split("_")

        if len(parts) < 4:
            continue

        try:
            pid = int(parts[2])
        except:
            continue

        idx[pid] = fp

    return idx


def load_pred(disease):
    pred_path = os.path.join(
        PRED_DIR,
        f"{disease}_fundus_vitsmallFT_test_best_pred.csv"
    )

    df = pd.read_csv(pred_path)

    if "ID" in df.columns:
        df = df.rename(columns={"ID": "pid"})

    df["pid"] = df["pid"].astype(int)
    df["risk"] = df["risk"].astype(float)

    return df


def load_and_resize_cam(fp):

    cam = load_cam(fp)

    # per-sample normalize
    cam = normalize(cam)

    cam = np.array(
        Image.fromarray(cam).resize(
            (TARGET_SIZE, TARGET_SIZE),
            resample=Image.BILINEAR
        )
    )

    return cam


def summarize_cams(pids, cam_index, label=""):

    cams = []
    missing = 0
    failed = 0

    for pid in pids:

        fp = cam_index.get(int(pid), None)

        if fp is None:
            missing += 1
            continue

        try:
            cam = load_and_resize_cam(fp)
            cams.append(cam)
        except Exception:
            failed += 1
            continue

    if len(cams) == 0:
        return None, 0, missing, failed

    cams = np.stack(cams, axis=0)

    mean_cam = cams.mean(axis=0)

    # center
    mean_cam = mean_cam - np.mean(mean_cam)

    # smooth
    mean_cam = gaussian_filter(mean_cam, sigma=GAUSSIAN_SIGMA)

    print(
        f"{label}: used={len(cams)}, missing={missing}, failed={failed}, "
        f"shape={cams.shape}"
    )

    return mean_cam, len(cams), missing, failed


def plot_difference_map_with_highlight(disease, low_cam, high_cam, n_low, n_high):
    """
    Plot the difference map (high - low) to show regions where high-risk patients
    have stronger/weaker Grad-CAM activation compared to low-risk patients.
    """
    
    # Calculate difference
    diff_map = high_cam - low_cam
    
    # Determine colormap limits (symmetric around 0)
    max_abs = np.nanmax(np.abs(diff_map))
    vmin_diff = -max_abs
    vmax_diff = max_abs
    
    # Create figure with 3 subplots
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    
    # Plot 1: Low-risk mean CAM
    im1 = axes[0, 0].imshow(low_cam, cmap='viridis', vmin=0, vmax=np.max([low_cam.max(), high_cam.max()]))
    axes[0, 0].set_title(f"Low Risk (n={n_low})\nMean Grad-CAM", fontsize=11)
    axes[0, 0].axis('off')
    plt.colorbar(im1, ax=axes[0, 0], fraction=0.046, pad=0.04)
    
    # Plot 2: High-risk mean CAM
    im2 = axes[0, 1].imshow(high_cam, cmap='viridis', vmin=0, vmax=np.max([low_cam.max(), high_cam.max()]))
    axes[0, 1].set_title(f"High Risk (n={n_high})\nMean Grad-CAM", fontsize=11)
    axes[0, 1].axis('off')
    plt.colorbar(im2, ax=axes[0, 1], fraction=0.046, pad=0.04)
    
    # Plot 3: Difference map (High - Low)
    im3 = axes[1, 0].imshow(diff_map, cmap=DIFF_CMAP, vmin=vmin_diff, vmax=vmax_diff)
    axes[1, 0].set_title("Difference Map\n(High Risk - Low Risk)", fontsize=11)
    axes[1, 0].axis('off')
    cbar3 = plt.colorbar(im3, ax=axes[1, 0], fraction=0.046, pad=0.04)
    cbar3.set_label('Δ Activation', fontsize=9)
    
    # Plot 4: Difference map with highlighted significant regions
    if SHOW_HIGHLIGHT:
        # Calculate percentiles for highlighting
        abs_diff = np.abs(diff_map)
        threshold = np.percentile(abs_diff, PERCENTILE_THRESH)
        
        # Create mask for significant changes
        significant_mask = abs_diff >= threshold
        
        # Plot difference map
        axes[1, 1].imshow(diff_map, cmap=DIFF_CMAP, vmin=vmin_diff, vmax=vmax_diff)
        
        # Find and highlight contiguous regions
        from scipy import ndimage
        
        # Label connected components
        labeled_mask, num_features = ndimage.label(significant_mask)
        
        # Get properties of each region
        from skimage.measure import regionprops
        regions = regionprops(labeled_mask)
        
        # Highlight regions with bounding boxes
        colors = ['red', 'orange', 'yellow']
        for idx, region in enumerate(regions):
            if region.area > 10:  # Filter small regions
                minr, minc, maxr, maxc = region.bbox
                rect = Rectangle((minc, minr), maxc-minc, maxr-minr,
                               linewidth=2, edgecolor=colors[idx % len(colors)],
                               facecolor='none', alpha=0.8)
                axes[1, 1].add_patch(rect)
        
        axes[1, 1].set_title(f"Significant Changes\n(top {100-PERCENTILE_THRESH}% absolute diff)", fontsize=11)
        axes[1, 1].axis('off')
        
        # Add legend for highlights
        from matplotlib.lines import Line2D
        legend_elements = [Line2D([0], [0], color='red', linewidth=2, label='Significant region')]
        axes[1, 1].legend(handles=legend_elements, loc='upper right', fontsize=8)
    else:
        axes[1, 1].axis('off')
    
    fig.suptitle(
        f"{DISPLAY_NAME.get(disease, disease)} | High vs Low Risk Grad-CAM Comparison\n"
        f"Red: Higher activation in High-risk | Blue: Higher activation in Low-risk",
        fontsize=12, y=0.98
    )
    
    plt.tight_layout()
    
    # Save outputs
    out_pdf = os.path.join(
        OUT_DIR,
        f"{disease}_high_minus_low_difference.pdf"
    )
    
    out_png = os.path.join(
        OUT_DIR,
        f"{disease}_high_minus_low_difference.png"
    )
    
    plt.savefig(out_pdf, dpi=300, bbox_inches='tight')
    plt.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.close(fig)
    
    print(f"Saved difference maps: {out_pdf}")
    print(f"  - Difference range: [{diff_map.min():.4f}, {diff_map.max():.4f}]")
    print(f"  - Positive changes (High>Low): {np.sum(diff_map > 0)} pixels")
    print(f"  - Negative changes (Low>High): {np.sum(diff_map < 0)} pixels")
    
    return diff_map


def plot_single_difference_only(disease, low_cam, high_cam, n_low, n_high):
    """
    Simplified version: Only plot the difference map with clear annotations.
    """
    
    diff_map = high_cam - low_cam
    
    # Determine symmetric limits
    max_abs = np.nanmax(np.abs(diff_map))
    vmin_diff = -max_abs
    vmax_diff = max_abs
    
    fig, ax = plt.subplots(1, 1, figsize=(6, 5))
    
    im = ax.imshow(diff_map, cmap=DIFF_CMAP, vmin=vmin_diff, vmax=vmax_diff)
    ax.set_title(f"{DISPLAY_NAME.get(disease, disease)}\nHigh Risk (n={n_high}) - Low Risk (n={n_low})", fontsize=12)
    ax.axis('off')
    
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Grad-CAM Activation Difference', fontsize=10)
    
    # Add annotation
    ax.text(0.02, 0.98, '↑ Red = Higher in High-risk\n↓ Blue = Higher in Low-risk',
            transform=ax.transAxes, fontsize=9,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    
    out_pdf = os.path.join(OUT_DIR, f"{disease}_high_minus_low_simple.pdf")
    out_png = os.path.join(OUT_DIR, f"{disease}_high_minus_low_simple.png")
    
    plt.savefig(out_pdf, dpi=300, bbox_inches='tight')
    plt.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.close(fig)
    
    print(f"Saved simplified difference map: {out_pdf}")


def export_difference_stats(disease, low_cam, high_cam, diff_map):
    """
    Export statistics about the difference map to a text file.
    """
    
    stats_file = os.path.join(OUT_DIR, f"{disease}_difference_stats.txt")
    
    with open(stats_file, 'w') as f:
        f.write(f"Difference Map Statistics for {disease}\n")
        f.write("="*50 + "\n\n")
        
        f.write(f"Low-risk group: {low_cam.shape[0]}x{low_cam.shape[1]} pixels\n")
        f.write(f"High-risk group: {high_cam.shape[0]}x{high_cam.shape[1]} pixels\n\n")
        
        f.write("Difference (High - Low) Statistics:\n")
        f.write(f"  Minimum: {diff_map.min():.6f}\n")
        f.write(f"  Maximum: {diff_map.max():.6f}\n")
        f.write(f"  Mean: {diff_map.mean():.6f}\n")
        f.write(f"  Std: {diff_map.std():.6f}\n\n")
        
        f.write("Region Analysis:\n")
        f.write(f"  Pixels with High > Low (positive): {np.sum(diff_map > 0)} ({100*np.sum(diff_map > 0)/diff_map.size:.1f}%)\n")
        f.write(f"  Pixels with Low > High (negative): {np.sum(diff_map < 0)} ({100*np.sum(diff_map < 0)/diff_map.size:.1f}%)\n\n")
        
        # Percentiles
        f.write("Percentiles of absolute differences:\n")
        for p in [50, 75, 90, 95, 99]:
            val = np.percentile(np.abs(diff_map), p)
            f.write(f"  {p}th percentile: {val:.6f}\n")
        
        # Find top 5 positive and negative regions
        f.write("\nTop 5 positive differences (High > Low):\n")
        flat_idx = np.argsort(diff_map.flatten())[-5:][::-1]
        for i, idx in enumerate(flat_idx):
            val = diff_map.flatten()[idx]
            coords = np.unravel_index(idx, diff_map.shape)
            f.write(f"  {i+1}. Value={val:.6f} at position ({coords[0]}, {coords[1]})\n")
        
        f.write("\nTop 5 negative differences (Low > High):\n")
        flat_idx = np.argsort(diff_map.flatten())[:5]
        for i, idx in enumerate(flat_idx):
            val = diff_map.flatten()[idx]
            coords = np.unravel_index(idx, diff_map.shape)
            f.write(f"  {i+1}. Value={val:.6f} at position ({coords[0]}, {coords[1]})\n")
    
    print(f"Saved statistics: {stats_file}")


# =========================================================
# Main
# =========================================================

def main():

    for disease in DISEASE_LIST:

        print("\n====================================================")
        print(f"Processing disease: {disease}")
        print("====================================================")

        pred_df = load_pred(disease)
        cam_index = build_cam_index(disease)

        print("Prediction samples:", len(pred_df))
        print("CAM files:", len(cam_index))

        pred_df = pred_df.sort_values("risk", ascending=True).copy()

        n_total = len(pred_df)
        n_low = max(1, int(np.floor(n_total * BOTTOM_FRAC)))
        n_high = max(1, int(np.floor(n_total * TOP_FRAC)))

        low_pids = pred_df.head(n_low)["pid"].astype(int).tolist()
        high_pids = pred_df.tail(n_high)["pid"].astype(int).tolist()

        print(f"Low risk selected: {len(low_pids)}")
        print(f"High risk selected: {len(high_pids)}")

        low_cam, n_low_used, _, _ = summarize_cams(
            low_pids,
            cam_index,
            label=f"{disease} low risk"
        )

        high_cam, n_high_used, _, _ = summarize_cams(
            high_pids,
            cam_index,
            label=f"{disease} high risk"
        )

        if low_cam is None or high_cam is None:
            print(f"[Skip] {disease}: low/high CAM missing")
            continue

        # Calculate and plot difference maps
        diff_map = plot_difference_map_with_highlight(
            disease=disease,
            low_cam=low_cam,
            high_cam=high_cam,
            n_low=n_low_used,
            n_high=n_high_used
        )
        
        # Also save a simplified version
        plot_single_difference_only(
            disease=disease,
            low_cam=low_cam,
            high_cam=high_cam,
            n_low=n_low_used,
            n_high=n_high_used
        )
        
        # Export statistics
        export_difference_stats(disease, low_cam, high_cam, diff_map)

    print("\nAll done.")
    print("Output folder:", OUT_DIR)
    print("\nInterpretation guide:")
    print("  - Red regions: Higher Grad-CAM activation in high-risk patients")
    print("  - Blue regions: Higher Grad-CAM activation in low-risk patients")
    print("  - Boxed regions: Top 10% most significant absolute differences")


# =========================================================
# Run
# =========================================================

if __name__ == "__main__":
    main()

### Restricted-data analysis not included

This block used raw ECG waveform loading and rendering. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import pandas as pd
import numpy as np

# =====================================================
# Config
# =====================================================

#PID = 2764
PID = 2257
CSV_PATH = (
    "code_02_02_save/ecg_20260203_1003/preds/"
    "CHD_ecg_transformer_h256_l3_test_best_pred.csv"
)

# =====================================================
# Read prediction file
# =====================================================

df = pd.read_csv(CSV_PATH)

# =====================================================
# Normalize risk to 0-1
# =====================================================

risk_min = df["risk"].min()
risk_max = df["risk"].max()

df["risk_norm"] = (
    (df["risk"] - risk_min)
    / (risk_max - risk_min)
)

# =====================================================
# Find target PID
# =====================================================

row = df[df["ID"] == PID]

if len(row) == 0:

    print(f"PID {PID} not found")

else:

    risk_raw = row["risk"].values[0]
    risk_norm = row["risk_norm"].values[0]
    time_val = row["time"].values[0]
    event_val = row["event"].values[0]

    print("\n==============================")
    print(f"PID            : {PID}")
    print(f"Raw risk       : {risk_raw:.6f}")
    print(f"Normalized risk: {risk_norm:.6f}")
    print(f"Time           : {time_val:.6f}")
    print(f"Event          : {event_val}")
    print("==============================\n")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from adjustText import adjust_text

import gseapy as gp

# ======================================================
# Config
# ======================================================

BASE = "data/private/inputs"

INPUT_DIR = os.path.join(
    BASE,
    "code_3_3_save/high_vs_low_risk_feature_analysis"
)

OUT_DIR = "fig3_final"
os.makedirs(OUT_DIR, exist_ok=True)

GO_DIR = os.path.join(
    OUT_DIR,
    "GO"
)
os.makedirs(GO_DIR, exist_ok=True)

ENDPOINTS = ["CHD", "istroke"]

FEATURE_TYPES = ["protein", "SNPgene"]

PVALUE_CUTOFF = 0.05


MAX_LABEL_PER_QUADRANT = 30

FIGSIZE = (8, 8)

GO_LIBRARY = "GO_Biological_Process_2023"

# GO only show FDR < 0.05
GO_FDR = 0.05

# GO top terms
TOP_GO = 8

# ======================================================
# Highlight genes
# ======================================================

HIGHLIGHT_GENES = set([
    "ADIPOQ", "VEGFC", "PLAT", "FAS", "IL1RAP", "FGR", "CLEC3B",
    "LDLR", "SORT1", "MMP9", "PLA2G2A", "AXL", "SLAMF8", "TNFRSF8",
    "JAM3", "ICAM4", "CD59", "TIMP3", "TNC",
    "CCL3", "CCL18", "CCL11", "MMP3", "GDF15", "CST3", "BCL2L11",
    "ACVRL1", "CLEC14A", "CD93", "DDR1", "MMP12", "EPO", "HSPB1", "WFDC2"
])

# ======================================================
# Colors
# ======================================================

QUAD_COLORS = {
    "Fundus high / ECG high": "#C44E52",
    "Fundus low / ECG high": "#4C72B0",
    "Fundus low / ECG low": "#6cb0c9",
    "Fundus high / ECG low": "#8172B2",
}

COLOR_NS = "#D0D0D0"

# ======================================================
# Utils
# ======================================================

def classify_quadrant(x, y):

    if x > 0 and y > 0:
        return "Fundus high / ECG high"

    elif x < 0 and y > 0:
        return "Fundus low / ECG high"

    elif x < 0 and y < 0:
        return "Fundus low / ECG low"

    elif x > 0 and y < 0:
        return "Fundus high / ECG low"

    else:
        return "Other"


def clean_gene(x):

    x = str(x)

    if x.startswith("seq."):
        x = x.replace("seq.", "", 1)

    x = re.sub(r"\.\d+$", "", x)

    return x


def get_sig(row):

    return (
        (row["fundus_pval"] < PVALUE_CUTOFF) or
        (row["ecg_pval"] < PVALUE_CUTOFF)
    )


# ======================================================
# GO enrichment
# ======================================================

def run_go_enrichment(gene_list, outdir, prefix):

    os.makedirs(outdir, exist_ok=True)

    gene_list = [
        clean_gene(g)
        for g in gene_list
    ]

    gene_list = sorted(list(set(gene_list)))

    gene_list = [
        g for g in gene_list
        if g != "nan"
    ]

    pd.DataFrame({
        "gene": gene_list
    }).to_csv(
        os.path.join(outdir, f"{prefix}_genes.csv"),
        index=False
    )

    print(f"[GO] {prefix}: {len(gene_list)} genes")

    if len(gene_list) < 5:
        print("[GO skip] too few genes")
        return None, set()

    try:

        enr = gp.enrichr(
            gene_list=gene_list,
            gene_sets=GO_LIBRARY,
            organism="Human",
            outdir=outdir,
            cutoff=1.0,
            no_plot=True
        )

        res = enr.results.copy()

        res.to_csv(
            os.path.join(outdir, f"{prefix}_GO.csv"),
            index=False
        )

        # ============================================
        # keep FDR < 0.05
        # ============================================

        res = res[
            res["Adjusted P-value"] < GO_FDR
        ].copy()

        if len(res) == 0:
            return None, set()

        res = res.sort_values(
            "Adjusted P-value",
            ascending=True
        ).head(TOP_GO)

        # ============================================
        # extract genes
        # ============================================

        go_genes = set()

        for gs in res["Genes"].tolist():

            for g in str(gs).split(";"):

                go_genes.add(clean_gene(g))

        # ============================================
        # barplot
        # ============================================

        res["minuslog10FDR"] = -np.log10(
            res["Adjusted P-value"] + 1e-300
        )

        quad_name = (
            prefix
            .replace("_", " ")
        )

        quad_color = None

        for k, v in QUAD_COLORS.items():

            if (
                "Fundus high ECG high" in quad_name
                and k == "Fundus high / ECG high"
            ):
                quad_color = v

            elif (
                "Fundus low ECG high" in quad_name
                and k == "Fundus low / ECG high"
            ):
                quad_color = v

            elif (
                "Fundus low ECG low" in quad_name
                and k == "Fundus low / ECG low"
            ):
                quad_color = v

            elif (
                "Fundus high ECG low" in quad_name
                and k == "Fundus high / ECG low"
            ):
                quad_color = v

        if quad_color is None:
            quad_color = "gray"

        fig_h = max(3, len(res) * 0.9)

        plt.figure(figsize=(6.5, fig_h))

        y = np.arange(len(res))

        plt.barh(
            y,
            res["minuslog10FDR"].values,
            color=quad_color,
            height=0.75
        )

        plt.yticks(
            y,
            res["Term"].tolist(),
            fontsize=9
        )

        plt.gca().invert_yaxis()

        # genes inside bar
        for yi, genes in zip(y, res["Genes"].tolist()):

            genes_txt = genes.replace(";", ", ")

            plt.text(
                0.05,
                yi,
                genes_txt,
                fontsize=7,
                va="center",
                ha="left",
                color="black"
            )

        plt.xlabel(r"$-\log_{10}(FDR)$", fontsize=11)

        plt.title(
            prefix.replace("_", " "),
            fontsize=12
        )

        plt.tight_layout()

        out_pdf = os.path.join(
            outdir,
            f"{prefix}_GO_barplot.pdf"
        )

        plt.savefig(
            out_pdf,
            dpi=300,
            transparent=True
        )

        plt.close()

        return res, go_genes

    except Exception as e:

        print("[GO ERROR]", prefix)
        print(e)

        return None, set()


# ======================================================
# Main plot
# ======================================================

def plot_one(endpoint, feature_type):

    print("\n================================================")
    print(endpoint, feature_type)
    print("================================================")

    fundus_file = os.path.join(
        INPUT_DIR,
        f"{endpoint}_Fundus_{feature_type}_high_vs_low.csv"
    )

    ecg_file = os.path.join(
        INPUT_DIR,
        f"{endpoint}_ECG_{feature_type}_high_vs_low.csv"
    )

    df_f = pd.read_csv(fundus_file)
    df_e = pd.read_csv(ecg_file)

    # ==================================================
    # rename
    # ==================================================

    df_f = df_f.rename(columns={
        "delta": "fundus_delta",
        "pval": "fundus_pval",
        "FDR": "fundus_FDR",
    })

    df_e = df_e.rename(columns={
        "delta": "ecg_delta",
        "pval": "ecg_pval",
        "FDR": "ecg_FDR",
    })

    merged = df_f.merge(
        df_e,
        on="gene",
        how="inner"
    )

    merged["gene_clean"] = merged["gene"].map(clean_gene)

    merged["quadrant"] = [
        classify_quadrant(x, y)
        for x, y in zip(
            merged["fundus_delta"],
            merged["ecg_delta"]
        )
    ]

    merged["sig"] = merged.apply(get_sig, axis=1)

    merged["combined_score"] = (
        -np.log10(merged["fundus_pval"] + 1e-300)
        +
        -np.log10(merged["ecg_pval"] + 1e-300)
    )

    # ==================================================
    # GO enrichment per quadrant
    # ==================================================

    GO_GENES = set()

    for quad in QUAD_COLORS.keys():

        sub = merged[
            (merged["quadrant"] == quad)
            &
            (merged["sig"])
        ].copy()

        if len(sub) == 0:
            continue

        quad_name = (
            quad
            .replace("/", "")
            .replace(" ", "_")
        )

        prefix = (
            f"{endpoint}_{feature_type}_{quad_name}"
        )

        _, go_genes = run_go_enrichment(
            sub["gene_clean"].tolist(),
            os.path.join(
                GO_DIR,
                endpoint,
                feature_type,
                quad_name
            ),
            prefix
        )

        GO_GENES.update(go_genes)

    print("GO-related genes:", len(GO_GENES))

    # ==================================================
    # plot
    # ==================================================

    fig, ax = plt.subplots(figsize=FIGSIZE)

    # background lines
    ax.axhline(
        0,
        linestyle="--",
        linewidth=1,
        color="gray",
        alpha=0.7
    )

    ax.axvline(
        0,
        linestyle="--",
        linewidth=1,
        color="gray",
        alpha=0.7
    )

    # ==================================================
    # non-significant
    # ==================================================

    ns = merged[
        ~merged["sig"]
    ].copy()

    ax.scatter(
        ns["fundus_delta"],
        ns["ecg_delta"],
        s=15,
        alpha=0.3,
        color=COLOR_NS,
        rasterized=True
    )

    # ==================================================
    # significant colored
    # ==================================================

    sig_all = merged[
        merged["sig"]
    ].copy()

    for quad, color in QUAD_COLORS.items():

        sub = sig_all[
            sig_all["quadrant"] == quad
        ]

        ax.scatter(
            sub["fundus_delta"],
            sub["ecg_delta"],
            s=28,
            alpha=0.9,
            color=color,
            label=f"{quad} (n={len(sub)})",
            rasterized=True
        )

    # ==================================================
    # label only top30 per quadrant
    # ==================================================

    texts = []

    for quad in QUAD_COLORS.keys():

        sub = sig_all[
            sig_all["quadrant"] == quad
        ].copy()

        if len(sub) == 0:
            continue

        # only top30 labels
        sub = sub.sort_values(
            "combined_score",
            ascending=False
        ).head(MAX_LABEL_PER_QUADRANT)

        print(
            quad,
            "label genes:",
            len(sub)
        )

        for _, row in sub.iterrows():

            gene = row["gene_clean"]

            # ======================================
            # GO-related genes -> red
            # ======================================

            if gene in GO_GENES:
                txt_color = "red"

            elif gene in HIGHLIGHT_GENES:
                txt_color = "red"

            else:
                txt_color = "black"

            texts.append(
                ax.text(
                    row["fundus_delta"],
                    row["ecg_delta"],
                    gene,
                    fontsize=7,
                    color=txt_color
                )
            )

    if len(texts) > 0:

        adjust_text(
            texts,
            ax=ax,
            arrowprops=dict(
                arrowstyle="-",
                color="gray",
                lw=0.5
            )
        )

    # ==================================================
    # aesthetics
    # ==================================================

    ax.set_xlabel(
        "Fundus high-risk vs low-risk Δ",
        fontsize=12
    )

    ax.set_ylabel(
        "ECG high-risk vs low-risk Δ",
        fontsize=12
    )

    ax.set_title(
        f"{endpoint} | {feature_type}",
        fontsize=14,
        pad=15
    )

    ax.legend(
        frameon=False,
        fontsize=8,
        loc="best"
    )

    # real range
    xmin = np.nanmin(merged["fundus_delta"])
    xmax = np.nanmax(merged["fundus_delta"])

    ymin = np.nanmin(merged["ecg_delta"])
    ymax = np.nanmax(merged["ecg_delta"])

    xpad = (xmax - xmin) * 0.08
    ypad = (ymax - ymin) * 0.08

    if xpad == 0:
        xpad = 0.1

    if ypad == 0:
        ypad = 0.1

    ax.set_xlim(
        xmin - xpad,
        xmax + xpad
    )

    ax.set_ylim(
        ymin - ypad,
        ymax + ypad
    )

    plt.tight_layout()

    out_pdf = os.path.join(
        OUT_DIR,
        f"{endpoint}_{feature_type}_quadrant_scatter_GO_final.pdf"
    )

    out_png = os.path.join(
        OUT_DIR,
        f"{endpoint}_{feature_type}_quadrant_scatter_GO_final.png"
    )

    plt.savefig(
        out_pdf,
        dpi=300,
        transparent=True
    )

    plt.savefig(
        out_png,
        dpi=300,
        transparent=True
    )

    plt.close()

    print("Saved:", out_pdf)


# ======================================================
# Run
# ======================================================

for endpoint in ENDPOINTS:

    for feature_type in FEATURE_TYPES:

        try:

            plot_one(
                endpoint,
                feature_type
            )

        except Exception as e:

            print("[ERROR]", endpoint, feature_type)
            print(e)

print("\nAll done.")
print("Output:", OUT_DIR)

### Restricted-data analysis not included

This block used sample-level protein/SNP input values. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests


# ======================================================
# Global matplotlib settings
# ======================================================

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"


# ======================================================
# Config
# ======================================================

BASE = "data/private/inputs"

OUT_DIR = os.path.join(
    BASE,
    "fig3_final",
    "protein_average_minus_single_modal_score_rankplot_ICVD_only"
)
os.makedirs(OUT_DIR, exist_ok=True)



TARGET_ENDPOINT = "ICVD"

ENDPOINT_LABELS = {
    "CHD": "CHD",
    "istroke": "Ischemic stroke",
    "ICVDr": "ASCVD",
    "ICVD": "ASCVD",
}


# ======================================================
# Risk models
# ======================================================

RISK_MODELS = {
    "SNP":
        "code_02_02_save/snp_20260202_1432/preds/{endpoint}_snp_TR_test_best_pred.csv",

    "Soma":
        "code_02_02_save/soma_20260202_2006/preds/{endpoint}_soma_TR_test_best_pred.csv",

    "ECG":
        "code_02_02_save/ecg_20260203_1003/preds/{endpoint}_ecg_transformer_h256_l3_test_best_pred.csv",

    "Fundus":
        "code_02_02_save/fundus_20251229_1519/preds/{endpoint}_fundus_vitsmallFT_test_best_pred.csv",
}

MODALITY_ORDER = [
    "SNP",
    "Soma",
    "Fundus",
    "ECG"
]


# ======================================================
# Protein input
# ======================================================

PROTEIN_INPUT = os.path.join(
    BASE,
    "code_3_3_save/soma_single/CHD_soma_TR_best_IG_input.csv"
)

PROTEIN_VALUE_COL = "ig_abs"


# ======================================================
# Parameters
# ======================================================

MIN_N = 20


TOP_N_POINT = 50
BOTTOM_N_POINT = 50


LABEL_TOP_GENES = [
    "GDF15",
    "IL6",
    "MMP12",
    "CTSS",
    "TIMP1",
    "ESAM",
    "IL18BP",
    "CXCL10",
    "COL18A1",
    "SMOC1",
]

LABEL_BOTTOM_GENES = [
    "F2R",
    "FLT3",
    "TNFRSF10A",
    "ANGPTL2",
    "HLA-A",
    "HLA-E",
    "CD276",
    "CLEC4G",
    "PKD1",
    "NPR1",
]

FIG_W_MM = 45
FIG_H_MM = 70
MM_TO_INCH = 1 / 25.4

FIGSIZE = (
    FIG_W_MM * MM_TO_INCH,
    FIG_H_MM * MM_TO_INCH
)

COLOR_NEG = "#4C6A92"
COLOR_POS = "#B64A4A"
COLOR_BG = "#BDBDBD"


# ======================================================
# Utils
# ======================================================

def check_file(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)


def clean_gene(x):
    x = str(x)

    x = re.sub(r"\.\d+$", "", x)

    if x.startswith("seq."):
        parts = x.split(".")
        if len(parts) > 1:
            x = parts[1]

    return x.upper()


def rank_norm(x):
    x = pd.Series(x).astype(float)
    r = x.rank(method="average").values

    if len(r) <= 1:
        return np.zeros_like(r, dtype=float)

    return (r - 1) / (len(r) - 1 + 1e-8)


def safe_spearman(x, y, min_n=10):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    keep = (
        ~np.isnan(x)
        &
        ~np.isnan(y)
    )

    if keep.sum() < min_n:
        return np.nan, np.nan, int(keep.sum())

    if np.nanstd(x[keep]) == 0 or np.nanstd(y[keep]) == 0:
        return np.nan, np.nan, int(keep.sum())

    rho, pval = spearmanr(
        x[keep],
        y[keep]
    )

    return rho, pval, int(keep.sum())


# ======================================================
# Load protein matrix
# ======================================================

def load_protein_matrix():
    check_file(PROTEIN_INPUT)

    df = pd.read_csv(PROTEIN_INPUT)

    required_cols = [
        "pid",
        "feature",
        PROTEIN_VALUE_COL
    ]

    for c in required_cols:
        if c not in df.columns:
            raise ValueError(
                f"Protein input must contain column: {c}. "
                f"Available columns: {list(df.columns)}"
            )

    df["pid"] = df["pid"].astype(int)
    df["feature"] = df["feature"].astype(str)
    df["gene"] = df["feature"].map(clean_gene)

    df[PROTEIN_VALUE_COL] = pd.to_numeric(
        df[PROTEIN_VALUE_COL],
        errors="coerce"
    )

    df = df.dropna(
        subset=[
            "pid",
            "gene",
            PROTEIN_VALUE_COL
        ]
    ).copy()

    mat = df.pivot_table(
        index="gene",
        columns="pid",
        values=PROTEIN_VALUE_COL,
        aggfunc="mean"
    )

    print("[Protein matrix] genes:", mat.shape[0])
    print("[Protein matrix] samples:", mat.shape[1])
    print("[Protein value column]:", PROTEIN_VALUE_COL)

    return mat


# ======================================================
# Load risk
# ======================================================

def load_risk(endpoint, modality):
    risk_fp = os.path.join(
        BASE,
        RISK_MODELS[modality].format(endpoint=endpoint)
    )

    check_file(risk_fp)

    risk_df = pd.read_csv(risk_fp)

    if "ID" in risk_df.columns:
        risk_df = risk_df.rename(columns={"ID": "pid"})

    if "pid" not in risk_df.columns:
        raise ValueError(
            f"No pid column in risk file: {risk_fp}. "
            f"Available columns: {list(risk_df.columns)}"
        )

    if "risk" not in risk_df.columns:
        raise ValueError(
            f"No risk column in risk file: {risk_fp}. "
            f"Available columns: {list(risk_df.columns)}"
        )

    risk_df["pid"] = risk_df["pid"].astype(int)
    risk_df["risk"] = pd.to_numeric(
        risk_df["risk"],
        errors="coerce"
    )

    risk_df = risk_df.dropna(
        subset=[
            "pid",
            "risk"
        ]
    ).copy()

    risk_df[f"{modality}_risk"] = rank_norm(
        risk_df["risk"].values
    )

    risk_df = risk_df[[
        "pid",
        f"{modality}_risk"
    ]].drop_duplicates("pid")

    print(
        f"[{endpoint} | {modality}] risk samples:",
        risk_df["pid"].nunique()
    )

    return risk_df


def load_all_risks(endpoint):
    risk_list = []

    for modality in MODALITY_ORDER:
        tmp = load_risk(
            endpoint,
            modality
        )
        risk_list.append(tmp)

    risk_df = risk_list[0]

    for tmp in risk_list[1:]:
        risk_df = risk_df.merge(
            tmp,
            on="pid",
            how="inner"
        )

    risk_cols = [
        f"{m}_risk"
        for m in MODALITY_ORDER
    ]

    risk_df["Average_risk"] = risk_df[
        risk_cols
    ].mean(axis=1)

    risk_df["Average_risk"] = rank_norm(
        risk_df["Average_risk"].values
    )

    print(
        f"[{endpoint}] common risk samples across 4 modalities:",
        risk_df["pid"].nunique()
    )

    return risk_df


# ======================================================
# Compute protein score
# ======================================================

def compute_endpoint_score(protein_mat, endpoint):
    print("\n====================================")
    print(endpoint)
    print("====================================")

    risk_df = load_all_risks(endpoint)

    common_pids = sorted(
        list(
            set(protein_mat.columns)
            &
            set(risk_df["pid"])
        )
    )

    print(
        f"[{endpoint}] common samples between protein and risks:",
        len(common_pids)
    )

    if len(common_pids) < MIN_N:
        raise ValueError(
            f"Too few common samples for endpoint {endpoint}: {len(common_pids)}"
        )

    sub_mat = protein_mat[
        common_pids
    ].copy()

    risk_df = risk_df[
        risk_df["pid"].isin(common_pids)
    ].copy()

    risk_df = risk_df.set_index("pid").loc[common_pids]

    risk_vectors = {
        "Average": risk_df["Average_risk"].values.astype(float)
    }

    for modality in MODALITY_ORDER:
        risk_vectors[modality] = risk_df[
            f"{modality}_risk"
        ].values.astype(float)

    rows = []

    total = len(sub_mat.index)

    for i, gene in enumerate(sub_mat.index):
        if i % 500 == 0:
            print(f"[{endpoint}] {i}/{total}")

        protein_vec = sub_mat.loc[
            gene,
            common_pids
        ].values.astype(float)

        avg_rho, avg_pval, avg_n = safe_spearman(
            protein_vec,
            risk_vectors["Average"],
            min_n=MIN_N
        )

        row = {
            "endpoint": endpoint,
            "gene": gene,
            "Average_rho": avg_rho,
            "Average_pval": avg_pval,
            "Average_n": avg_n,
        }

        diffs = []

        for modality in MODALITY_ORDER:
            rho, pval, n = safe_spearman(
                protein_vec,
                risk_vectors[modality],
                min_n=MIN_N
            )

            row[f"{modality}_rho"] = rho
            row[f"{modality}_pval"] = pval
            row[f"{modality}_n"] = n

            diff = avg_rho - rho

            row[f"Average_minus_{modality}"] = diff

            if not np.isnan(diff):
                diffs.append(diff)

        if len(diffs) > 0:
            row["score"] = np.mean(diffs)
            row["score_abs"] = np.abs(row["score"])
        else:
            row["score"] = np.nan
            row["score_abs"] = np.nan

        rows.append(row)

    df = pd.DataFrame(rows)

    pval_cols = [
        "Average_pval"
    ] + [
        f"{m}_pval"
        for m in MODALITY_ORDER
    ]

    for pcol in pval_cols:
        fdr_col = pcol.replace("_pval", "_FDR")

        ok = df[pcol].notna()

        df[fdr_col] = np.nan

        if ok.sum() > 0:
            df.loc[ok, fdr_col] = multipletests(
                df.loc[ok, pcol].values,
                method="fdr_bh"
            )[1]

    df = df.sort_values(
        "score",
        ascending=True
    ).reset_index(drop=True)

    df["score_rank_low_to_high"] = np.arange(
        1,
        len(df) + 1
    )

    df["score_rank_high_to_low"] = df["score"].rank(
        ascending=False,
        method="first"
    ).astype(int)

    out_csv = os.path.join(
        OUT_DIR,
        f"{endpoint}_protein_average_minus_single_modal_score_table.csv"
    )

    df.to_csv(
        out_csv,
        index=False
    )

    print("Saved score table:", out_csv)

    return df


# ======================================================
# Plot score rank plot
# ======================================================

def plot_score_rank(endpoint, endpoint_label, df):
    plot_df = df.dropna(
        subset=["score"]
    ).copy()

    plot_df = plot_df.sort_values(
        "score",
        ascending=True
    ).reset_index(drop=True)

    plot_df["rank"] = np.arange(
        1,
        len(plot_df) + 1
    )

    n_total = len(plot_df)

    
    bottom50_df = plot_df.head(BOTTOM_N_POINT).copy()
    top50_df = plot_df.tail(TOP_N_POINT).copy()

    
    label_bottom_df = bottom50_df[
        bottom50_df["gene"].isin(LABEL_BOTTOM_GENES)
    ].copy()

    label_top_df = top50_df[
        top50_df["gene"].isin(LABEL_TOP_GENES)
    ].copy()

    missing_top = [
        g for g in LABEL_TOP_GENES
        if g not in set(label_top_df["gene"])
    ]

    missing_bottom = [
        g for g in LABEL_BOTTOM_GENES
        if g not in set(label_bottom_df["gene"])
    ]

    if len(missing_top) > 0:
        print("[Warning] These selected TOP genes are not in computed top50:", missing_top)

    if len(missing_bottom) > 0:
        print("[Warning] These selected BOTTOM genes are not in computed bottom50:", missing_bottom)

    print("\n[Top 50 score proteins]")
    print(
        top50_df[[
            "endpoint",
            "gene",
            "Average_rho",
            "SNP_rho",
            "Soma_rho",
            "Fundus_rho",
            "ECG_rho",
            "score"
        ]]
    )

    print("\n[Bottom 50 score proteins]")
    print(
        bottom50_df[[
            "endpoint",
            "gene",
            "Average_rho",
            "SNP_rho",
            "Soma_rho",
            "Fundus_rho",
            "ECG_rho",
            "score"
        ]]
    )

    print("\n[Labeled top genes]")
    print(
        label_top_df[[
            "endpoint",
            "gene",
            "score",
            "Average_rho",
            "SNP_rho",
            "Soma_rho",
            "Fundus_rho",
            "ECG_rho"
        ]]
    )

    print("\n[Labeled bottom genes]")
    print(
        label_bottom_df[[
            "endpoint",
            "gene",
            "score",
            "Average_rho",
            "SNP_rho",
            "Soma_rho",
            "Fundus_rho",
            "ECG_rho"
        ]]
    )

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    # ==================================================
    # Scatter
    # ==================================================

    # all proteins
    ax.scatter(
        plot_df["rank"],
        plot_df["score"],
        s=4.5,
        c=COLOR_BG,
        edgecolors="none",
        alpha=0.65,
        zorder=1
    )

    # bottom 50 points
    ax.scatter(
        bottom50_df["rank"],
        bottom50_df["score"],
        s=13,
        c=COLOR_NEG,
        edgecolors="none",
        alpha=0.90,
        zorder=2,
        label=f"Bottom {BOTTOM_N_POINT}"
    )

    # top 50 points
    ax.scatter(
        top50_df["rank"],
        top50_df["score"],
        s=13,
        c=COLOR_POS,
        edgecolors="none",
        alpha=0.90,
        zorder=2,
        label=f"Top {TOP_N_POINT}"
    )

    # selected labeled bottom points
    ax.scatter(
        label_bottom_df["rank"],
        label_bottom_df["score"],
        s=24,
        c=COLOR_NEG,
        edgecolors="black",
        linewidths=0.28,
        alpha=1.0,
        zorder=4
    )

    # selected labeled top points
    ax.scatter(
        label_top_df["rank"],
        label_top_df["score"],
        s=24,
        c=COLOR_POS,
        edgecolors="black",
        linewidths=0.28,
        alpha=1.0,
        zorder=4
    )

    ax.axhline(
        0,
        color="#888888",
        lw=0.55,
        ls="--",
        zorder=0
    )

    # ==================================================
    # Label layout
    # ==================================================

    y_min = np.nanmin(plot_df["score"].values)
    y_max = np.nanmax(plot_df["score"].values)
    y_range = y_max - y_min

    y_pad = y_range * 0.18 + 1e-6

    ax.set_ylim(
        y_min - y_pad,
        y_max + y_pad
    )

    # bottom label y positions
    label_bottom_df = label_bottom_df.sort_values(
        "score",
        ascending=True
    ).reset_index(drop=True)

    bottom_label_y = np.linspace(
        y_min + y_range * 0.020,
        y_min + y_range * 0.420,
        len(label_bottom_df)
    )

    # top label y positions
    label_top_df = label_top_df.sort_values(
        "score",
        ascending=True
    ).reset_index(drop=True)

    top_label_y = np.linspace(
        y_max - y_range * 0.420,
        y_max - y_range * 0.020,
        len(label_top_df)
    )

    # bottom labels
    bottom_x_text = n_total * 0.105

    for (_, row), y_text in zip(label_bottom_df.iterrows(), bottom_label_y):
        ax.plot(
            [row["rank"], bottom_x_text - n_total * 0.012],
            [row["score"], y_text],
            color=COLOR_NEG,
            lw=0.32,
            alpha=0.75,
            zorder=3
        )

        ax.text(
            bottom_x_text,
            y_text,
            row["gene"],
            ha="left",
            va="center",
            fontsize=4.2,
            color=COLOR_NEG,
            fontstyle="italic",
            zorder=5
        )

    # top labels
    top_x_text = n_total * 0.765

    for (_, row), y_text in zip(label_top_df.iterrows(), top_label_y):
        ax.plot(
            [row["rank"], top_x_text + n_total * 0.010],
            [row["score"], y_text],
            color=COLOR_POS,
            lw=0.32,
            alpha=0.75,
            zorder=3
        )

        ax.text(
            top_x_text,
            y_text,
            row["gene"],
            ha="right",
            va="center",
            fontsize=4.2,
            color=COLOR_POS,
            fontstyle="italic",
            zorder=5
        )

    # ==================================================
    # Axis style
    # ==================================================

    ax.set_title(
        endpoint_label,
        fontsize=8.8,
        pad=2.5
    )

    ax.set_xlabel(
        "Protein rank by Δρ",
        fontsize=7.2,
        labelpad=1.5
    )

    ax.set_ylabel(
        "Δ Spearman ρ\nAverage − single-modal mean",
        fontsize=7.2,
        labelpad=1.5
    )

    ax.tick_params(
        axis="both",
        labelsize=6.4,
        width=0.6,
        length=2.0,
        pad=1.5
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_linewidth(0.65)
    ax.spines["bottom"].set_linewidth(0.65)

    
    ax.legend(
        frameon=False,
        fontsize=5.6,
        loc="lower right",
        handletextpad=0.3,
        borderpad=0.1,
        labelspacing=0.30,
        markerscale=0.8
    )

    ax.set_xlim(
        -n_total * 0.015,
        n_total * 1.035
    )

    plt.tight_layout(pad=0.25)

    out_pdf = os.path.join(
        OUT_DIR,
        f"{endpoint}_protein_average_minus_single_modal_score_rankplot_top50_label10.pdf"
    )

    out_svg = out_pdf.replace(
        ".pdf",
        ".svg"
    )

    plt.savefig(
        out_pdf,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.savefig(
        out_svg,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.close()

    print("Saved score rank plot:", out_pdf)
    print("Saved score rank plot:", out_svg)


# ======================================================
# Main
# ======================================================

def main():
    protein_mat = load_protein_matrix()

    df = compute_endpoint_score(
        protein_mat,
        TARGET_ENDPOINT
    )

    plot_score_rank(
        TARGET_ENDPOINT,
        ENDPOINT_LABELS.get(TARGET_ENDPOINT, TARGET_ENDPOINT),
        df
    )

    print("\nAll done.")
    print("Output:", OUT_DIR)


if __name__ == "__main__":
    main()

### Restricted-data analysis not included

This block used superseded raw-input correlation block. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from matplotlib_venn import venn3, venn3_circles


# ======================================================
# Global matplotlib settings
# ======================================================

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"


# ======================================================
# Config
# ======================================================

BASE = "data/private/inputs"

OUT_DIR = os.path.join(
    BASE,
    "fig3_final",
    "protein_four_modal_risk_venn"
)

os.makedirs(OUT_DIR, exist_ok=True)


# ======================================================
# Endpoints
# ======================================================

TARGET_ENDPOINTS = [
    "CHD",
    "istroke",
    "ICVD"
]

ENDPOINT_LABELS = {
    "CHD": "CHD",
    "istroke": "Ischemic stroke",
    "ICVD": "ASCVD"
}


# ======================================================
# Risk models
# ======================================================

RISK_MODELS = {

    "SNP":
        "code_02_02_save/snp_20260202_1432/preds/{endpoint}_snp_TR_test_best_pred.csv",

    "Soma":
        "code_02_02_save/soma_20260202_2006/preds/{endpoint}_soma_TR_test_best_pred.csv",

    "ECG":
        "code_02_02_save/ecg_20260203_1003/preds/{endpoint}_ecg_transformer_h256_l3_test_best_pred.csv",

    "Fundus":
        "code_02_02_save/fundus_20251229_1519/preds/{endpoint}_fundus_vitsmallFT_test_best_pred.csv",
}

MODALITY_ORDER = [
    "SNP",
    "Soma",
    "Fundus",
    "ECG"
]

# ======================================================


# ======================================================

VENN_MODALITIES = [
    "SNP",
    "Soma",
    "Fundus"
]

assert len(VENN_MODALITIES) == 3, "matplotlib-venn only supports 3 sets well."


# ======================================================
# Protein input
# ======================================================

PROTEIN_INPUT = os.path.join(
    BASE,
    "code_3_3_save/soma_single/CHD_soma_TR_best_IG_input.csv"
)


# ======================================================
# Correlation cutoff
# ======================================================

POS_CUTOFF = 0.10
NEG_CUTOFF = -0.10



FDR_CUTOFF = 1.0

MIN_N = 10


# ======================================================
# Figure settings
# ======================================================

FIG_W_MM = 95
FIG_H_MM = 55

MM_TO_INCH = 1 / 25.4

FIGSIZE = (
    FIG_W_MM * MM_TO_INCH,
    FIG_H_MM * MM_TO_INCH
)

VENN_COLORS = {
    "SNP": "#8DA0CB",
    "Soma": "#FC8D62",
    "Fundus": "#66C2A5",
    "ECG": "#A6D854",
}


# ======================================================
# Utils
# ======================================================

def clean_gene(x):

    x = str(x)

    x = re.sub(
        r"\.\d+$",
        "",
        x
    )

    if x.startswith("seq."):

        x = x.split(".")[1]

    x = x.upper()

    return x


def rank_norm(x):

    r = pd.Series(x).rank(
        method="average"
    ).values

    return (
        (r - 1)
        / (len(r) - 1 + 1e-8)
    )


def check_file(path):

    if not os.path.exists(path):

        raise FileNotFoundError(path)


# ======================================================
# Load protein matrix
# ======================================================

def load_protein_matrix():

    check_file(PROTEIN_INPUT)

    df = pd.read_csv(
        PROTEIN_INPUT
    )

    required_cols = [
        "pid",
        "feature",
        "ig_abs"
    ]

    for c in required_cols:

        if c not in df.columns:

            raise ValueError(
                f"Protein input must contain column: {c}. "
                f"Available columns: {list(df.columns)}"
            )

    df["pid"] = df["pid"].astype(int)

    df["feature"] = df["feature"].astype(str)

    df["gene"] = df["feature"].map(
        clean_gene
    )

    df["ig_abs"] = pd.to_numeric(
        df["ig_abs"],
        errors="coerce"
    )

    df = df.dropna(
        subset=[
            "pid",
            "gene",
            "ig_abs"
        ]
    )

    mat = df.pivot_table(
        index="gene",
        columns="pid",
        values="ig_abs",
        aggfunc="mean"
    )

    print(
        "[Protein matrix] genes:",
        mat.shape[0]
    )

    print(
        "[Protein matrix] samples:",
        mat.shape[1]
    )

    return mat


# ======================================================
# Load risk
# ======================================================

def load_risk(endpoint, modality):

    risk_fp = os.path.join(
        BASE,
        RISK_MODELS[modality].format(
            endpoint=endpoint
        )
    )

    check_file(risk_fp)

    risk_df = pd.read_csv(
        risk_fp
    )

    if "ID" in risk_df.columns:

        risk_df = risk_df.rename(
            columns={
                "ID": "pid"
            }
        )

    if "pid" not in risk_df.columns:

        raise ValueError(
            f"No pid column in risk file: {risk_fp}. "
            f"Available columns: {list(risk_df.columns)}"
        )

    if "risk" not in risk_df.columns:

        raise ValueError(
            f"No risk column in risk file: {risk_fp}. "
            f"Available columns: {list(risk_df.columns)}"
        )

    risk_df["pid"] = risk_df["pid"].astype(int)

    risk_df["risk"] = pd.to_numeric(
        risk_df["risk"],
        errors="coerce"
    )

    risk_df = risk_df.dropna(
        subset=[
            "pid",
            "risk"
        ]
    ).copy()

    risk_df["risk_norm"] = rank_norm(
        risk_df["risk"].values
    )

    risk_df = risk_df[[
        "pid",
        "risk_norm"
    ]].copy()

    print(
        f"[{endpoint} | {modality}] risk samples:",
        risk_df["pid"].nunique()
    )

    return risk_df


# ======================================================
# Compute correlation
# ======================================================

def compute_gene_corr(
    protein_mat,
    risk_df,
    endpoint,
    modality
):

    common_pids = sorted(
        list(
            set(protein_mat.columns)
            &
            set(risk_df["pid"])
        )
    )

    print(
        f"[{endpoint} | {modality}] common samples:",
        len(common_pids)
    )

    if len(common_pids) < MIN_N:

        raise ValueError(
            f"Too few common samples for {endpoint} {modality}: {len(common_pids)}"
        )

    sub_mat = protein_mat[
        common_pids
    ]

    risk_df = risk_df[
        risk_df["pid"].isin(
            common_pids
        )
    ].copy()

    risk_df = risk_df.set_index(
        "pid"
    )

    risk_vec = risk_df.loc[
        common_pids,
        "risk_norm"
    ].values.astype(float)

    results = []

    total = len(
        sub_mat.index
    )

    for idx, gene in enumerate(
        sub_mat.index
    ):

        if idx % 500 == 0:

            print(
                f"[{endpoint} | {modality}] {idx}/{total}"
            )

        vals = sub_mat.loc[
            gene,
            common_pids
        ].values.astype(float)

        keep = (
            ~np.isnan(vals)
            &
            ~np.isnan(risk_vec)
        )

        if np.sum(keep) < MIN_N:

            continue

        if (
            np.nanstd(vals[keep]) == 0
            or
            np.nanstd(risk_vec[keep]) == 0
        ):

            continue

        try:

            rho, pval = spearmanr(
                vals[keep],
                risk_vec[keep]
            )

            results.append({
                "gene": gene,
                f"{modality}_rho": rho,
                f"{modality}_pval": pval,
                f"{modality}_n": int(np.sum(keep))
            })

        except Exception:

            continue

    res = pd.DataFrame(
        results
    )

    if len(res) > 0:

        p_col = f"{modality}_pval"
        fdr_col = f"{modality}_FDR"

        res[fdr_col] = multipletests(
            res[p_col].values,
            method="fdr_bh"
        )[1]

    return res


# ======================================================
# Compute all modalities for one endpoint
# ======================================================

def compute_endpoint_corr(
    protein_mat,
    endpoint
):

    corr_list = []

    for modality in MODALITY_ORDER:

        print("\n--------------------------------")
        print(endpoint, modality)
        print("--------------------------------")

        risk_df = load_risk(
            endpoint,
            modality
        )

        corr_df = compute_gene_corr(
            protein_mat,
            risk_df,
            endpoint,
            modality
        )

        corr_list.append(
            corr_df
        )

    merged = corr_list[0]

    for x in corr_list[1:]:

        merged = merged.merge(
            x,
            on="gene",
            how="outer"
        )

    rho_cols = [
        f"{m}_rho"
        for m in MODALITY_ORDER
    ]

    merged["max_abs_rho"] = merged[
        rho_cols
    ].abs().max(
        axis=1
    )

    merged = merged.sort_values(
        "max_abs_rho",
        ascending=False
    )

    out_csv = os.path.join(
        OUT_DIR,
        f"{endpoint}_protein_four_modal_risk_spearman_corr.csv"
    )

    merged.to_csv(
        out_csv,
        index=False
    )

    print(
        "Saved correlation table:",
        out_csv
    )

    return merged


# ======================================================
# Build modality gene sets
# ======================================================

def build_modality_gene_sets(
    corr_df,
    mode
):

    gene_sets = {}

    for modality in VENN_MODALITIES:

        rho_col = f"{modality}_rho"
        fdr_col = f"{modality}_FDR"

        if rho_col not in corr_df.columns:

            raise ValueError(
                f"Missing column: {rho_col}"
            )

        if fdr_col not in corr_df.columns:

            raise ValueError(
                f"Missing column: {fdr_col}"
            )

        sub = corr_df.copy()

        sub = sub[
            sub[rho_col].notna()
            &
            sub[fdr_col].notna()
        ].copy()

        sub = sub[
            sub[fdr_col] <= FDR_CUTOFF
        ].copy()

        if mode == "positive":

            sub = sub[
                sub[rho_col] > POS_CUTOFF
            ].copy()

        elif mode == "negative":

            sub = sub[
                sub[rho_col] < NEG_CUTOFF
            ].copy()

        else:

            raise ValueError(mode)

        gene_sets[modality] = set(
            sub["gene"]
            .dropna()
            .astype(str)
            .tolist()
        )

    return gene_sets


def save_venn_region_gene_tables(
    endpoint,
    mode,
    gene_sets
):

    a, b, c = VENN_MODALITIES

    A = gene_sets[a]
    B = gene_sets[b]
    C = gene_sets[c]

    regions = {
        f"{a}_only": A - B - C,
        f"{b}_only": B - A - C,
        f"{c}_only": C - A - B,
        f"{a}_{b}": (A & B) - C,
        f"{a}_{c}": (A & C) - B,
        f"{b}_{c}": (B & C) - A,
        f"{a}_{b}_{c}": A & B & C,
    }

    rows = []

    for region, genes in regions.items():

        genes = sorted(
            list(genes)
        )

        out_csv = os.path.join(
            OUT_DIR,
            f"{endpoint}_{mode}_{region}_genes.csv"
        )

        pd.DataFrame({
            "gene": genes
        }).to_csv(
            out_csv,
            index=False
        )

        rows.append({
            "endpoint": endpoint,
            "direction": mode,
            "region": region,
            "n_genes": len(genes),
            "genes": ";".join(genes)
        })

    summary = pd.DataFrame(
        rows
    )

    out_summary = os.path.join(
        OUT_DIR,
        f"{endpoint}_{mode}_venn_region_gene_summary.csv"
    )

    summary.to_csv(
        out_summary,
        index=False
    )

    print(
        "Saved Venn region summary:",
        out_summary
    )

    return summary


# ======================================================
# Plot Venn
# ======================================================

def plot_one_venn(
    endpoint,
    endpoint_label,
    mode,
    gene_sets
):

    if mode == "positive":

        subtitle = f"rho > {POS_CUTOFF}"

    else:

        subtitle = f"rho < {NEG_CUTOFF}"

    if FDR_CUTOFF < 1:

        subtitle += f", FDR < {FDR_CUTOFF}"

    labels = VENN_MODALITIES

    sets = [
        gene_sets[labels[0]],
        gene_sets[labels[1]],
        gene_sets[labels[2]],
    ]

    set_labels = [
        f"{m}\n$n$ = {len(gene_sets[m])}"
        for m in labels
    ]

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    v = venn3(
        subsets=sets,
        set_labels=set_labels,
        set_colors=[
            VENN_COLORS[labels[0]],
            VENN_COLORS[labels[1]],
            VENN_COLORS[labels[2]],
        ],
        alpha=0.55,
        ax=ax
    )

    venn3_circles(
        subsets=sets,
        linewidth=0.8,
        color="white",
        ax=ax
    )

    
    for text in v.subset_labels:

        if text is not None:

            text.set_fontsize(8)
            text.set_color("black")

    
    for text in v.set_labels:

        if text is not None:

            text.set_fontsize(8)
            text.set_color("black")

    ax.set_title(
        f"{endpoint_label}\n{subtitle}",
        fontsize=11,
        pad=8
    )

    out_pdf = os.path.join(
        OUT_DIR,
        (
            f"{endpoint}_{mode}_"
            f"{'_'.join(VENN_MODALITIES)}_"
            f"venn_rho_{abs(POS_CUTOFF):.2f}_"
            f"FDR_{FDR_CUTOFF:.2f}.pdf"
        )
    )

    out_svg = out_pdf.replace(
        ".pdf",
        ".svg"
    )

    plt.savefig(
        out_pdf,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.savefig(
        out_svg,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.close()

    print(
        "Saved Venn:",
        out_pdf
    )


def plot_endpoint_venn(
    endpoint,
    endpoint_label,
    corr_df
):

    for mode in [
        "positive",
        "negative"
    ]:

        gene_sets = build_modality_gene_sets(
            corr_df,
            mode
        )

        save_venn_region_gene_tables(
            endpoint,
            mode,
            gene_sets
        )

        plot_one_venn(
            endpoint,
            endpoint_label,
            mode,
            gene_sets
        )


# ======================================================
# Optional: combined positive + negative panel
# ======================================================

def plot_endpoint_venn_combined(
    endpoint,
    endpoint_label,
    corr_df
):

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(FIGSIZE[0] * 1.8, FIGSIZE[1])
    )

    for ax, mode in zip(
        axes,
        ["positive", "negative"]
    ):

        gene_sets = build_modality_gene_sets(
            corr_df,
            mode
        )

        if mode == "positive":

            subtitle = f"rho > {POS_CUTOFF}"

        else:

            subtitle = f"rho < {NEG_CUTOFF}"

        if FDR_CUTOFF < 1:

            subtitle += f", FDR < {FDR_CUTOFF}"

        labels = VENN_MODALITIES

        sets = [
            gene_sets[labels[0]],
            gene_sets[labels[1]],
            gene_sets[labels[2]],
        ]

        set_labels = [
            f"{m}\n$n$ = {len(gene_sets[m])}"
            for m in labels
        ]

        v = venn3(
            subsets=sets,
            set_labels=set_labels,
            set_colors=[
                VENN_COLORS[labels[0]],
                VENN_COLORS[labels[1]],
                VENN_COLORS[labels[2]],
            ],
            alpha=0.55,
            ax=ax
        )

        venn3_circles(
            subsets=sets,
            linewidth=0.8,
            color="white",
            ax=ax
        )

        for text in v.subset_labels:

            if text is not None:

                text.set_fontsize(8)
                text.set_color("black")

        for text in v.set_labels:

            if text is not None:

                text.set_fontsize(8)
                text.set_color("black")

        ax.set_title(
            f"{endpoint_label}\n{subtitle}",
            fontsize=11,
            pad=8
        )

    plt.tight_layout(
        w_pad=2
    )

    out_pdf = os.path.join(
        OUT_DIR,
        (
            f"{endpoint}_positive_negative_"
            f"{'_'.join(VENN_MODALITIES)}_"
            f"venn_combined.pdf"
        )
    )

    out_svg = out_pdf.replace(
        ".pdf",
        ".svg"
    )

    plt.savefig(
        out_pdf,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.savefig(
        out_svg,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.close()

    print(
        "Saved combined Venn:",
        out_pdf
    )


# ======================================================
# Main
# ======================================================

def main():

    protein_mat = load_protein_matrix()

    all_region_summaries = []

    for endpoint in TARGET_ENDPOINTS:

        print("\n====================================")
        print(endpoint)
        print("====================================")

        corr_df = compute_endpoint_corr(
            protein_mat,
            endpoint
        )

        
        plot_endpoint_venn(
            endpoint,
            ENDPOINT_LABELS[endpoint],
            corr_df
        )

        
        plot_endpoint_venn_combined(
            endpoint,
            ENDPOINT_LABELS[endpoint],
            corr_df
        )

        
        for mode in [
            "positive",
            "negative"
        ]:

            summary_fp = os.path.join(
                OUT_DIR,
                f"{endpoint}_{mode}_venn_region_gene_summary.csv"
            )

            if os.path.exists(summary_fp):

                tmp = pd.read_csv(
                    summary_fp
                )

                all_region_summaries.append(
                    tmp
                )

    if len(all_region_summaries) > 0:

        all_df = pd.concat(
            all_region_summaries,
            axis=0,
            ignore_index=True
        )

        out_all = os.path.join(
            OUT_DIR,
            (
                f"ALL_endpoints_"
                f"{'_'.join(VENN_MODALITIES)}_"
                f"venn_region_gene_summary.csv"
            )
        )

        all_df.to_csv(
            out_all,
            index=False
        )

        print(
            "Saved all Venn region summary:",
            out_all
        )

    print("\nAll done.")
    print("Output:", OUT_DIR)


if __name__ == "__main__":

    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ======================================================
# Config
# ======================================================

BASE = "data/private/inputs"

GO_DIR = os.path.join(
    BASE,
    "code_3_3_save/high_vs_low_quadrant_scatter_GO_final",
    "GO"
)

OUT_DIR = "fig4_final"
os.makedirs(OUT_DIR, exist_ok=True)

ENDPOINTS = ["CHD", "istroke"]
FEATURE_TYPE = "protein"

GO_FDR = 0.05

BAR_HEIGHT = 0.75
Y_SPACING = 1.25
ROW_H = 0.55

FIG_W = 7.0
LEFT_MARGIN = 0.42
RIGHT_MARGIN = 0.97

# ======================================================
# Selected GO terms only
# ======================================================

SELECTED_TERMS = {
    "istroke": [
        "Extracellular Matrix Organization",
        "Positive Regulation Of Hydrolase Activity",
        "Regulation Of Biomineral Tissue Development",
        "Positive Regulation Of Peptidyl-Tyrosine Phosphorylation",
        "Regulation Of Steroid Biosynthetic Process",
        "Regulation Of Angiogenesis",
        "Hemostasis",
    ],
    "CHD": [
        "Regulation Of Neurogenesis",
        "Phospholipid Metabolic Process",
        "Regulation Of Endopeptidase Activity",
        "Negative Regulation Of Metabolic Process",
        "Positive Regulation Of MAPK Cascade",
        "Cellular Response To Oxidative Stress",
        "Regulation Of Low-Density Lipoprotein Particle Clearance",
        "Negative Regulation Of Immune System Process",
    ],
}

# ======================================================
# Quadrants
# ======================================================

QUADRANTS = [
    "Fundus_high__ECG_high",
    "Fundus_low__ECG_high",
    "Fundus_low__ECG_low",
    "Fundus_high__ECG_low",
]

QUAD_TITLES = {
    "Fundus_high__ECG_high": "Fundus high / ECG high",
    "Fundus_low__ECG_high": "Fundus low / ECG high",
    "Fundus_low__ECG_low": "Fundus low / ECG low",
    "Fundus_high__ECG_low": "Fundus high / ECG low",
}

QUAD_COLORS = {
    "Fundus_high__ECG_high": "#C44E52",
    "Fundus_low__ECG_high": "#4C72B0",
    "Fundus_low__ECG_low": "#6cb0c9",
    "Fundus_high__ECG_low": "#8172B2",
}


# ======================================================
# Helpers
# ======================================================

def normalize_term(x):
    x = str(x)
    x = re.sub(r"\s*\(GO:\d+\)", "", x)
    x = x.replace("\n", " ")
    x = re.sub(r"\s+", " ", x)
    x = x.strip().lower()
    return x


def clean_term_for_display(x):
    x = str(x)
    x = re.sub(r"\s*\(GO:\d+\)", "", x)

    x = x.replace("Positive Regulation Of ", "Positive reg. ")
    x = x.replace("Negative Regulation Of ", "Negative reg. ")
    x = x.replace("Regulation Of ", "Reg. ")

    x = x.replace("Extracellular Matrix Organization", "Extracellular matrix\norganization")
    x = x.replace("Hydrolase Activity", "Hydrolase\nactivity")
    x = x.replace("Biomineral Tissue Development", "Biomineral tissue\ndevelopment")
    x = x.replace("Peptidyl-Tyrosine Phosphorylation", "Peptidyl-tyrosine\nphosphorylation")
    x = x.replace("Steroid Biosynthetic Process", "Steroid biosynthetic\nprocess")
    x = x.replace("Angiogenesis", "Angiogenesis")
    x = x.replace("Hemostasis", "Hemostasis")

    x = x.replace("Phospholipid Metabolic Process", "Phospholipid metabolic\nprocess")
    x = x.replace("Endopeptidase Activity", "Endopeptidase\nactivity")
    x = x.replace("Metabolic Process", "Metabolic\nprocess")
    x = x.replace("MAPK Cascade", "MAPK cascade")
    x = x.replace("Cellular Response To Oxidative Stress", "Cellular response to\noxidative stress")
    x = x.replace("Low-Density Lipoprotein Particle Clearance", "LDL particle\nclearance")
    x = x.replace("Immune System Process", "Immune system\nprocess")
    x = x.replace("Neurogenesis", "Neurogenesis")

    return x


def genes_key(genes):
    genes = str(genes)
    gs = [
        g.strip()
        for g in genes.replace(",", ";").split(";")
        if g.strip()
    ]
    return ";".join(sorted(set(gs)))


def deduplicate_same_gene_terms(df):
    if df is None or len(df) == 0:
        return df

    df = df.copy()
    df["Genes_key"] = df["Genes"].map(genes_key)
    df["Term_len"] = df["Term"].astype(str).str.len()

    df = (
        df.sort_values(
            ["Genes_key", "Term_len", "Adjusted P-value"],
            ascending=[True, True, True]
        )
        .drop_duplicates("Genes_key", keep="first")
        .drop(columns=["Genes_key", "Term_len"])
    )

    return df


def load_go(endpoint, quadrant):
    go_path = os.path.join(
        GO_DIR,
        endpoint,
        FEATURE_TYPE,
        quadrant,
        f"{endpoint}_{FEATURE_TYPE}_{quadrant}_GO.csv"
    )

    if not os.path.exists(go_path):
        return None

    df = pd.read_csv(go_path)

    if len(df) == 0:
        return None

    df = df[df["Adjusted P-value"] < GO_FDR].copy()

    if len(df) == 0:
        return None

    selected_norm = set([
        normalize_term(x)
        for x in SELECTED_TERMS[endpoint]
    ])

    df["Term_norm"] = df["Term"].map(normalize_term)

    df = df[df["Term_norm"].isin(selected_norm)].copy()

    if len(df) == 0:
        return None

    df = deduplicate_same_gene_terms(df)

    df["minuslog10FDR"] = -np.log10(
        df["Adjusted P-value"] + 1e-300
    )

    
    order_map = {
        normalize_term(t): i
        for i, t in enumerate(SELECTED_TERMS[endpoint])
    }

    df["selected_order"] = df["Term_norm"].map(order_map)

    df = df.sort_values(
        ["selected_order", "Adjusted P-value"],
        ascending=[True, True]
    ).copy()

    df["Term_display"] = df["Term"].map(clean_term_for_display)

    return df


def plot_one(endpoint):

    print("\n================================================")
    print(endpoint, FEATURE_TYPE)
    print("================================================")

    dfs = {}
    xmax = 0
    non_empty_quads = []

    for quad in QUADRANTS:
        df = load_go(endpoint, quad)
        dfs[quad] = df

        if df is not None and len(df) > 0:
            non_empty_quads.append(quad)
            xmax = max(xmax, df["minuslog10FDR"].max())

    if len(non_empty_quads) == 0:
        print("[Skip] no selected significant GO terms")
        return

    xmax = xmax * 1.12

    height_ratios = []
    for quad in non_empty_quads:
        n = len(dfs[quad])
        height_ratios.append(max(1.0, n * ROW_H))

    fig_h = sum(height_ratios) + 1.2

    fig, axes = plt.subplots(
        nrows=len(non_empty_quads),
        ncols=1,
        figsize=(FIG_W, fig_h),
        sharex=True,
        gridspec_kw={"height_ratios": height_ratios}
    )

    if len(non_empty_quads) == 1:
        axes = [axes]

    for ax, quad in zip(axes, non_empty_quads):

        df = dfs[quad]
        color = QUAD_COLORS[quad]

        y = np.arange(len(df)) * Y_SPACING

        ax.barh(
            y=y,
            width=df["minuslog10FDR"].values,
            left=0,
            color=color,
            height=BAR_HEIGHT,
            zorder=1
        )

        ax.set_yticks(y)
        ax.set_yticklabels(
            df["Term_display"].tolist(),
            fontsize=9
        )

        for lab in ax.get_yticklabels():
            lab.set_ha("right")
            lab.set_va("center")
            lab.set_multialignment("right")

        for yi, genes in zip(y, df["Genes"].tolist()):
            genes_txt = str(genes).replace(";", ", ")

            ax.text(
                0.03,
                yi,
                genes_txt,
                ha="left",
                va="center",
                fontsize=7,
                color="black",
                clip_on=True
            )

        ax.set_title(
            QUAD_TITLES[quad],
            fontsize=11,
            color=color,
            pad=8
        )

        ax.set_xlim(0, xmax)
        ax.set_ylim(y[-1] + Y_SPACING * 0.55, -Y_SPACING * 0.55)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.spines["left"].set_linewidth(1.4)
        ax.spines["bottom"].set_linewidth(1.4)

        ax.tick_params(
            axis="both",
            width=1.2,
            length=5
        )

        ax.grid(False)

    axes[-1].set_xlabel(
        r"$-\log_{10}(FDR)$",
        fontsize=12
    )
    '''
    fig.suptitle(
        f"{endpoint} | protein | selected GO Biological Process",
        fontsize=14,
        y=0.995
    )
    '''
    plt.subplots_adjust(
        left=LEFT_MARGIN,
        right=RIGHT_MARGIN,
        top=0.94,
        bottom=0.07,
        hspace=0.55
    )

    out_pdf = os.path.join(
        OUT_DIR,
        f"{endpoint}_protein_selected_GO_4quadrants.pdf"
    )

    out_png = os.path.join(
        OUT_DIR,
        f"{endpoint}_protein_selected_GO_4quadrants.png"
    )
    mpl.rcParams['pdf.fonttype'] = 42
    mpl.rcParams['ps.fonttype'] = 42
    mpl.rcParams['svg.fonttype'] = 'none'
    plt.savefig(
        out_pdf,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.savefig(
        out_png,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.close()

    print("Saved:", out_pdf)


# ======================================================
# Run
# ======================================================

for endpoint in ENDPOINTS:
    plot_one(endpoint)

print("\nDone.")

### Restricted-data analysis not included

This block used superseded raw-input correlation block. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

try:
    import gseapy as gp
    HAS_GSEAPY = True
except Exception:
    HAS_GSEAPY = False


# ======================================================
# Global matplotlib settings
# ======================================================

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

# Make fonts editable in Adobe Illustrator / Inkscape
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

# Reduce Illustrator PDF parsing problems
mpl.rcParams["pdf.compression"] = 0

# Avoid Arial warning
mpl.rcParams["font.family"] = "DejaVu Sans"

# Keep text selectable
mpl.rcParams["text.usetex"] = False


# ======================================================
# Config
# ======================================================

BASE = "data/private/inputs"

CORR_DIR = os.path.join(BASE, "code_3_3_save/risk_feature_correlation")

OUT_DIR = os.path.join(
    "fig3_final",
    "protein_heatmap_Fundus_vs_ECG_same_direction_top_bottom"
)
os.makedirs(OUT_DIR, exist_ok=True)

COLORBAR_DIR = os.path.join(OUT_DIR, "colorbar")
os.makedirs(COLORBAR_DIR, exist_ok=True)

GO_DIR = os.path.join(OUT_DIR, "GO_enrichment")
os.makedirs(GO_DIR, exist_ok=True)

ENDPOINTS = ["CHD", "istroke"]
# ENDPOINTS = ["CHD"]

P_CUTOFF = 1

TOP_N = 100
BOTTOM_N = 100

SMOOTH_SIGMA_X = 5.0

Z_CLIP = 2.0
FIG_DPI = 300

GO_LIBRARIES = [
    "GO_Biological_Process_2023",
    "GO_Molecular_Function_2023",
    "GO_Cellular_Component_2023",
]

RISK_PRED = {
    "ECG": {
        "pred_dir": os.path.join(BASE, "code_02_02_save/ecg_20260203_1003/preds"),
        "pattern": "{endpoint}_ecg_transformer_h256_l3_test_best_pred.csv",
    },
    "Fundus": {
        "pred_dir": os.path.join(BASE, "code_02_02_save/fundus_20251229_1519/preds"),
        "pattern": "{endpoint}_fundus_vitsmallFT_test_best_pred.csv",
    },
}

FEATURE_INPUT = {
    "input_dir": os.path.join(BASE, "code_3_3_save/soma_single"),
    "pattern": "{endpoint}_soma_TR_best_IG_input.csv",
    "value_col": "ig_abs",
}

HIGHLIGHT_GENES = set([
    "ANGPTL3", "ADIPOQ", "VEGFC", "PLAT", "FAS", "IL1RAP", "FGR", "CLEC3B",
    "LDLR", "SORT1", "MMP9", "PLA2G2A", "AXL", "SLAMF8", "TNFRSF8",
    "JAM3", "ICAM4", "CD59", "TIMP3", "TNC",
    "CCL3", "CCL18", "CCL11", "MMP3", "GDF15", "CST3", "BCL2L11",
    "ACVRL1", "CLEC14A", "CD93", "DDR1", "MMP12", "EPO", "HSPB1", "WFDC2"
])


# ======================================================
# Utils
# ======================================================

def rank_norm(x):
    r = pd.Series(x).rank(method="average").values
    return (r - 1) / (len(r) - 1 + 1e-8)


def clean_protein_name(x):
    x = str(x)
    if x.startswith("seq."):
        x = x.replace("seq.", "", 1)
    return x


def load_risk(endpoint, risk_model):
    cfg = RISK_PRED[risk_model]

    path = os.path.join(
        cfg["pred_dir"],
        cfg["pattern"].format(endpoint=endpoint)
    )

    df = pd.read_csv(path)

    if "ID" in df.columns:
        df = df.rename(columns={"ID": "pid"})

    df = df[["pid", "risk", "time", "event"]].copy()
    df["pid"] = df["pid"].astype(int)
    df["risk"] = df["risk"].astype(float)
    df["risk_norm"] = rank_norm(df["risk"].values)

    return df


def load_corr(endpoint, risk_model):
    path = os.path.join(
        CORR_DIR,
        f"{endpoint}_{risk_model}_vs_SomaProtein_pearson.csv"
    )

    corr = pd.read_csv(path)
    corr = corr[corr["pval"] < P_CUTOFF].copy()
    corr = corr.dropna(subset=["feature", "pearson_r", "pval"])
    corr["feature"] = corr["feature"].astype(str)

    return corr


def get_top_and_bottom(corr):
    top_pos = (
        corr[corr["pearson_r"] > 0]
        .sort_values("pearson_r", ascending=False)
        .head(TOP_N)
        .copy()
    )

    top_neg = (
        corr[corr["pearson_r"] < 0]
        .sort_values("pearson_r", ascending=True)
        .head(BOTTOM_N)
        .copy()
    )

    return top_pos, top_neg


def load_common_same_direction_features(endpoint):
    corr_fundus = load_corr(endpoint, "Fundus")
    corr_ecg = load_corr(endpoint, "ECG")

    fundus_top, fundus_bottom = get_top_and_bottom(corr_fundus)
    ecg_top, ecg_bottom = get_top_and_bottom(corr_ecg)

    both_top_features = (
        set(fundus_top["feature"].astype(str)) &
        set(ecg_top["feature"].astype(str))
    )

    both_bottom_features = (
        set(fundus_bottom["feature"].astype(str)) &
        set(ecg_bottom["feature"].astype(str))
    )

    selected_features = sorted(both_top_features | both_bottom_features)

    print(f"\n=== {endpoint} ===")
    print(f"Fundus p<{P_CUTOFF}:", len(corr_fundus))
    print(f"ECG p<{P_CUTOFF}:", len(corr_ecg))
    print(f"Fundus top{TOP_N}:", len(fundus_top))
    print(f"ECG top{TOP_N}:", len(ecg_top))
    print(f"Fundus bottom{BOTTOM_N}:", len(fundus_bottom))
    print(f"ECG bottom{BOTTOM_N}:", len(ecg_bottom))
    print("Both top positive:", len(both_top_features))
    print("Both bottom negative:", len(both_bottom_features))
    print("Selected same-direction features:", len(selected_features))

    if len(selected_features) == 0:
        return [], None

    fundus_sel = pd.concat([fundus_top, fundus_bottom], axis=0)
    ecg_sel = pd.concat([ecg_top, ecg_bottom], axis=0)

    score = (
        fundus_sel[fundus_sel["feature"].isin(selected_features)]
        [["feature", "pearson_r", "pval"]]
        .rename(columns={
            "pearson_r": "fundus_r",
            "pval": "fundus_p"
        })
        .merge(
            ecg_sel[ecg_sel["feature"].isin(selected_features)]
            [["feature", "pearson_r", "pval"]]
            .rename(columns={
                "pearson_r": "ecg_r",
                "pval": "ecg_p"
            }),
            on="feature",
            how="inner"
        )
    )

    score["same_direction_group"] = np.where(
        (score["fundus_r"] > 0) & (score["ecg_r"] > 0),
        "both_positive",
        "both_negative"
    )

    score["mean_abs_r"] = (
        score["fundus_r"].abs() + score["ecg_r"].abs()
    ) / 2

    score["group_order"] = score["same_direction_group"].map({
        "both_positive": 0,
        "both_negative": 1
    })

    score = score.sort_values(
        ["group_order", "mean_abs_r"],
        ascending=[True, False]
    )

    ordered_features = score["feature"].astype(str).tolist()

    return ordered_features, score


def load_expression_matrix(endpoint, features, pids_keep):
    path = os.path.join(
        FEATURE_INPUT["input_dir"],
        FEATURE_INPUT["pattern"].format(endpoint=endpoint)
    )

    print("Loading expression:", path)

    df = pd.read_csv(path)

    df["pid"] = df["pid"].astype(int)
    df["feature"] = df["feature"].astype(str)

    df = df[
        df["feature"].isin(features) &
        df["pid"].isin(set(pids_keep))
    ].copy()

    df[FEATURE_INPUT["value_col"]] = pd.to_numeric(
        df[FEATURE_INPUT["value_col"]],
        errors="coerce"
    )

    df = df.dropna(subset=[FEATURE_INPUT["value_col"]])

    mat = df.pivot_table(
        index="feature",
        columns="pid",
        values=FEATURE_INPUT["value_col"],
        aggfunc="mean"
    )

    return mat


def zscore_rows(mat):
    arr = mat.values.astype(float)

    mean = np.nanmean(arr, axis=1, keepdims=True)
    std = np.nanstd(arr, axis=1, keepdims=True)

    z = (arr - mean) / (std + 1e-8)
    z = np.clip(z, -Z_CLIP, Z_CLIP)

    return z


def smooth_rows_only_x(z):
    return gaussian_filter1d(
        z,
        sigma=SMOOTH_SIGMA_X,
        axis=1,
        mode="nearest"
    )


def prepare_matrix_for_model(mat, risk_df, order_direction):
    if order_direction == "descending":
        risk_df = risk_df.sort_values("risk", ascending=False).copy()
    elif order_direction == "ascending":
        risk_df = risk_df.sort_values("risk", ascending=True).copy()
    else:
        raise ValueError("order_direction must be 'ascending' or 'descending'")

    ordered_pids = risk_df["pid"].astype(int).tolist()
    keep_cols = [p for p in ordered_pids if p in mat.columns]

    mat_model = mat.loc[:, keep_cols]

    z = zscore_rows(mat_model)
    z = smooth_rows_only_x(z)

    risk_bar = (
        risk_df.set_index("pid")
        .loc[mat_model.columns]["risk_norm"]
        .values
        .reshape(1, -1)
    )

    return z, risk_bar, mat_model.columns.tolist()


# ======================================================
# Colorbars
# ======================================================

def save_colorbars():
    fig, ax = plt.subplots(figsize=(1.2, 4))

    norm = mpl.colors.Normalize(vmin=-Z_CLIP, vmax=Z_CLIP)
    sm = mpl.cm.ScalarMappable(cmap="RdBu_r", norm=norm)
    sm.set_array([])

    cbar = plt.colorbar(sm, cax=ax)
    cbar.set_label("Expression z-score", fontsize=10)
    cbar.ax.tick_params(labelsize=8)

    fig.savefig(
        os.path.join(COLORBAR_DIR, "expression_colorbar.pdf"),
        dpi=FIG_DPI,
        transparent=False,
        facecolor="white",
        bbox_inches="tight",
        pad_inches=0.05
    )
    fig.savefig(
        os.path.join(COLORBAR_DIR, "expression_colorbar.png"),
        dpi=FIG_DPI,
        transparent=False,
        facecolor="white",
        bbox_inches="tight",
        pad_inches=0.05
    )
    fig.savefig(
        os.path.join(COLORBAR_DIR, "expression_colorbar.svg"),
        dpi=FIG_DPI,
        transparent=False,
        facecolor="white",
        bbox_inches="tight",
        pad_inches=0.05
    )
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(1.2, 4))

    norm = mpl.colors.Normalize(vmin=0, vmax=1)
    sm = mpl.cm.ScalarMappable(cmap="Greens", norm=norm)
    sm.set_array([])

    cbar = plt.colorbar(sm, cax=ax)
    cbar.set_label("Normalized risk", fontsize=10)
    cbar.ax.tick_params(labelsize=8)

    fig.savefig(
        os.path.join(COLORBAR_DIR, "risk_colorbar.pdf"),
        dpi=FIG_DPI,
        transparent=False,
        facecolor="white",
        bbox_inches="tight",
        pad_inches=0.05
    )
    fig.savefig(
        os.path.join(COLORBAR_DIR, "risk_colorbar.png"),
        dpi=FIG_DPI,
        transparent=False,
        facecolor="white",
        bbox_inches="tight",
        pad_inches=0.05
    )
    fig.savefig(
        os.path.join(COLORBAR_DIR, "risk_colorbar.svg"),
        dpi=FIG_DPI,
        transparent=False,
        facecolor="white",
        bbox_inches="tight",
        pad_inches=0.05
    )
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(1.2, 4))

    norm = mpl.colors.Normalize(vmin=-0.3, vmax=0.3)
    sm = mpl.cm.ScalarMappable(cmap="PuOr_r", norm=norm)
    sm.set_array([])

    cbar = plt.colorbar(sm, cax=ax)
    cbar.set_label("Pearson correlation", fontsize=10)
    cbar.ax.tick_params(labelsize=8)

    fig.savefig(
        os.path.join(COLORBAR_DIR, "correlation_colorbar.pdf"),
        dpi=FIG_DPI,
        transparent=False,
        facecolor="white",
        bbox_inches="tight",
        pad_inches=0.05
    )
    fig.savefig(
        os.path.join(COLORBAR_DIR, "correlation_colorbar.png"),
        dpi=FIG_DPI,
        transparent=False,
        facecolor="white",
        bbox_inches="tight",
        pad_inches=0.05
    )
    fig.savefig(
        os.path.join(COLORBAR_DIR, "correlation_colorbar.svg"),
        dpi=FIG_DPI,
        transparent=False,
        facecolor="white",
        bbox_inches="tight",
        pad_inches=0.05
    )
    plt.close(fig)

    print("Saved colorbars:", COLORBAR_DIR)


# ======================================================
# GO enrichment
# ======================================================

def run_go_enrichment(endpoint, score_df):
    if not HAS_GSEAPY:
        print("[GO skip] gseapy not installed.")
        return

    go_out = os.path.join(GO_DIR, endpoint)
    os.makedirs(go_out, exist_ok=True)

    groups = {
        "both_positive": score_df[score_df["same_direction_group"] == "both_positive"],
        "both_negative": score_df[score_df["same_direction_group"] == "both_negative"],
    }

    for group_name, gdf in groups.items():
        genes = sorted(set([
            clean_protein_name(x)
            for x in gdf["feature"].astype(str).tolist()
        ]))

        genes = [g for g in genes if g and g.lower() != "nan"]

        gene_table = os.path.join(go_out, f"{group_name}_genes.csv")
        pd.DataFrame({"gene": genes}).to_csv(gene_table, index=False)

        print(f"[GO] {endpoint} {group_name}: n_genes={len(genes)}")

        if len(genes) < 5:
            print(f"[GO skip] {endpoint} {group_name}: too few genes")
            continue

        group_out = os.path.join(go_out, group_name)
        os.makedirs(group_out, exist_ok=True)

        for lib in GO_LIBRARIES:
            try:
                enr = gp.enrichr(
                    gene_list=genes,
                    gene_sets=lib,
                    organism="Human",
                    outdir=group_out,
                    cutoff=1.0,
                    no_plot=True
                )

                if enr.results is not None and len(enr.results) > 0:
                    res = enr.results.copy()
                    res.to_csv(
                        os.path.join(group_out, f"{lib}_enrichr_results.csv"),
                        index=False
                    )

                    if "Adjusted P-value" in res.columns:
                        res = (
                            res.sort_values("Adjusted P-value", ascending=True)
                            .head(15)
                            .copy()
                        )
                        res["minus_log10_fdr"] = -np.log10(
                            res["Adjusted P-value"] + 1e-300
                        )

                        fig, ax = plt.subplots(figsize=(6, 4))
                        ax.barh(
                            np.arange(len(res)),
                            res["minus_log10_fdr"].values
                        )
                        ax.set_yticks(np.arange(len(res)))
                        ax.set_yticklabels(res["Term"].values, fontsize=7)
                        ax.invert_yaxis()
                        ax.set_xlabel("-log10(FDR)")
                        ax.set_title(f"{endpoint} {group_name}\n{lib}", fontsize=10)

                        fig.savefig(
                            os.path.join(group_out, f"{lib}_top15_barplot.pdf"),
                            dpi=FIG_DPI,
                            transparent=False,
                            facecolor="white",
                            bbox_inches="tight",
                            pad_inches=0.05
                        )
                        fig.savefig(
                            os.path.join(group_out, f"{lib}_top15_barplot.png"),
                            dpi=FIG_DPI,
                            transparent=False,
                            facecolor="white",
                            bbox_inches="tight",
                            pad_inches=0.05
                        )
                        fig.savefig(
                            os.path.join(group_out, f"{lib}_top15_barplot.svg"),
                            dpi=FIG_DPI,
                            transparent=False,
                            facecolor="white",
                            bbox_inches="tight",
                            pad_inches=0.05
                        )
                        plt.close(fig)

            except Exception as e:
                print(f"[GO failed] {endpoint} {group_name} {lib}: {e}")


# ======================================================
# Plot
# ======================================================

def plot_combined(endpoint, features, score_df, risk_fundus, risk_ecg, mat):
    mat = mat.loc[[f for f in features if f in mat.index]]

    if mat.empty:
        print(f"[Skip] {endpoint}: empty same-direction matrix")
        return

    score_df = score_df.set_index("feature").loc[mat.index].reset_index()

    z_fundus, risk_bar_fundus, cols_fundus = prepare_matrix_for_model(
        mat,
        risk_fundus,
        order_direction="descending"
    )

    z_ecg, risk_bar_ecg, cols_ecg = prepare_matrix_for_model(
        mat,
        risk_ecg,
        order_direction="ascending"
    )

    fundus_corr = score_df["fundus_r"].values.reshape(-1, 1)
    ecg_corr = score_df["ecg_r"].values.reshape(-1, 1)

    corr_lim = max(
        0.05,
        np.nanmax(np.abs(score_df[["fundus_r", "ecg_r"]].values))
    )

    n_features = mat.shape[0]
    n_fundus = len(cols_fundus)
    n_ecg = len(cols_ecg)

    # Important:
    # CHD may have very few proteins, so a pure n_features-based height
    # can make the PDF canvas too flat and unstable in Illustrator.
    fig_h = max(2.8, n_features * 0.18)
    fig_w = 7.5

    fig = plt.figure(figsize=(fig_w, fig_h), constrained_layout=False)

    gs = fig.add_gridspec(
        2,
        5,
        width_ratios=[0.28, 5.0, 1.7, 5.0, 0.28],
        height_ratios=[0.45, 5.0],
        wspace=0.04,
        hspace=0.08
    )

    ax_risk_fundus = fig.add_subplot(gs[0, 1])
    ax_risk_ecg = fig.add_subplot(gs[0, 3])

    ax_corr_fundus = fig.add_subplot(gs[1, 0])
    ax_heat_fundus = fig.add_subplot(gs[1, 1])
    ax_names = fig.add_subplot(gs[1, 2])
    ax_heat_ecg = fig.add_subplot(gs[1, 3])
    ax_corr_ecg = fig.add_subplot(gs[1, 4])

    # -------------------------------
    # Rasterized image layers
    # -------------------------------
    ax_risk_fundus.imshow(
        risk_bar_fundus,
        aspect="auto",
        cmap="gray_r",
        vmin=0,
        vmax=1,
        interpolation="nearest",
        rasterized=True
    )
    ax_risk_fundus.set_title("Fundus risk", fontsize=12, pad=8)
    ax_risk_fundus.set_xticks([])
    ax_risk_fundus.set_yticks([])

    ax_risk_ecg.imshow(
        risk_bar_ecg,
        aspect="auto",
        cmap="gray_r",
        vmin=0,
        vmax=1,
        interpolation="nearest",
        rasterized=True
    )
    ax_risk_ecg.set_title("ECG risk", fontsize=12, pad=8)
    ax_risk_ecg.set_xticks([])
    ax_risk_ecg.set_yticks([])

    ax_corr_fundus.imshow(
        fundus_corr,
        aspect="auto",
        cmap="PuOr_r",
        vmin=-corr_lim,
        vmax=corr_lim,
        interpolation="nearest",
        rasterized=True
    )
    ax_corr_fundus.set_xticks([])
    ax_corr_fundus.set_yticks([])
    ax_corr_fundus.set_ylabel("Fundus\ncorr.", fontsize=8)

    ax_corr_ecg.imshow(
        ecg_corr,
        aspect="auto",
        cmap="PuOr_r",
        vmin=-corr_lim,
        vmax=corr_lim,
        interpolation="nearest",
        rasterized=True
    )
    ax_corr_ecg.set_xticks([])
    ax_corr_ecg.set_yticks([])
    ax_corr_ecg.set_ylabel("ECG\ncorr.", fontsize=8)
    ax_corr_ecg.yaxis.set_label_position("right")

    ax_heat_fundus.imshow(
        z_fundus,
        aspect="auto",
        cmap="RdBu_r",
        vmin=-Z_CLIP,
        vmax=Z_CLIP,
        interpolation="nearest",
        rasterized=True
    )
    ax_heat_fundus.set_xticks([])
    ax_heat_fundus.set_yticks([])

    ax_heat_ecg.imshow(
        z_ecg,
        aspect="auto",
        cmap="RdBu_r",
        vmin=-Z_CLIP,
        vmax=Z_CLIP,
        interpolation="nearest",
        rasterized=True
    )
    ax_heat_ecg.set_xticks([])
    ax_heat_ecg.set_yticks([])

    # -------------------------------
    # Middle protein names
    # These remain vector/editable text.
    # -------------------------------
    ax_names.set_xlim(0, 1)
    ax_names.set_ylim(n_features, 0)
    ax_names.axis("off")

    for i, feat in enumerate(mat.index.tolist()):
        name = clean_protein_name(feat)
        color = "red" if name in HIGHLIGHT_GENES else "black"

        ax_names.text(
            0.5,
            i + 0.5,
            name,
            fontsize=8,
            color=color,
            ha="center",
            va="center",
            fontstyle="italic"
        )

    # Remove unnecessary spines from all axes
    for ax in [
        ax_risk_fundus, ax_risk_ecg,
        ax_corr_fundus, ax_heat_fundus,
        ax_names, ax_heat_ecg, ax_corr_ecg
    ]:
        for spine in ax.spines.values():
            spine.set_linewidth(0.8)

    out_prefix = os.path.join(
        OUT_DIR,
        f"{endpoint}_same_direction_Fundus_left_ECG_right_top{TOP_N}_bottom{BOTTOM_N}_corrbar"
    )

    # Give enough room for titles and side labels.
    fig.subplots_adjust(
        left=0.04,
        right=0.98,
        top=0.86,
        bottom=0.08
    )

    # -------------------------------
    # Save outputs
    # -------------------------------
    # PDF: compatible with Illustrator because imshow layers are rasterized
    # and text remains vector.
    fig.savefig(
        out_prefix + ".pdf",
        dpi=FIG_DPI,
        transparent=False,
        facecolor="white",
        edgecolor="white",
        bbox_inches="tight",
        pad_inches=0.08
    )

    # PNG: reference raster output
    fig.savefig(
        out_prefix + ".png",
        dpi=FIG_DPI,
        transparent=False,
        facecolor="white",
        edgecolor="white",
        bbox_inches="tight",
        pad_inches=0.08
    )

    # SVG: often the safest format for Adobe Illustrator editing
    fig.savefig(
        out_prefix + ".svg",
        dpi=FIG_DPI,
        transparent=False,
        facecolor="white",
        edgecolor="white",
        bbox_inches="tight",
        pad_inches=0.08
    )

    plt.close(fig)

    print("Saved:", out_prefix + ".pdf")
    print("Saved:", out_prefix + ".png")
    print("Saved:", out_prefix + ".svg")
    print(f"n_features={n_features}, n_fundus_samples={n_fundus}, n_ecg_samples={n_ecg}")


# ======================================================
# Main
# ======================================================

def main():
    save_colorbars()

    for endpoint in ENDPOINTS:
        print("\n====================================================")
        print(f"Processing endpoint: {endpoint}")
        print("====================================================")

        features, score_df = load_common_same_direction_features(endpoint)

        if len(features) == 0:
            print(f"[Skip] No same-direction features for {endpoint}")
            continue

        run_go_enrichment(endpoint, score_df)

        risk_fundus = load_risk(endpoint, "Fundus")
        risk_ecg = load_risk(endpoint, "ECG")

        all_pids = sorted(
            set(risk_fundus["pid"].astype(int).tolist()) |
            set(risk_ecg["pid"].astype(int).tolist())
        )

        mat = load_expression_matrix(
            endpoint=endpoint,
            features=features,
            pids_keep=all_pids
        )

        if mat.empty:
            print(f"[Skip] Empty expression matrix for {endpoint}")
            continue

        keep_features = [f for f in features if f in mat.index]
        mat = mat.loc[keep_features]

        score_out = os.path.join(
            OUT_DIR,
            f"{endpoint}_same_direction_top{TOP_N}_bottom{BOTTOM_N}_protein_table.csv"
        )

        score_df.to_csv(score_out, index=False)
        print("Saved selected protein table:", score_out)

        plot_combined(
            endpoint=endpoint,
            features=keep_features,
            score_df=score_df,
            risk_fundus=risk_fundus,
            risk_ecg=risk_ecg,
            mat=mat
        )

    print("\nAll done.")
    print("Output folder:", OUT_DIR)


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import logging
import warnings

import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt


# ======================================================
# Global matplotlib settings
# ======================================================

# AI / Illustrator editable text
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

# Avoid Arial findfont warning
mpl.rcParams["font.family"] = "DejaVu Sans"

# Suppress font warnings
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")


# ======================================================
# Config
# ======================================================

BASE = "data/private/inputs"

INPUT_DIR = os.path.join(
    BASE,
    "code_3_3_save/toy_multimodal_risk_combination"
)

OUT_DIR = os.path.join(
    "fig3_final",
    "improvement"
)

os.makedirs(OUT_DIR, exist_ok=True)


# ======================================================
# Endpoints
# ======================================================

ENDPOINTS = [
    "CHD",
    "istroke",
    "ICVDr"
]

ENDPOINT_LABELS = {
    "CHD": "CHD",
    "istroke": "Ischemic stroke",
    "ICVDr": "ASCVD"
}


# ======================================================
# Fixed modality order
# From top to bottom in each subplot:
# SNP, Soma, Fundus, ECG
# ======================================================

MODEL_ORDER = [
    "SNP",
    "Soma",
    "Fundus",
    "ECG"
]


MODEL_RENAME = {
    "Proteomics": "Soma",
    "Soma": "Soma",
    "SNP": "SNP",
    "Fundus": "Fundus",
    "ECG": "ECG",
    "ToyCombined": "ToyCombined"
}


# ======================================================
# Colors
# ======================================================

AUC_COLOR = "#C44E52"
CINDEX_COLOR = "#4C72B0"


# ======================================================
# Helper
# ======================================================

def load_endpoint(endpoint):
    fp = os.path.join(
        INPUT_DIR,
        f"{endpoint}_toy_combined_metrics.csv"
    )

    df = pd.read_csv(fp)

    # Rename model names for display and ordering
    df["model"] = df["model"].replace(MODEL_RENAME)

    # Convert improvement to percentage
    df["AUC_gain_pct"] = df["AUC_gain_vs_combined"] * 100
    df["Cindex_gain_pct"] = df["Cindex_gain_vs_combined"] * 100

    # Keep only single modalities in fixed order
    single = (
        df[df["model"].isin(MODEL_ORDER)]
        .set_index("model")
        .loc[MODEL_ORDER]
        .reset_index()
    )

    return single


# ======================================================
# Plot function
# ======================================================

def make_plot(metric="AUC"):

    if metric == "AUC":
        color = AUC_COLOR
        gain_col = "AUC_gain_pct"
        xlabel = "Improvement over single modality (%)"
        out_name = "toy_combined_auc_improvement_barh_fixed_order.pdf"
        suptitle = "Improvement after averaging risks (AUC)"

    elif metric == "CINDEX":
        color = CINDEX_COLOR
        gain_col = "Cindex_gain_pct"
        xlabel = "Improvement over single modality (%)"
        out_name = "toy_combined_cindex_improvement_barh_fixed_order.pdf"
        suptitle = "Improvement after averaging risks (C-index)"

    else:
        raise ValueError("metric must be 'AUC' or 'CINDEX'")

    # ==================================================
    # Figure
    # ==================================================

    fig, axes = plt.subplots(
        nrows=3,
        ncols=1,
        figsize=(4.2, 8),
        sharex=True
    )

    # ==================================================
    # Loop endpoints
    # ==================================================

    for ax, endpoint in zip(axes, ENDPOINTS):

        single = load_endpoint(endpoint)

        y = np.arange(len(single))
        vals = single[gain_col].values
        labels = single["model"].values

        # ==============================================
        # barh
        # ==============================================

        ax.barh(
            y,
            vals,
            color=color,
            alpha=0.9,
            height=0.65
        )

        
        
        # SNP, Soma, Fundus, ECG
        ax.invert_yaxis()

        # ==============================================
        # text
        # ==============================================

        x_range = np.nanmax(vals) - np.nanmin(vals)
        text_offset = max(0.02, x_range * 0.02)

        for i, v in enumerate(vals):

            ax.text(
                v + text_offset,
                i,
                f"{v:.2f}%",
                va="center",
                fontsize=10
            )

        # ==============================================
        # axes
        # ==============================================

        ax.set_yticks(y)

        ax.set_yticklabels(
            labels,
            fontsize=11
        )

        ax.set_title(
            ENDPOINT_LABELS[endpoint],
            fontsize=15,
            pad=10
        )

        ax.axvline(
            0,
            linestyle="--",
            color="gray",
            linewidth=1
        )

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.grid(
            axis="x",
            linestyle="--",
            alpha=0.3
        )

        ax.tick_params(
            axis="x",
            labelsize=10
        )

    # ==================================================
    # xlabel
    # ==================================================

    axes[-1].set_xlabel(
        xlabel,
        fontsize=13
    )

    # ==================================================
    # overall title
    # ==================================================

    fig.suptitle(
        suptitle,
        fontsize=18,
        y=0.995
    )

    # ==================================================
    # layout and save
    # ==================================================

    plt.tight_layout(
        rect=[0, 0, 1, 0.98]
    )

    out_pdf = os.path.join(
        OUT_DIR,
        out_name
    )

    plt.savefig(
        out_pdf,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.savefig(
        out_pdf.replace(".pdf", ".svg"),
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.close()

    print("Saved:", out_pdf)
    print("Saved:", out_pdf.replace(".pdf", ".svg"))


# ======================================================
# Run
# ======================================================

if __name__ == "__main__":

    make_plot(metric="AUC")

    make_plot(metric="CINDEX")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from lifelines.utils import concordance_index

# ======================================================
# Config
# ======================================================

BASE = "data/private/inputs"

OUT_DIR = os.path.join(
    BASE,
    "code_3_3_save/toy_multimodal_risk_combination"
)

os.makedirs(OUT_DIR, exist_ok=True)

# ======================================================
# 🔥 NEW: pred dir
# ======================================================

PRED_DIR = os.path.join(
    OUT_DIR,
    "preds"
)

os.makedirs(PRED_DIR, exist_ok=True)

# ======================================================

#ENDPOINTS = ["CHD", "stroke", "istroke", "ICVDr"]
ENDPOINTS = ["ICVD"]
HORIZON = 20

RISK_FILES = {
    "SNP": {
        "dir": os.path.join(BASE, "code_02_02_save/snp_20260202_1432/preds"),
        "pattern": "{endpoint}_snp_TR_test_best_pred.csv"
    },
    "Proteomics": {
        "dir": os.path.join(BASE, "code_02_02_save/soma_20260202_2006/preds"),
        "pattern": "{endpoint}_soma_TR_test_best_pred.csv"
    },
    "ECG": {
        "dir": os.path.join(BASE, "code_02_02_save/ecg_20260203_1003/preds"),
        "pattern": "{endpoint}_ecg_transformer_h256_l3_test_best_pred.csv"
    },
    "Fundus": {
        "dir": os.path.join(BASE, "code_02_02_save/fundus_20251229_1519/preds"),
        "pattern": "{endpoint}_fundus_vitsmallFT_test_best_pred.csv"
    }
}


# ======================================================
# Utils
# ======================================================

def rank_norm(x):

    x = pd.Series(x)

    return x.rank(
        method="average",
        pct=True
    ).values


def load_pred(endpoint, modality):

    cfg = RISK_FILES[modality]

    path = os.path.join(
        cfg["dir"],
        cfg["pattern"].format(endpoint=endpoint)
    )

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    df = pd.read_csv(path)

    if "ID" in df.columns:
        df = df.rename(columns={"ID": "pid"})

    df = df[[
        "pid",
        "time",
        "event",
        "risk"
    ]].copy()

    df["pid"] = df["pid"].astype(int)

    df["risk"] = pd.to_numeric(
        df["risk"],
        errors="coerce"
    )

    df["time"] = pd.to_numeric(
        df["time"],
        errors="coerce"
    )

    df["event"] = pd.to_numeric(
        df["event"],
        errors="coerce"
    ).fillna(0).astype(int)

    df = df.dropna(
        subset=["risk", "time"]
    )

    df = df.rename(columns={
        "risk": f"{modality}_risk",
        "time": f"{modality}_time",
        "event": f"{modality}_event"
    })

    return df


def merge_modalities(endpoint):

    dfs = []

    for modality in RISK_FILES.keys():

        df = load_pred(
            endpoint,
            modality
        )

        dfs.append(df)

    merged = dfs[0]

    for df in dfs[1:]:

        merged = merged.merge(
            df,
            on="pid",
            how="inner"
        )

    first = list(
        RISK_FILES.keys()
    )[0]

    merged["time"] = merged[
        f"{first}_time"
    ]

    merged["event"] = merged[
        f"{first}_event"
    ]

    merged["event_binary"] = (
        (merged["event"] == 1) &
        (merged["time"] <= HORIZON)
    ).astype(int)

    return merged


def compute_auc_cindex(df, risk_col):

    # ==================================================
    # AUC
    # ==================================================

    if df["event_binary"].nunique() < 2:

        auc = np.nan

    else:

        auc = roc_auc_score(
            df["event_binary"],
            df[risk_col]
        )

    # ==================================================
    # C-index
    # ==================================================

    try:

        cindex = concordance_index(
            df["time"],
            -df[risk_col],
            df["event"]
        )

    except Exception:

        cindex = np.nan

    return auc, cindex


def add_toy_combined_risk(df):

    risk_cols = [
        f"{m}_risk"
        for m in RISK_FILES.keys()
    ]

    # ==================================================
    # rank normalize
    # ==================================================

    for c in risk_cols:

        df[c + "_rank"] = rank_norm(
            df[c].values
        )

    rank_cols = [
        c + "_rank"
        for c in risk_cols
    ]

    # ==================================================
    # toy ensemble
    # ==================================================

    df["ToyCombined_risk"] = df[
        rank_cols
    ].mean(axis=1)

    return df


# ======================================================
# Main
# ======================================================

all_results = []

for endpoint in ENDPOINTS:

    print("\n================================================")
    print("Endpoint:", endpoint)
    print("================================================")

    try:

        df = merge_modalities(endpoint)

    except Exception as e:

        print("[Skip]", endpoint, e)

        continue

    # ==================================================
    # add toy combined
    # ==================================================

    df = add_toy_combined_risk(df)

    print("Merged samples:", len(df))

    print(
        "Events within horizon:",
        df["event_binary"].sum()
    )

    endpoint_results = []

    # ==================================================
    # Single modalities
    # ==================================================

    for modality in RISK_FILES.keys():

        risk_col = f"{modality}_risk"

        auc, cindex = compute_auc_cindex(
            df,
            risk_col
        )

        endpoint_results.append({

            "endpoint": endpoint,
            "model": modality,
            "risk_col": risk_col,

            "AUC": auc,
            "Cindex": cindex,

            "n": len(df),

            "n_event_binary":
                int(df["event_binary"].sum()),

            "n_event_survival":
                int(df["event"].sum())
        })

    # ==================================================
    # Toy combined
    # ==================================================

    auc, cindex = compute_auc_cindex(
        df,
        "ToyCombined_risk"
    )

    endpoint_results.append({

        "endpoint": endpoint,
        "model": "ToyCombined",
        "risk_col": "ToyCombined_risk",

        "AUC": auc,
        "Cindex": cindex,

        "n": len(df),

        "n_event_binary":
            int(df["event_binary"].sum()),

        "n_event_survival":
            int(df["event"].sum())
    })

    # ==================================================
    # 🔥 NEW: SAVE PRED FILE
    # ==================================================

    pred_df = df[[
        "pid",
        "ToyCombined_risk",
        "time",
        "event"
    ]].copy()

    pred_df.columns = [
        "ID",
        "risk",
        "time",
        "event"
    ]

    pred_out = os.path.join(
        PRED_DIR,
        f"{endpoint}_toy_combined_pred.csv"
    )

    pred_df.to_csv(
        pred_out,
        index=False
    )

    print(
        "Saved ToyCombined pred:",
        pred_out
    )

    # ==================================================
    # metrics dataframe
    # ==================================================

    res = pd.DataFrame(
        endpoint_results
    )

    # ==================================================
    # improvement
    # ==================================================

    single = res[
        res["model"] != "ToyCombined"
    ].copy()

    combined = res[
        res["model"] == "ToyCombined"
    ].iloc[0]

    best_auc_single = single["AUC"].max()

    best_c_single = single["Cindex"].max()

    res["AUC_gain_vs_combined"] = (
        combined["AUC"] - res["AUC"]
    )

    res["Cindex_gain_vs_combined"] = (
        combined["Cindex"] - res["Cindex"]
    )

    res["ToyCombined_AUC_gain_vs_best_single"] = (
        combined["AUC"] - best_auc_single
    )

    res["ToyCombined_Cindex_gain_vs_best_single"] = (
        combined["Cindex"] - best_c_single
    )

    # ==================================================
    # save metric csv
    # ==================================================

    out_csv = os.path.join(
        OUT_DIR,
        f"{endpoint}_toy_combined_metrics.csv"
    )

    res.to_csv(
        out_csv,
        index=False
    )

    print(res[[
        "model",
        "AUC",
        "Cindex",
        "AUC_gain_vs_combined",
        "Cindex_gain_vs_combined"
    ]])

    print("Saved:", out_csv)

    all_results.append(res)

    # ==================================================
    # merged risk table
    # ==================================================

    merged_out = os.path.join(
        OUT_DIR,
        f"{endpoint}_merged_risks_with_toy_combined.csv"
    )

    df.to_csv(
        merged_out,
        index=False
    )

    print(
        "Saved merged:",
        merged_out
    )
'''
# ======================================================
# Combined result
# ======================================================

if len(all_results) > 0:

    all_df = pd.concat(
        all_results,
        axis=0,
        ignore_index=True
    )

    all_df.to_csv(
        os.path.join(
            OUT_DIR,
            "ALL_endpoints_toy_combined_metrics.csv"
        ),
        index=False
    )

    print("\nSaved all results.")

    print("Output:", OUT_DIR)

else:

    print("No results generated.")
'''

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import itertools
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

# GO enrichment
import gseapy as gp


# ======================================================
# Global matplotlib settings
# ======================================================
PLOT_ZERO_BAR = True
ZERO_BAR_HEIGHT = 0.65
warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"


# ======================================================
# Config
# ======================================================

BASE = "data/private/inputs"

OUT_DIR = os.path.join(
    BASE,
    "fig3_final",
    "protein_four_modal_risk_upset"
)

os.makedirs(OUT_DIR, exist_ok=True)

GO_DIR = os.path.join(
    OUT_DIR,
    "GO_enrichment_intersections"
)

os.makedirs(GO_DIR, exist_ok=True)


# ======================================================
# Endpoints
# ======================================================

TARGET_ENDPOINTS = [
    "CHD",
    "istroke",
    "ICVD"
]

ENDPOINT_LABELS = {
    "CHD": "CHD",
    "istroke": "Ischemic stroke",
    "ICVD": "ASCVD"
}


# ======================================================
# Models / modalities
# ======================================================

# ======================================================

RISK_MODELS = {

    "SNP":
        "code_02_02_save/snp_20260202_1432/preds/{endpoint}_snp_TR_test_best_pred.csv",

    "Soma":
        "code_02_02_save/soma_20260202_2006/preds/{endpoint}_soma_TR_test_best_pred.csv",

    "ECG":
        "code_02_02_save/ecg_20260203_1003/preds/{endpoint}_ecg_transformer_h256_l3_test_best_pred.csv",

    "Fundus":
        "code_02_02_save/fundus_20251229_1519/preds/{endpoint}_fundus_vitsmallFT_test_best_pred.csv",
}

MODALITY_ORDER = [
    "SNP",
    "Soma",
    "Fundus",
    "ECG"
]


# ======================================================
# Protein input
# ======================================================


#   pid
#   feature
#   ig_abs
# ======================================================

PROTEIN_INPUT = os.path.join(
    BASE,
    "code_3_3_save/soma_single/CHD_soma_TR_best_IG_input.csv"
)


# ======================================================
# Correlation cutoff
# ======================================================

POS_CUTOFF = 0.0
NEG_CUTOFF = -0.0


FDR_CUTOFF = 0.2

MIN_N = 10


# ======================================================
# GO enrichment config
# ======================================================

GO_LIBRARY = "GO_Biological_Process_2023"
GO_MIN_GENES = 50
GO_FDR_CUTOFF = 0.05


# ======================================================
# Figure size

# ======================================================

FIG_W_MM = 95
FIG_H_MM = 55

MM_TO_INCH = 1 / 25.4

FIGSIZE = (
    FIG_W_MM * MM_TO_INCH,
    FIG_H_MM * MM_TO_INCH
)


# ======================================================
# Colors
# ======================================================

POS_COLOR = "#B85C5C"
NEG_COLOR = "#5D7FA3"

DOT_COLOR = "#333333"
LINE_COLOR = "#777777"
EMPTY_DOT_COLOR = "#D0D0D0"


# ======================================================
# Utils
# ======================================================

def clean_gene(x):

    x = str(x)

    x = re.sub(
        r"\.\d+$",
        "",
        x
    )

    if x.startswith("seq."):

        x = x.split(".")[1]

    x = x.upper()

    return x


def rank_norm(x):

    r = pd.Series(x).rank(
        method="average"
    ).values

    return (
        (r - 1)
        / (len(r) - 1 + 1e-8)
    )


def check_file(path):

    if not os.path.exists(path):

        raise FileNotFoundError(path)


def safe_name(x):

    x = str(x)

    x = x.replace("|", "_")
    x = x.replace("/", "_")
    x = x.replace(" ", "_")
    x = re.sub(r"[^A-Za-z0-9_\-\.]", "_", x)

    return x


# ======================================================
# Load protein matrix
# ======================================================

#   feature -> gene
#   value = ig_abs
#   matrix = gene x pid
# ======================================================

def load_protein_matrix():

    check_file(PROTEIN_INPUT)

    df = pd.read_csv(
        PROTEIN_INPUT
    )

    required_cols = [
        "pid",
        "feature",
        "ig_abs"
    ]

    for c in required_cols:

        if c not in df.columns:

            raise ValueError(
                f"Protein input must contain column: {c}. "
                f"Available columns: {list(df.columns)}"
            )

    df["pid"] = df["pid"].astype(int)

    df["feature"] = df["feature"].astype(str)

    df["gene"] = df["feature"].map(
        clean_gene
    )

    df["ig_abs"] = pd.to_numeric(
        df["ig_abs"],
        errors="coerce"
    )

    df = df.dropna(
        subset=[
            "pid",
            "gene",
            "ig_abs"
        ]
    )

    mat = df.pivot_table(
        index="gene",
        columns="pid",
        values="ig_abs",
        aggfunc="mean"
    )

    print(
        "[Protein matrix] genes:",
        mat.shape[0]
    )

    print(
        "[Protein matrix] samples:",
        mat.shape[1]
    )

    return mat


# ======================================================
# Load risk
# ======================================================

def load_risk(endpoint, modality):

    risk_fp = os.path.join(
        BASE,
        RISK_MODELS[modality].format(
            endpoint=endpoint
        )
    )

    check_file(risk_fp)

    risk_df = pd.read_csv(
        risk_fp
    )

    if "ID" in risk_df.columns:

        risk_df = risk_df.rename(
            columns={
                "ID": "pid"
            }
        )

    if "pid" not in risk_df.columns:

        raise ValueError(
            f"No pid column in risk file: {risk_fp}. "
            f"Available columns: {list(risk_df.columns)}"
        )

    if "risk" not in risk_df.columns:

        raise ValueError(
            f"No risk column in risk file: {risk_fp}. "
            f"Available columns: {list(risk_df.columns)}"
        )

    risk_df["pid"] = risk_df["pid"].astype(int)

    risk_df["risk"] = pd.to_numeric(
        risk_df["risk"],
        errors="coerce"
    )

    risk_df = risk_df.dropna(
        subset=[
            "pid",
            "risk"
        ]
    ).copy()

    risk_df["risk_norm"] = rank_norm(
        risk_df["risk"].values
    )

    risk_df = risk_df[[
        "pid",
        "risk_norm"
    ]].copy()

    print(
        f"[{endpoint} | {modality}] risk samples:",
        risk_df["pid"].nunique()
    )

    return risk_df


# ======================================================
# Compute correlation
# ======================================================

def compute_gene_corr(
    protein_mat,
    risk_df,
    endpoint,
    modality
):

    common_pids = sorted(
        list(
            set(protein_mat.columns)
            &
            set(risk_df["pid"])
        )
    )

    print(
        f"[{endpoint} | {modality}] common samples:",
        len(common_pids)
    )

    if len(common_pids) < MIN_N:

        raise ValueError(
            f"Too few common samples for {endpoint} {modality}: {len(common_pids)}"
        )

    sub_mat = protein_mat[
        common_pids
    ]

    risk_df = risk_df[
        risk_df["pid"].isin(
            common_pids
        )
    ].copy()

    risk_df = risk_df.set_index(
        "pid"
    )

    risk_vec = risk_df.loc[
        common_pids,
        "risk_norm"
    ].values.astype(float)

    results = []

    total = len(
        sub_mat.index
    )

    for idx, gene in enumerate(
        sub_mat.index
    ):

        if idx % 500 == 0:

            print(
                f"[{endpoint} | {modality}] {idx}/{total}"
            )

        vals = sub_mat.loc[
            gene,
            common_pids
        ].values.astype(float)

        keep = (
            ~np.isnan(vals)
            &
            ~np.isnan(risk_vec)
        )

        if np.sum(keep) < MIN_N:

            continue

        if (
            np.nanstd(vals[keep]) == 0
            or
            np.nanstd(risk_vec[keep]) == 0
        ):

            continue

        try:

            rho, pval = spearmanr(
                vals[keep],
                risk_vec[keep]
            )

            results.append({
                "gene": gene,
                f"{modality}_rho": rho,
                f"{modality}_pval": pval,
                f"{modality}_n": int(np.sum(keep))
            })

        except Exception:

            continue

    res = pd.DataFrame(
        results
    )

    if len(res) > 0:

        p_col = f"{modality}_pval"
        fdr_col = f"{modality}_FDR"

        res[fdr_col] = multipletests(
            res[p_col].values,
            method="fdr_bh"
        )[1]

    return res


# ======================================================
# Compute all modalities for one endpoint
# ======================================================

def compute_endpoint_corr(
    protein_mat,
    endpoint
):

    corr_list = []

    for modality in MODALITY_ORDER:

        print("\n--------------------------------")
        print(endpoint, modality)
        print("--------------------------------")

        risk_df = load_risk(
            endpoint,
            modality
        )

        corr_df = compute_gene_corr(
            protein_mat,
            risk_df,
            endpoint,
            modality
        )

        corr_list.append(
            corr_df
        )

    merged = corr_list[0]

    for x in corr_list[1:]:

        merged = merged.merge(
            x,
            on="gene",
            how="outer"
        )

    rho_cols = [
        f"{m}_rho"
        for m in MODALITY_ORDER
    ]

    merged["max_abs_rho"] = merged[
        rho_cols
    ].abs().max(
        axis=1
    )

    merged = merged.sort_values(
        "max_abs_rho",
        ascending=False
    )

    out_csv = os.path.join(
        OUT_DIR,
        f"{endpoint}_protein_four_modal_risk_spearman_corr.csv"
    )

    merged.to_csv(
        out_csv,
        index=False
    )

    print(
        "Saved correlation table:",
        out_csv
    )

    return merged


# ======================================================
# UpSet statistics
# ======================================================

def all_combinations_ordered():

    combos = []

    for degree in [
        4,
        3,
        2,
        1
    ]:

        for comb in itertools.combinations(
            MODALITY_ORDER,
            degree
        ):

            combos.append(
                tuple(comb)
            )

    return combos


def get_signature(
    row,
    mode
):

    active = []

    for modality in MODALITY_ORDER:

        rho = row.get(
            f"{modality}_rho",
            np.nan
        )

        fdr = row.get(
            f"{modality}_FDR",
            np.nan
        )

        if pd.isna(rho) or pd.isna(fdr):

            continue

        if fdr > FDR_CUTOFF:

            continue

        if mode == "positive":

            if rho > POS_CUTOFF:

                active.append(
                    modality
                )

        elif mode == "negative":

            if rho < NEG_CUTOFF:

                active.append(
                    modality
                )

        else:

            raise ValueError(mode)

    return tuple(active)


def build_upset_counts(
    corr_df,
    endpoint,
    mode
):

    df = corr_df.copy()

    df["signature_tuple"] = df.apply(
        lambda row: get_signature(
            row,
            mode
        ),
        axis=1
    )

    df = df[
        df["signature_tuple"].map(len) > 0
    ].copy()

    df["signature"] = df[
        "signature_tuple"
    ].map(
        lambda x: "|".join(x)
    )

    out_list = os.path.join(
        OUT_DIR,
        (
            f"{endpoint}_{mode}_"
            f"rho_{abs(POS_CUTOFF):.2f}_"
            f"FDR_{FDR_CUTOFF:.2f}_protein_list.csv"
        )
    )

    df.to_csv(
        out_list,
        index=False
    )

    print(
        f"Saved {mode} protein list:",
        out_list
    )

    rows = []

    for comb in all_combinations_ordered():

        comb_txt = "|".join(comb)

        count = int(
            (
                df["signature_tuple"]
                ==
                comb
            ).sum()
        )

        rows.append({
            "endpoint": endpoint,
            "direction": mode,
            "combination": comb_txt,
            "degree": len(comb),
            "count": count
        })

    count_df = pd.DataFrame(
        rows
    )

    out_count = os.path.join(
        OUT_DIR,
        (
            f"{endpoint}_{mode}_"
            f"rho_{abs(POS_CUTOFF):.2f}_"
            f"FDR_{FDR_CUTOFF:.2f}_upset_counts.csv"
        )
    )

    count_df.to_csv(
        out_count,
        index=False
    )

    print(
        f"Saved {mode} counts:",
        out_count
    )

    return count_df, df


# ======================================================
# Plot manual UpSet-style panel
# ======================================================


def plot_upset_panel(
    fig,
    gs_cell,
    count_df,
    direction,
    title
):

    if direction == "positive":

        color = POS_COLOR
        subtitle = f"rho > {POS_CUTOFF}"

    else:

        color = NEG_COLOR
        subtitle = f"rho < {NEG_CUTOFF}"

    if FDR_CUTOFF < 1:

        subtitle += f", FDR < {FDR_CUTOFF}"

    sub_gs = gs_cell.subgridspec(
        2,
        1,
        height_ratios=[
            2.4,
            1.15
        ],
        hspace=0.02
    )

    ax_bar = fig.add_subplot(
        sub_gs[0, 0]
    )

    ax_dot = fig.add_subplot(
        sub_gs[1, 0]
    )

    # ==================================================
    
    
    # ==================================================

    count_df = count_df.copy()

    if len(count_df) == 0:

        ax_bar.text(
            0.5,
            0.5,
            "No intersections",
            ha="center",
            va="center",
            fontsize=5
        )

        ax_bar.axis("off")
        ax_dot.axis("off")

        return

    x = np.arange(
        len(count_df)
    )

    counts = count_df[
        "count"
    ].values.astype(float)

    combos = count_df[
        "combination"
    ].tolist()

    # ==================================================
    
    
    # ==================================================

    plot_counts = counts.copy()

    if PLOT_ZERO_BAR:

        plot_counts[
            plot_counts == 0
        ] = ZERO_BAR_HEIGHT

    # bar colors
    bar_colors = []

    for c in counts:

        if c == 0:

            bar_colors.append(
                "#E0E0E0"
            )

        else:

            bar_colors.append(
                color
            )

    ax_bar.bar(
        x,
        plot_counts,
        color=bar_colors,
        width=0.70,
        linewidth=0
    )

    # --------------------------------------------------
    # log y-axis
    # --------------------------------------------------

    ax_bar.set_yscale(
        "log"
    )

    ymin = ZERO_BAR_HEIGHT * 0.75

    ymax = np.max(
        plot_counts
    ) * 2.2

    if ymax < 10:

        ymax = 10

    ax_bar.set_ylim(
        ymin,
        ymax
    )

    # count labels
    for xi, c, pc in zip(
        x,
        counts,
        plot_counts
    ):

        if c == 0:

            label_y = pc * 1.10

        else:

            label_y = pc * 1.08

        ax_bar.text(
            xi,
            label_y,
            str(int(c)),
            ha="center",
            va="bottom",
            fontsize=5.0,
            color="black"
        )

    ax_bar.set_title(
        f"{title}\n{subtitle}",
        fontsize=7.2,
        pad=3
    )

    ax_bar.set_ylabel(
        "N",
        fontsize=6
    )

    ax_bar.set_xticks([])

    ax_bar.tick_params(
        axis="y",
        labelsize=5.5,
        length=2
    )

    for s in [
        "top",
        "right"
    ]:

        ax_bar.spines[s].set_visible(False)

    ax_bar.spines["left"].set_linewidth(0.8)
    ax_bar.spines["bottom"].set_linewidth(0.8)

    # ==================================================
    # Dot matrix
    # ==================================================

    y_pos = {
        m: i
        for i, m in enumerate(
            MODALITY_ORDER
        )
    }

    ax_dot.set_xlim(
        -0.5,
        len(count_df) - 0.5
    )

    ax_dot.set_ylim(
        -0.5,
        len(MODALITY_ORDER) - 0.5
    )

    ax_dot.invert_yaxis()

    for xi, comb_txt in zip(
        x,
        combos
    ):

        comb = comb_txt.split("|")

        active_y = [
            y_pos[m]
            for m in comb
        ]

        for m in MODALITY_ORDER:

            ax_dot.scatter(
                xi,
                y_pos[m],
                s=13,
                color=EMPTY_DOT_COLOR,
                zorder=1
            )

        if len(active_y) >= 2:

            ax_dot.plot(
                [xi, xi],
                [
                    min(active_y),
                    max(active_y)
                ],
                color=LINE_COLOR,
                linewidth=0.8,
                zorder=2
            )

        for m in comb:

            ax_dot.scatter(
                xi,
                y_pos[m],
                s=16,
                color=DOT_COLOR,
                zorder=3
            )

    ax_dot.set_yticks(
        list(y_pos.values())
    )

    ax_dot.set_yticklabels(
        MODALITY_ORDER,
        fontsize=7.0
    )

    ax_dot.set_xticks([])

    ax_dot.tick_params(
        length=0
    )

    for s in [
        "top",
        "right",
        "left",
        "bottom"
    ]:

        ax_dot.spines[s].set_visible(False)






def plot_endpoint_upset(
    endpoint,
    endpoint_label,
    pos_counts,
    neg_counts
):

    fig = plt.figure(
        figsize=FIGSIZE
    )

    gs = fig.add_gridspec(
        1,
        2,
        wspace=0.32
    )

    plot_upset_panel(
        fig,
        gs[0, 0],
        pos_counts,
        "positive",
        endpoint_label
    )

    plot_upset_panel(
        fig,
        gs[0, 1],
        neg_counts,
        "negative",
        endpoint_label
    )

    out_pdf = os.path.join(
        OUT_DIR,
        (
            f"{endpoint}_protein_risk_upset_pos_neg_"
            f"{FIG_W_MM}x{FIG_H_MM}mm_logy.pdf"
        )
    )

    out_svg = out_pdf.replace(
        ".pdf",
        ".svg"
    )

    plt.savefig(
        out_pdf,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.savefig(
        out_svg,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.close()

    print(
        "Saved UpSet:",
        out_pdf
    )


# ======================================================
# GO enrichment
# ======================================================

def run_go_for_one_intersection(
    genes,
    endpoint,
    direction,
    combination
):

    genes = [
        clean_gene(g)
        for g in genes
    ]

    genes = sorted(
        list(
            set(
                [
                    g for g in genes
                    if g not in ["", "NAN", "NONE"]
                ]
            )
        )
    )

    n_genes = len(genes)

    if n_genes <= GO_MIN_GENES:

        return None

    combo_safe = safe_name(
        combination
    )

    prefix = (
        f"{endpoint}_{direction}_{combo_safe}_n{n_genes}"
    )

    out_dir = os.path.join(
        GO_DIR,
        endpoint,
        direction,
        combo_safe
    )

    os.makedirs(
        out_dir,
        exist_ok=True
    )

    # save input genes
    gene_out = os.path.join(
        out_dir,
        f"{prefix}_input_genes.csv"
    )

    pd.DataFrame({
        "gene": genes
    }).to_csv(
        gene_out,
        index=False
    )

    print(
        f"[GO] {endpoint} | {direction} | {combination} | n={n_genes}"
    )

    try:

        enr = gp.enrichr(
            gene_list=genes,
            gene_sets=GO_LIBRARY,
            organism="Human",
            outdir=None,
            cutoff=1.0,
            no_plot=True
        )

        res = enr.results.copy()

        if res is None or len(res) == 0:

            return None

        # Add metadata
        res.insert(
            0,
            "endpoint",
            endpoint
        )

        res.insert(
            1,
            "direction",
            direction
        )

        res.insert(
            2,
            "combination",
            combination
        )

        res.insert(
            3,
            "n_input_genes",
            n_genes
        )

        
        
        if "Genes" not in res.columns:

            possible_gene_cols = [
                c for c in res.columns
                if c.lower() == "genes"
            ]

            if len(possible_gene_cols) > 0:

                res = res.rename(
                    columns={
                        possible_gene_cols[0]: "Genes"
                    }
                )

        raw_out = os.path.join(
            out_dir,
            f"{prefix}_GO_raw.csv"
        )

        res.to_csv(
            raw_out,
            index=False
        )

        if "Adjusted P-value" in res.columns:

            sig = res[
                res["Adjusted P-value"] < GO_FDR_CUTOFF
            ].copy()

        else:

            sig = res.copy()

        sig_out = os.path.join(
            out_dir,
            f"{prefix}_GO_FDR_{GO_FDR_CUTOFF:.2f}.csv"
        )

        sig.to_csv(
            sig_out,
            index=False
        )

        print(
            "Saved GO raw:",
            raw_out
        )

        print(
            "Saved GO filtered:",
            sig_out
        )

        return res

    except Exception as e:

        print(
            "[GO ERROR]",
            endpoint,
            direction,
            combination,
            e
        )

        return None


def run_go_for_large_intersections(
    all_signature_tables
):

    if len(all_signature_tables) == 0:

        print("[GO] No signature table.")

        return

    sig_all = pd.concat(
        all_signature_tables,
        axis=0,
        ignore_index=True
    )

    sig_all = sig_all[
        sig_all["signature"].notna()
    ].copy()

    sig_all = sig_all[
        sig_all["signature"] != ""
    ].copy()

    all_go = []

    group_cols = [
        "endpoint",
        "direction",
        "signature"
    ]

    for (
        endpoint,
        direction,
        signature
    ), sub in sig_all.groupby(
        group_cols
    ):

        genes = sorted(
            sub["gene"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        if len(genes) <= GO_MIN_GENES:

            continue

        res = run_go_for_one_intersection(
            genes,
            endpoint,
            direction,
            signature
        )

        if res is not None and len(res) > 0:

            all_go.append(
                res
            )

    if len(all_go) > 0:

        all_go_df = pd.concat(
            all_go,
            axis=0,
            ignore_index=True
        )

        out_all = os.path.join(
            GO_DIR,
            (
                f"ALL_large_intersections_"
                f"GO_{GO_LIBRARY}_raw.csv"
            )
        )

        all_go_df.to_csv(
            out_all,
            index=False
        )

        if "Adjusted P-value" in all_go_df.columns:

            all_sig = all_go_df[
                all_go_df["Adjusted P-value"] < GO_FDR_CUTOFF
            ].copy()

            out_sig = os.path.join(
                GO_DIR,
                (
                    f"ALL_large_intersections_"
                    f"GO_{GO_LIBRARY}_FDR_{GO_FDR_CUTOFF:.2f}.csv"
                )
            )

            all_sig.to_csv(
                out_sig,
                index=False
            )

            print(
                "[GO] Saved combined significant GO:",
                out_sig
            )

        print(
            "[GO] Saved combined raw GO:",
            out_all
        )

    else:

        print(
            "[GO] No large intersections with successful enrichment."
        )


# ======================================================
# Main
# ======================================================

def main():
    protein_mat = load_protein_matrix()

    all_counts = []
    all_signature_tables = []

    for endpoint in TARGET_ENDPOINTS:

        print("\n====================================")
        print(endpoint)
        print("====================================")

        corr_df = compute_endpoint_corr(
            protein_mat,
            endpoint
        )

        pos_counts, pos_sig = build_upset_counts(
            corr_df,
            endpoint,
            "positive"
        )

        neg_counts, neg_sig = build_upset_counts(
            corr_df,
            endpoint,
            "negative"
        )

        plot_endpoint_upset(
            endpoint,
            ENDPOINT_LABELS[endpoint],
            pos_counts,
            neg_counts
        )

        all_counts.append(
            pos_counts
        )

        all_counts.append(
            neg_counts
        )

        all_signature_tables.append(
            pos_sig
        )

        all_signature_tables.append(
            neg_sig
        )

    # ==================================================
    # Save combined counts
    # ==================================================

    all_counts_df = pd.concat(
        all_counts,
        axis=0,
        ignore_index=True
    )

    out_all_counts = os.path.join(
        OUT_DIR,
        (
            f"ALL_endpoints_positive_negative_"
            f"rho_{abs(POS_CUTOFF):.2f}_"
            f"FDR_{FDR_CUTOFF:.2f}_upset_counts.csv"
        )
    )

    all_counts_df.to_csv(
        out_all_counts,
        index=False
    )

    print(
        "Combined counts:",
        out_all_counts
    )

    # ==================================================
    # Save combined signature-gene table
    # ==================================================

    all_sig_df = pd.concat(
        all_signature_tables,
        axis=0,
        ignore_index=True
    )

    out_all_sig = os.path.join(
        OUT_DIR,
        (
            f"ALL_endpoints_positive_negative_"
            f"rho_{abs(POS_CUTOFF):.2f}_"
            f"FDR_{FDR_CUTOFF:.2f}_signature_gene_table.csv"
        )
    )

    all_sig_df.to_csv(
        out_all_sig,
        index=False
    )

    print(
        "Combined signature-gene table:",
        out_all_sig
    )

    # ==================================================
    # GO enrichment after all plots
    # ==================================================

    print("\n====================================")
    print("GO enrichment for intersections with > 50 genes")
    print("====================================")

    run_go_for_large_intersections(
        all_signature_tables
    )

    print("\nAll done.")
    print("Output:", OUT_DIR)
    print("GO output:", GO_DIR)


if __name__ == "__main__":

    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import itertools
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

# Four-set Venn
# pip install venn
try:
    from venn import venn
except ImportError:
    raise ImportError(
        "Please install four-set venn package first:\n"
        "pip install venn"
    )

# GO enrichment
import gseapy as gp


# ======================================================
# Global matplotlib settings
# ======================================================

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"


# ======================================================
# Config
# ======================================================

BASE = "data/private/inputs"

OUT_DIR = os.path.join(
    BASE,
    "fig3_final",
    "protein_four_modal_risk_4set_venn"
)

os.makedirs(OUT_DIR, exist_ok=True)

VENN_DIR = os.path.join(
    OUT_DIR,
    "venn_figures"
)

os.makedirs(VENN_DIR, exist_ok=True)

REGION_DIR = os.path.join(
    OUT_DIR,
    "venn_region_genes"
)

os.makedirs(REGION_DIR, exist_ok=True)

GO_DIR = os.path.join(
    OUT_DIR,
    "GO_enrichment_intersections"
)

os.makedirs(GO_DIR, exist_ok=True)


# ======================================================
# Endpoints
# ======================================================

TARGET_ENDPOINTS = [
    "CHD",
    "istroke",
    "ICVD"
]

ENDPOINT_LABELS = {
    "CHD": "CHD",
    "istroke": "Ischemic stroke",
    "ICVD": "ASCVD"
}


# ======================================================
# Risk models
# ======================================================

RISK_MODELS = {

    "SNP":
        "code_02_02_save/snp_20260202_1432/preds/{endpoint}_snp_TR_test_best_pred.csv",

    "Soma":
        "code_02_02_save/soma_20260202_2006/preds/{endpoint}_soma_TR_test_best_pred.csv",

    "Fundus":
        "code_02_02_save/fundus_20251229_1519/preds/{endpoint}_fundus_vitsmallFT_test_best_pred.csv",

    "ECG":
        "code_02_02_save/ecg_20260203_1003/preds/{endpoint}_ecg_transformer_h256_l3_test_best_pred.csv",
}

MODALITY_ORDER = [
    "SNP",
    "Soma",
    "Fundus",
    "ECG"
]


# ======================================================
# Protein input


#   pid
#   feature
#   ig_abs
# ======================================================

PROTEIN_INPUT = os.path.join(
    BASE,
    "code_3_3_save/soma_single/CHD_soma_TR_best_IG_input.csv"
)


# ======================================================
# Correlation cutoff
# ======================================================

POS_CUTOFF = 0.10
NEG_CUTOFF = -0.10



FDR_CUTOFF = 1.0

MIN_N = 10


# ======================================================
# GO enrichment config
# ======================================================

GO_LIBRARY = "GO_Biological_Process_2023"
GO_MIN_GENES = 50
GO_FDR_CUTOFF = 0.05


# ======================================================
# Figure size
# ======================================================

FIG_W_MM = 120
FIG_H_MM = 70

MM_TO_INCH = 1 / 25.4

FIGSIZE_SINGLE = (
    FIG_W_MM * MM_TO_INCH,
    FIG_H_MM * MM_TO_INCH
)

FIGSIZE_COMBINED = (
    FIG_W_MM * 2.0 * MM_TO_INCH,
    FIG_H_MM * MM_TO_INCH
)


# ======================================================
# Colors
# ======================================================
'''
VENN_COLORS = {
    "SNP": "#8DA0CB",
    "Soma": "#FC8D62",
    "Fundus": "#66C2A5",
    "ECG": "#E78AC3",
}
'''

VENN_COLORS = {
    "Fundus": "#778368",
    "ECG": "#658a9a",
    "Soma": "#9382a0",
    "SNP": "#9a6c6c",
}


# ======================================================
# Utils
# ======================================================

def clean_gene(x):

    x = str(x)

    x = re.sub(
        r"\.\d+$",
        "",
        x
    )

    if x.startswith("seq."):

        x = x.split(".")[1]

    x = x.upper()

    return x


def rank_norm(x):

    r = pd.Series(x).rank(
        method="average"
    ).values

    return (
        (r - 1)
        / (len(r) - 1 + 1e-8)
    )


def check_file(path):

    if not os.path.exists(path):

        raise FileNotFoundError(path)


def safe_name(x):

    x = str(x)
    x = x.replace("|", "_")
    x = x.replace("/", "_")
    x = x.replace(" ", "_")
    x = re.sub(r"[^A-Za-z0-9_\-\.]", "_", x)

    return x


# ======================================================
# Load protein matrix
# ======================================================

def load_protein_matrix():

    check_file(PROTEIN_INPUT)

    df = pd.read_csv(
        PROTEIN_INPUT
    )

    required_cols = [
        "pid",
        "feature",
        "ig_abs"
    ]

    for c in required_cols:

        if c not in df.columns:

            raise ValueError(
                f"Protein input must contain column: {c}. "
                f"Available columns: {list(df.columns)}"
            )

    df["pid"] = df["pid"].astype(int)

    df["feature"] = df["feature"].astype(str)

    df["gene"] = df["feature"].map(
        clean_gene
    )

    df["ig_abs"] = pd.to_numeric(
        df["ig_abs"],
        errors="coerce"
    )

    df = df.dropna(
        subset=[
            "pid",
            "gene",
            "ig_abs"
        ]
    )

    mat = df.pivot_table(
        index="gene",
        columns="pid",
        values="ig_abs",
        aggfunc="mean"
    )

    print(
        "[Protein matrix] genes:",
        mat.shape[0]
    )

    print(
        "[Protein matrix] samples:",
        mat.shape[1]
    )

    return mat


# ======================================================
# Load risk
# ======================================================

def load_risk(endpoint, modality):

    risk_fp = os.path.join(
        BASE,
        RISK_MODELS[modality].format(
            endpoint=endpoint
        )
    )

    check_file(risk_fp)

    risk_df = pd.read_csv(
        risk_fp
    )

    if "ID" in risk_df.columns:

        risk_df = risk_df.rename(
            columns={
                "ID": "pid"
            }
        )

    if "pid" not in risk_df.columns:

        raise ValueError(
            f"No pid column in risk file: {risk_fp}. "
            f"Available columns: {list(risk_df.columns)}"
        )

    if "risk" not in risk_df.columns:

        raise ValueError(
            f"No risk column in risk file: {risk_fp}. "
            f"Available columns: {list(risk_df.columns)}"
        )

    risk_df["pid"] = risk_df["pid"].astype(int)

    risk_df["risk"] = pd.to_numeric(
        risk_df["risk"],
        errors="coerce"
    )

    risk_df = risk_df.dropna(
        subset=[
            "pid",
            "risk"
        ]
    ).copy()

    risk_df["risk_norm"] = rank_norm(
        risk_df["risk"].values
    )

    risk_df = risk_df[[
        "pid",
        "risk_norm"
    ]].copy()

    print(
        f"[{endpoint} | {modality}] risk samples:",
        risk_df["pid"].nunique()
    )

    return risk_df


# ======================================================
# Compute correlation
# ======================================================

def compute_gene_corr(
    protein_mat,
    risk_df,
    endpoint,
    modality
):

    common_pids = sorted(
        list(
            set(protein_mat.columns)
            &
            set(risk_df["pid"])
        )
    )

    print(
        f"[{endpoint} | {modality}] common samples:",
        len(common_pids)
    )

    if len(common_pids) < MIN_N:

        raise ValueError(
            f"Too few common samples for {endpoint} {modality}: {len(common_pids)}"
        )

    sub_mat = protein_mat[
        common_pids
    ]

    risk_df = risk_df[
        risk_df["pid"].isin(
            common_pids
        )
    ].copy()

    risk_df = risk_df.set_index(
        "pid"
    )

    risk_vec = risk_df.loc[
        common_pids,
        "risk_norm"
    ].values.astype(float)

    results = []

    total = len(
        sub_mat.index
    )

    for idx, gene in enumerate(
        sub_mat.index
    ):

        if idx % 500 == 0:

            print(
                f"[{endpoint} | {modality}] {idx}/{total}"
            )

        vals = sub_mat.loc[
            gene,
            common_pids
        ].values.astype(float)

        keep = (
            ~np.isnan(vals)
            &
            ~np.isnan(risk_vec)
        )

        if np.sum(keep) < MIN_N:

            continue

        if (
            np.nanstd(vals[keep]) == 0
            or
            np.nanstd(risk_vec[keep]) == 0
        ):

            continue

        try:

            rho, pval = spearmanr(
                vals[keep],
                risk_vec[keep]
            )

            results.append({
                "gene": gene,
                f"{modality}_rho": rho,
                f"{modality}_pval": pval,
                f"{modality}_n": int(np.sum(keep))
            })

        except Exception:

            continue

    res = pd.DataFrame(
        results
    )

    if len(res) > 0:

        p_col = f"{modality}_pval"
        fdr_col = f"{modality}_FDR"

        res[fdr_col] = multipletests(
            res[p_col].values,
            method="fdr_bh"
        )[1]

    return res


# ======================================================
# Compute all modalities for one endpoint
# ======================================================

def compute_endpoint_corr(
    protein_mat,
    endpoint
):

    corr_list = []

    for modality in MODALITY_ORDER:

        print("\n--------------------------------")
        print(endpoint, modality)
        print("--------------------------------")

        risk_df = load_risk(
            endpoint,
            modality
        )

        corr_df = compute_gene_corr(
            protein_mat,
            risk_df,
            endpoint,
            modality
        )

        corr_list.append(
            corr_df
        )

    merged = corr_list[0]

    for x in corr_list[1:]:

        merged = merged.merge(
            x,
            on="gene",
            how="outer"
        )

    rho_cols = [
        f"{m}_rho"
        for m in MODALITY_ORDER
    ]

    merged["max_abs_rho"] = merged[
        rho_cols
    ].abs().max(
        axis=1
    )

    merged = merged.sort_values(
        "max_abs_rho",
        ascending=False
    )

    out_csv = os.path.join(
        OUT_DIR,
        f"{endpoint}_protein_four_modal_risk_spearman_corr.csv"
    )

    merged.to_csv(
        out_csv,
        index=False
    )

    print(
        "Saved correlation table:",
        out_csv
    )

    return merged


# ======================================================
# Build gene sets for four modalities
# ======================================================

def build_modality_gene_sets(
    corr_df,
    mode
):

    gene_sets = {}

    for modality in MODALITY_ORDER:

        rho_col = f"{modality}_rho"
        fdr_col = f"{modality}_FDR"

        if rho_col not in corr_df.columns:

            raise ValueError(
                f"Missing column: {rho_col}"
            )

        if fdr_col not in corr_df.columns:

            raise ValueError(
                f"Missing column: {fdr_col}"
            )

        sub = corr_df.copy()

        sub = sub[
            sub[rho_col].notna()
            &
            sub[fdr_col].notna()
        ].copy()

        sub = sub[
            sub[fdr_col] <= FDR_CUTOFF
        ].copy()

        if mode == "positive":

            sub = sub[
                sub[rho_col] > POS_CUTOFF
            ].copy()

        elif mode == "negative":

            sub = sub[
                sub[rho_col] < NEG_CUTOFF
            ].copy()

        else:

            raise ValueError(mode)

        gene_sets[modality] = set(
            sub["gene"]
            .dropna()
            .astype(str)
            .tolist()
        )

    return gene_sets


# ======================================================
# Exact Venn regions
# ======================================================

def all_nonempty_combinations():

    combos = []

    for k in range(
        1,
        len(MODALITY_ORDER) + 1
    ):

        for comb in itertools.combinations(
            MODALITY_ORDER,
            k
        ):

            combos.append(
                tuple(comb)
            )

    return combos


def compute_exact_regions(
    gene_sets
):

    all_modalities = set(
        MODALITY_ORDER
    )

    rows = []
    region_gene_tables = {}

    for comb in all_nonempty_combinations():

        active = set(comb)
        inactive = all_modalities - active

        active_sets = [
            gene_sets[m]
            for m in active
        ]

        genes = set.intersection(
            *active_sets
        )

        for m in inactive:

            genes = genes - gene_sets[m]

        genes = sorted(
            list(genes)
        )

        region_name = "|".join(comb)

        rows.append({
            "combination": region_name,
            "degree": len(comb),
            "count": len(genes),
            "genes": ";".join(genes)
        })

        region_gene_tables[region_name] = genes

    summary = pd.DataFrame(
        rows
    ).sort_values(
        ["degree", "combination"],
        ascending=[
            False,
            True
        ]
    )

    return summary, region_gene_tables


def save_exact_region_tables(
    endpoint,
    mode,
    region_summary,
    region_gene_tables
):

    out_dir = os.path.join(
        REGION_DIR,
        endpoint,
        mode
    )

    os.makedirs(
        out_dir,
        exist_ok=True
    )

    summary_out = os.path.join(
        out_dir,
        f"{endpoint}_{mode}_4set_venn_exact_region_summary.csv"
    )

    region_summary.to_csv(
        summary_out,
        index=False
    )

    print(
        "Saved region summary:",
        summary_out
    )

    for region_name, genes in region_gene_tables.items():

        safe_region = safe_name(
            region_name
        )

        out_csv = os.path.join(
            out_dir,
            f"{endpoint}_{mode}_{safe_region}_genes.csv"
        )

        pd.DataFrame({
            "gene": genes
        }).to_csv(
            out_csv,
            index=False
        )


# ======================================================
# Plot four-set Venn
# ======================================================

def plot_one_4set_venn(
    endpoint,
    endpoint_label,
    mode,
    gene_sets
):

    if mode == "positive":

        subtitle = f"rho > {POS_CUTOFF}"

    else:

        subtitle = f"rho < {NEG_CUTOFF}"

    if FDR_CUTOFF < 1:

        subtitle += f", FDR < {FDR_CUTOFF}"

    venn_input = {
        f"{m}\n$n$ = {len(gene_sets[m])}": gene_sets[m]
        for m in MODALITY_ORDER
    }

    fig, ax = plt.subplots(
        figsize=FIGSIZE_SINGLE
    )

    venn(
        venn_input,
        fmt="{size}",
        cmap=[
            VENN_COLORS[m]
            for m in MODALITY_ORDER
        ],
        fontsize=8,
        legend_loc=None,
        ax=ax
    )

    ax.set_title(
        f"{endpoint_label}\n{subtitle}",
        fontsize=12,
        pad=8
    )

    plt.tight_layout()

    out_pdf = os.path.join(
        VENN_DIR,
        (
            f"{endpoint}_{mode}_4set_venn_"
            f"rho_{abs(POS_CUTOFF):.2f}_"
            f"FDR_{FDR_CUTOFF:.2f}.pdf"
        )
    )

    out_svg = out_pdf.replace(
        ".pdf",
        ".svg"
    )

    plt.savefig(
        out_pdf,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.savefig(
        out_svg,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.close()

    print(
        "Saved 4-set Venn:",
        out_pdf
    )


def plot_endpoint_4set_venn_combined(
    endpoint,
    endpoint_label,
    corr_df
):

    fig, axes = plt.subplots(
        1,
        2,
        figsize=FIGSIZE_COMBINED
    )

    for ax, mode in zip(
        axes,
        ["positive", "negative"]
    ):

        gene_sets = build_modality_gene_sets(
            corr_df,
            mode
        )

        if mode == "positive":

            subtitle = f"rho > {POS_CUTOFF}"

        else:

            subtitle = f"rho < {NEG_CUTOFF}"

        if FDR_CUTOFF < 1:

            subtitle += f", FDR < {FDR_CUTOFF}"

        venn_input = {
            f"{m}\n$n$ = {len(gene_sets[m])}": gene_sets[m]
            for m in MODALITY_ORDER
        }

        venn(
            venn_input,
            fmt="{size}",
            cmap=[
                VENN_COLORS[m]
                for m in MODALITY_ORDER
            ],
            fontsize=8,
            legend_loc=None,
            ax=ax
        )

        ax.set_title(
            f"{endpoint_label}\n{subtitle}",
            fontsize=12,
            pad=8
        )

    plt.tight_layout(
        w_pad=2.0
    )

    out_pdf = os.path.join(
        VENN_DIR,
        (
            f"{endpoint}_positive_negative_4set_venn_combined_"
            f"rho_{abs(POS_CUTOFF):.2f}_"
            f"FDR_{FDR_CUTOFF:.2f}.pdf"
        )
    )

    out_svg = out_pdf.replace(
        ".pdf",
        ".svg"
    )

    plt.savefig(
        out_pdf,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.savefig(
        out_svg,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.close()

    print(
        "Saved combined 4-set Venn:",
        out_pdf
    )


# ======================================================
# GO enrichment
# ======================================================

def run_go_for_one_intersection(
    genes,
    endpoint,
    direction,
    combination
):

    genes = [
        clean_gene(g)
        for g in genes
    ]

    genes = sorted(
        list(
            set(
                [
                    g for g in genes
                    if g not in ["", "NAN", "NONE"]
                ]
            )
        )
    )

    n_genes = len(genes)

    if n_genes <= GO_MIN_GENES:

        return None

    combo_safe = safe_name(
        combination
    )

    prefix = (
        f"{endpoint}_{direction}_{combo_safe}_n{n_genes}"
    )

    out_dir = os.path.join(
        GO_DIR,
        endpoint,
        direction,
        combo_safe
    )

    os.makedirs(
        out_dir,
        exist_ok=True
    )

    gene_out = os.path.join(
        out_dir,
        f"{prefix}_input_genes.csv"
    )

    pd.DataFrame({
        "gene": genes
    }).to_csv(
        gene_out,
        index=False
    )

    print(
        f"[GO] {endpoint} | {direction} | {combination} | n={n_genes}"
    )

    try:

        enr = gp.enrichr(
            gene_list=genes,
            gene_sets=GO_LIBRARY,
            organism="Human",
            outdir=None,
            cutoff=1.0,
            no_plot=True
        )

        res = enr.results.copy()

        if res is None or len(res) == 0:

            return None

        res.insert(
            0,
            "endpoint",
            endpoint
        )

        res.insert(
            1,
            "direction",
            direction
        )

        res.insert(
            2,
            "combination",
            combination
        )

        res.insert(
            3,
            "n_input_genes",
            n_genes
        )

        if "Genes" not in res.columns:

            possible_gene_cols = [
                c for c in res.columns
                if c.lower() == "genes"
            ]

            if len(possible_gene_cols) > 0:

                res = res.rename(
                    columns={
                        possible_gene_cols[0]: "Genes"
                    }
                )

        raw_out = os.path.join(
            out_dir,
            f"{prefix}_GO_raw.csv"
        )

        res.to_csv(
            raw_out,
            index=False
        )

        if "Adjusted P-value" in res.columns:

            sig = res[
                res["Adjusted P-value"] < GO_FDR_CUTOFF
            ].copy()

        else:

            sig = res.copy()

        sig_out = os.path.join(
            out_dir,
            f"{prefix}_GO_FDR_{GO_FDR_CUTOFF:.2f}.csv"
        )

        sig.to_csv(
            sig_out,
            index=False
        )

        print(
            "Saved GO raw:",
            raw_out
        )

        print(
            "Saved GO filtered:",
            sig_out
        )

        return res

    except Exception as e:

        print(
            "[GO ERROR]",
            endpoint,
            direction,
            combination,
            e
        )

        return None


def run_go_for_large_regions(
    all_region_rows
):

    if len(all_region_rows) == 0:

        print("[GO] No region rows.")

        return

    region_df = pd.concat(
        all_region_rows,
        axis=0,
        ignore_index=True
    )

    all_go = []

    for _, row in region_df.iterrows():

        endpoint = row["endpoint"]
        direction = row["direction"]
        combination = row["combination"]
        count = int(row["count"])

        if count <= GO_MIN_GENES:

            continue

        genes = str(row["genes"]).split(";")

        genes = [
            g for g in genes
            if g != ""
        ]

        res = run_go_for_one_intersection(
            genes,
            endpoint,
            direction,
            combination
        )

        if res is not None and len(res) > 0:

            all_go.append(
                res
            )

    if len(all_go) == 0:

        print(
            "[GO] No large regions with successful enrichment."
        )

        return

    all_go_df = pd.concat(
        all_go,
        axis=0,
        ignore_index=True
    )

    out_raw = os.path.join(
        GO_DIR,
        (
            f"ALL_4set_venn_large_regions_"
            f"GO_{GO_LIBRARY}_raw.csv"
        )
    )

    all_go_df.to_csv(
        out_raw,
        index=False
    )

    print(
        "[GO] Saved combined raw GO:",
        out_raw
    )

    if "Adjusted P-value" in all_go_df.columns:

        all_sig = all_go_df[
            all_go_df["Adjusted P-value"] < GO_FDR_CUTOFF
        ].copy()

        out_sig = os.path.join(
            GO_DIR,
            (
                f"ALL_4set_venn_large_regions_"
                f"GO_{GO_LIBRARY}_FDR_{GO_FDR_CUTOFF:.2f}.csv"
            )
        )

        all_sig.to_csv(
            out_sig,
            index=False
        )

        print(
            "[GO] Saved combined significant GO:",
            out_sig
        )


# ======================================================
# Main
# ======================================================

def main():

    protein_mat = load_protein_matrix()

    all_region_rows = []

    for endpoint in TARGET_ENDPOINTS:

        print("\n====================================")
        print(endpoint)
        print("====================================")

        corr_df = compute_endpoint_corr(
            protein_mat,
            endpoint
        )

        # ----------------------------------------------
        # positive / negative
        # ----------------------------------------------

        for mode in [
            "positive",
            "negative"
        ]:

            gene_sets = build_modality_gene_sets(
                corr_df,
                mode
            )

            # save exact regions
            region_summary, region_gene_tables = compute_exact_regions(
                gene_sets
            )

            region_summary.insert(
                0,
                "endpoint",
                endpoint
            )

            region_summary.insert(
                1,
                "direction",
                mode
            )

            save_exact_region_tables(
                endpoint,
                mode,
                region_summary,
                region_gene_tables
            )

            all_region_rows.append(
                region_summary
            )

            # single Venn
            plot_one_4set_venn(
                endpoint,
                ENDPOINT_LABELS[endpoint],
                mode,
                gene_sets
            )

        # combined positive + negative Venn
        plot_endpoint_4set_venn_combined(
            endpoint,
            ENDPOINT_LABELS[endpoint],
            corr_df
        )

    # ==================================================
    # Save all exact region summary
    # ==================================================

    all_region_df = pd.concat(
        all_region_rows,
        axis=0,
        ignore_index=True
    )

    out_all_region = os.path.join(
        REGION_DIR,
        (
            f"ALL_endpoints_4set_venn_exact_region_summary_"
            f"rho_{abs(POS_CUTOFF):.2f}_"
            f"FDR_{FDR_CUTOFF:.2f}.csv"
        )
    )

    all_region_df.to_csv(
        out_all_region,
        index=False
    )

    print(
        "Saved all region summary:",
        out_all_region
    )

    # ==================================================
    # GO enrichment after all Venn plots
    # ==================================================

    print("\n====================================")
    print("GO enrichment for exact Venn regions with > 50 genes")
    print("====================================")

    run_go_for_large_regions(
        all_region_rows
    )

    print("\nAll done.")
    print("Output:", OUT_DIR)
    print("Venn output:", VENN_DIR)
    print("Region output:", REGION_DIR)
    print("GO output:", GO_DIR)


if __name__ == "__main__":

    main()

In [ ]:
# ======================================================
# Save separate legend
# ======================================================

from matplotlib.patches import Patch


def save_venn_legend():

    handles = []

    for modality in MODALITY_ORDER:

        handles.append(
            Patch(
                facecolor=VENN_COLORS[modality],
                edgecolor="none",
                alpha=0.65,
                label=modality
            )
        )

    fig, ax = plt.subplots(
        figsize=(2.0, 0.45)
    )

    ax.axis("off")

    legend = ax.legend(
        handles=handles,
        labels=MODALITY_ORDER,
        loc="center",
        ncol=len(MODALITY_ORDER),
        frameon=False,
        fontsize=8,
        handlelength=1.1,
        handleheight=0.8,
        handletextpad=0.35,
        columnspacing=0.8,
        borderaxespad=0.0
    )

    out_pdf = os.path.join(
        VENN_DIR,
        "four_modal_venn_legend.pdf"
    )

    out_svg = out_pdf.replace(
        ".pdf",
        ".svg"
    )

    plt.savefig(
        out_pdf,
        dpi=300,
        transparent=True,
        bbox_inches="tight",
        pad_inches=0.02
    )

    plt.savefig(
        out_svg,
        dpi=300,
        transparent=True,
        bbox_inches="tight",
        pad_inches=0.02
    )

    plt.close()

    print("Saved legend:", out_pdf)
save_venn_legend()

### Restricted-data analysis not included

This block used protein abundance/expression-compatible panel scoring. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.


### Restricted-data analysis not included

This block used protein abundance/expression-compatible panel scoring. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.


In [ ]:

'''
PANEL_GENES = [
    "REN","NTproBNP","LGALS4","GDF15","APOM","NPPB","DCBLD2","PON3","APOC1",
    "ANGPTL3","PCSK9","GAST","MMP12","CTSB","ENPP2","PLA2G7","CRELD2",
    "CD300LG","PRSS8","VSIG2","IL7R","ADGRG2","IFI30","CHGA","ACRV1",
    "FGFBP1","PLAT","HYOU1","CD99L2","SCT","ACE2","KDR","MET","XPNPEP2",
    "PSPN","PGA4","EGFR","TNFRSF10B","RAB6A","IGFBP3","NFASC","KLK3",
    "GDF15","MMP12","HAVCR1","PGF","ACRV1","ADGRG2","PLTP","CXCL17","PLA2G7",
    "MXRA8","CDCP1","EDA2R","PSPN","ACE2","NTproBNP","PON3","ADAMTS13","CD1C",
    "CDHR2","INSL3","SLIT2","CHCHD10","PARD3","PLAT","VSIG2","REN","HGF","MMP3",
    "TNFRSF10B","PRSS8","CALCA","CNTN1","CST3","PCSK9","TNXB","PHOSPHO1","CPB1",
    "MYBPC1","CD300LG","APOC1","NRP2","CLUL1","PLAUR","LGALS1","ACP5","LCN2",
    "KRT8","NOTCH3","PRAP1","LY6D","CTRB1","PTGDS","DSG2","SPINT2","ENTPD5",
    "NCS1","CDKN2D","CRIP2","APOA1","SPINT3","PKD1","SPINT1","SMOC1"
]#CHD

PANEL_GENES = [
    "REN","NTproBNP","DCBLD2","GDF15","F10","FGFBP1","PLA2G7","ANGPTL3","IL11",
    "NEFL","PROS1","FST","TNFRSF11A","TNFRSF19","COL6A3","TNFSF13","GHRL","EGFR",
    "MET","RASGRF1","GFAP","CA4","CCL24","OMG","NPTXR","RAB6A","PCSK9","FGL1",
    "NT5E","F7","ZNF75D","MMP3","GPC5","MMP12","IL6","LGALS1","PRSS53",
    "DEFB4A_DEFB4B","GP1BA","BCAN","KLB","POSTN","FGFR4","ADGRG2","ITGA11",
    "PLA2G2A","CRLF1","ITGAV","VEGFB","CST1","ATRN","MILR1","ICAM5","CHGB",
    "RSPO3","IFNGR2","GDF2","EDA2R","SMOC1","CRHR1","GHR","PON2","INSL4",
    "LGALS4","BGLAP","IGSF8","TFPI","CA11","STAB2","YAP1","IL10","PLAU",
    "STK24","IL22RA1","LAG3","TCN2","CD38","DCDC2C","LMOD2","VSIG4","COPE",
    "RELT","PCDHB15","KLK1","FCRL2","ATP2B4","RABEPK","NPPB","ADM","LGALS3",
    "CGREF1","PFDN6","MXRA8","MRPL46","CLEC2L","CDHR2","SERPINA5","L1CAM",
    "CD59","PGF","TDGF1","IL17D","PTGDS","SEMA3G","GUCY2C","PIK3IP1","EPHA2",
    "KLK10"
]#istorke
'''

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
import gseapy as gp

# ======================================================
# AI editable text
# ======================================================

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'DejaVu Sans'

# ======================================================
# Config
# ======================================================

BASE = "data/private/inputs"

OUT_DIR = os.path.join(
    BASE,
    "code_3_3_save/protein_risk_correlation_gsea_directional"
)

os.makedirs(OUT_DIR, exist_ok=True)

# ======================================================
# endpoints
# ======================================================

ENDPOINTS = [
    "CHD",
    "istroke",
    "ICVDr"
]

ENDPOINT_LABELS = {
    "CHD": "CHD",
    "istroke": "Ischemic stroke",
    "ICVDr": "ASCVD"
}

# ======================================================
# models
# ======================================================

RISK_MODELS = {

    "SNP":
        "code_02_02_save/snp_20260202_1432/preds/{endpoint}_snp_TR_test_best_pred.csv",

    "Proteomics":
        "code_02_02_save/soma_20260202_2006/preds/{endpoint}_soma_TR_test_best_pred.csv",

    "ECG":
        "code_02_02_save/ecg_20260203_1003/preds/{endpoint}_ecg_transformer_h256_l3_test_best_pred.csv",

    "Fundus":
        "code_02_02_save/fundus_20251229_1519/preds/{endpoint}_fundus_vitsmallFT_test_best_pred.csv",

    "ToyCombined":
        "code_3_3_save/toy_multimodal_risk_combination/preds/{endpoint}_toy_combined_pred.csv"
}

MODEL_ORDER = [
    "SNP",
    "Proteomics",
    "ECG",
    "Fundus",
    "ToyCombined"
]

# ======================================================
# protein input
# ======================================================

PROTEIN_INPUT = (
    BASE +
    "/code_3_3_save/soma_single/CHD_soma_TR_best_IG_input.csv"
)

# ======================================================
# GSEA libraries
# ======================================================

GENESETS = {
    "Hallmark": "MSigDB_Hallmark_2020",
    #"KEGG": "KEGG_2021_Human"
}

# ======================================================
# thresholds
# ======================================================

PVALUE_THRES = 1

RHO_THRES = 0.0

# ======================================================
# colors
# ======================================================

MODEL_COLORS = {
    "SNP": "#4C72B0",
    "Proteomics": "#55A868",
    "ECG": "#8172B2",
    "Fundus": "#DD8452",
    "ToyCombined": "#C44E52"
}

# ======================================================
# Utils
# ======================================================

def clean_gene(x):

    x = str(x)

    x = re.sub(r"\.\d+$", "", x)

    if x.startswith("seq."):
        x = x.split(".")[1]#("seq.", "")
    x = x.upper()
    return x


def rank_norm(x):

    r = pd.Series(x).rank(method="average").values

    return (r - 1) / (len(r) - 1 + 1e-8)


# ======================================================
# Load protein matrix
# ======================================================

def load_protein_matrix(endpoint):

    fp = PROTEIN_INPUT.format(
        endpoint=endpoint
    )

    df = pd.read_csv(fp)

    df["pid"] = df["pid"].astype(int)

    df["feature"] = df["feature"].astype(str)

    df["gene"] = df["feature"].map(clean_gene)

    mat = df.pivot_table(
        index="gene",
        columns="pid",
        values="ig_abs",
        aggfunc="mean"
    )

    return mat


# ======================================================
# Correlation analysis
# ======================================================

def compute_gene_corr(
    protein_mat,
    risk_df
):

    common_pids = sorted(
        list(
            set(protein_mat.columns) &
            set(risk_df["pid"])
        )
    )

    protein_mat = protein_mat[common_pids]

    risk_df = risk_df[
        risk_df["pid"].isin(common_pids)
    ].copy()

    risk_df = risk_df.set_index("pid")

    risk_vec = risk_df.loc[
        common_pids,
        "risk_norm"
    ].values

    results = []

    total = len(protein_mat.index)

    for idx, gene in enumerate(protein_mat.index):

        if idx % 500 == 0:
            print(f"{idx}/{total}")

        vals = protein_mat.loc[
            gene,
            common_pids
        ].values.astype(float)

        keep = ~np.isnan(vals)

        if np.sum(keep) < 10:
            continue

        try:

            rho, pval = spearmanr(
                vals[keep],
                risk_vec[keep]
            )

            results.append({
                "gene": gene,
                "rho": rho,
                "pval": pval
            })

        except:
            continue

    res = pd.DataFrame(results)

    res = res.dropna()

    # FDR
    res["FDR"] = multipletests(
        res["pval"],
        method="fdr_bh"
    )[1]

    res = res.sort_values(
        "rho",
        ascending=False
    )

    return res


# ======================================================
# Directional GSEA
# ======================================================

def run_directional_gsea(
    corr_df,
    geneset_name,
    direction="positive"
):

    # ==================================================
    # positive
    # ==================================================

    if direction == "positive":

        sub = corr_df[
            (corr_df["rho"] > RHO_THRES) 
        ].copy()

        sub = sub.sort_values(
            "rho",
            ascending=False
        )

    # ==================================================
    # negative
    # ==================================================

    else:

        sub = corr_df[
            (corr_df["rho"] < -RHO_THRES)
        ].copy()

        # reverse sign
        sub["rho"] = -sub["rho"]

        sub = sub.sort_values(
            "rho",
            ascending=False
        )

    # ==================================================
    # too few genes
    # ==================================================

    if len(sub) < 20:

        print(
            f"[Skip] too few genes for {direction}"
        )

        return None

    ranking = sub[[
        "gene",
        "rho"
    ]].drop_duplicates()

    ranking.columns = [
        "gene",
        "score"
    ]
    ranking.columns = [
    "gene",
    "score"
    ]
    
    # ==================================================
    # DEBUG overlap
    # ==================================================
    
    print("\n===================================")
    print("DEBUG GSEA")
    print("direction:", direction)
    print("geneset:", geneset_name)
    
    print("\nTop ranking genes:")
    print(ranking.head(30))
    
    gmt = gp.get_library(
        name=geneset_name,
        organism="Human"
    )
    
    all_pathway_genes = set()
    
    for pathway in gmt.values():
    
        all_pathway_genes.update(pathway)
    
    ranking_genes = set(ranking["gene"])
    
    overlap = ranking_genes & all_pathway_genes
    
    print("\nRanking genes:", len(ranking_genes))
    print("Pathway genes:", len(all_pathway_genes))
    print("Overlap:", len(overlap))
    
    print("\nOverlap genes:")
    print(list(overlap)[:50])
    
    print("===================================\n")

    pre_res = gp.prerank(
        rnk=ranking,
        gene_sets=geneset_name,
        permutation_num=1000,
        outdir=None,
        seed=42,
        min_size=5,
        max_size=500,
        verbose=False
    )

    gsea_res = pre_res.res2d.copy()

    # clean columns
    gsea_res.columns = [
        c.replace(" ", "_").replace("-", "_")
        for c in gsea_res.columns
    ]

    gsea_res = gsea_res.sort_values(
        "NES",
        ascending=False
    )

    return gsea_res


# ======================================================
# Main
# ======================================================

for endpoint in ENDPOINTS:

    print("\n================================================")
    print(endpoint)
    print("================================================")

    protein_mat = load_protein_matrix(
        endpoint
    )

    # ==================================================
    # positive / negative separately
    # ==================================================

    for direction in [
        "positive",
        "negative"
    ]:

        print("\n############################################")
        print(direction)
        print("############################################")

        all_gsea = {}

        # ==============================================
        # loop models
        # ==============================================

        for model in MODEL_ORDER:

            print("\n--------------------------------")
            print(model)
            print("--------------------------------")

            risk_fp = os.path.join(
                BASE,
                RISK_MODELS[model].format(
                    endpoint=endpoint
                )
            )

            risk_df = pd.read_csv(risk_fp)

            if "ID" in risk_df.columns:

                risk_df = risk_df.rename(
                    columns={"ID":"pid"}
                )

            risk_df["pid"] = risk_df[
                "pid"
            ].astype(int)

            risk_df["risk_norm"] = rank_norm(
                risk_df["risk"].values
            )

            # ==========================================
            # correlation
            # ==========================================

            corr_df = compute_gene_corr(
                protein_mat,
                risk_df
            )

            # save
            corr_out = os.path.join(
                OUT_DIR,
                f"{endpoint}_{model}_{direction}_protein_corr.csv"
            )

            corr_df.to_csv(
                corr_out,
                index=False
            )

            print("Saved:", corr_out)

            # ==========================================
            # GSEA
            # ==========================================

            for gs_name, gs_db in GENESETS.items():

                gsea_res = run_directional_gsea(
                    corr_df,
                    gs_db,
                    direction=direction
                )

                if gsea_res is None:
                    continue

                gsea_res["model"] = model
                gsea_res["endpoint"] = endpoint
                gsea_res["geneset_db"] = gs_name
                gsea_res["direction"] = direction

                key = (
                    f"{endpoint}_"
                    f"{gs_name}_"
                    f"{direction}"
                )

                if key not in all_gsea:
                    all_gsea[key] = []

                all_gsea[key].append(
                    gsea_res
                )

        # ==================================================
        # Plot
        # ==================================================

        for key in all_gsea:

            merged = pd.concat(
                all_gsea[key],
                axis=0
            )

            # top pathways
            top_terms = (
                merged.groupby("Term")["NES"]
                .mean()
                .sort_values(ascending=False)
                .head(20)
                .index
            )

            plot_df = merged[
                merged["Term"].isin(top_terms)
            ].copy()

            # ==============================================
            # each pathway
            # ==============================================

            for term in top_terms:

                sub = plot_df[
                    plot_df["Term"] == term
                ].copy()

                sub["model"] = pd.Categorical(
                    sub["model"],
                    categories=MODEL_ORDER,
                    ordered=True
                )

                sub = sub.sort_values(
                    "model"
                )

                plt.figure(figsize=(8,5))

                x = np.arange(len(sub))

                plt.bar(
                    x,
                    sub["NES"],
                    color=[
                        MODEL_COLORS[m]
                        for m in sub["model"]
                    ],
                    alpha=0.9
                )

                # ==========================================
                # FDR text
                # ==========================================

                for i, row in enumerate(
                    sub.itertuples()
                ):

                    padj = sub.iloc[i]["FDR_q_val"]

                    plt.text(
                        i,
                        row.NES + 0.1,
                        f"FDR={padj:.2e}",
                        ha="center",
                        fontsize=8,
                        rotation=90
                    )

                plt.xticks(
                    x,
                    sub["model"],
                    rotation=25
                )

                plt.ylabel("NES")

                plt.title(
                    f"{ENDPOINT_LABELS[endpoint]}\n"
                    f"{direction.upper()} | {term}"
                )

                plt.axhline(
                    0,
                    linestyle="--",
                    color="gray"
                )

                plt.gca().spines["top"].set_visible(False)
                plt.gca().spines["right"].set_visible(False)

                plt.tight_layout()

                safe_term = (
                    term.replace("/", "_")
                    .replace(" ", "_")
                )

                out_pdf = os.path.join(
                    OUT_DIR,
                    f"{endpoint}_"
                    f"{direction}_"
                    f"{safe_term}_NES_barplot.pdf"
                )

                plt.savefig(
                    out_pdf,
                    dpi=300,
                    transparent=True,
                    bbox_inches="tight"
                )

                plt.close()

                print("Saved:", out_pdf)

print("\nAll done.")
print("Output:", OUT_DIR)

In [ ]:
# ======================================================
# NEW:
# ToyCombined dominant pathway plots
# ======================================================

DOMINANT_DIR = os.path.join(
    OUT_DIR,
    "toy_model_dominant_pathways"
)

os.makedirs(DOMINANT_DIR, exist_ok=True)

# ======================================================
# New colors
# ======================================================

MODEL_COLORS_NEW = {
    "SNP": "#6EB8D2",
    "Proteomics": "#479D88",
    "ECG": "#435385",
    "Fundus": "#8790B2",
    "ToyCombined": "#E6A083"
}

# ======================================================
# only CHD / istroke
# ======================================================

TARGET_ENDPOINTS = [
    "CHD",
    "istroke"
]

# ======================================================
# loop endpoint + direction
# ======================================================

for endpoint in TARGET_ENDPOINTS:

    for direction in ["positive", "negative"]:

        print("\n====================================")
        print(endpoint, direction)
        print("====================================")

        all_res = []

        # ==================================================
        # read all GSEA results
        # ==================================================

        for model in MODEL_ORDER:

            for gs_name in GENESETS.keys():

                pattern = (
                    f"{endpoint}_"
                    f"{model}_"
                    f"{direction}_protein_corr.csv"
                )

        # ==================================================
        # collect pathway NES
        # ==================================================

        merged_all = []

        for model in MODEL_ORDER:

            risk_fp = os.path.join(
                BASE,
                RISK_MODELS[model].format(
                    endpoint=endpoint
                )
            )

            risk_df = pd.read_csv(risk_fp)

            if "ID" in risk_df.columns:

                risk_df = risk_df.rename(
                    columns={"ID":"pid"}
                )

            risk_df["pid"] = risk_df["pid"].astype(int)

            risk_df["risk_norm"] = rank_norm(
                risk_df["risk"].values
            )

            corr_df = compute_gene_corr(
                protein_mat,
                risk_df
            )

            for gs_name, gs_db in GENESETS.items():

                gsea_res = run_directional_gsea(
                    corr_df,
                    gs_db,
                    direction=direction
                )

                if gsea_res is None:
                    continue

                gsea_res["model"] = model

                merged_all.append(gsea_res)

        if len(merged_all) == 0:
            continue

        merged = pd.concat(
            merged_all,
            axis=0
        )

        # ==================================================
        # pathway filter:
        # ToyCombined > all others
        # ==================================================

        dominant_terms = []

        for term in merged["Term"].unique():

            sub = merged[
                merged["Term"] == term
            ].copy()

            if "ToyCombined" not in sub["model"].values:
                continue

            toy_nes = sub[
                sub["model"] == "ToyCombined"
            ]["NES"].values[0]

            others = sub[
                sub["model"] != "ToyCombined"
            ]["NES"].values

            if len(others) == 0:
                continue

            if toy_nes > np.max(others):

                dominant_terms.append(term)

        print(
            "Dominant pathways:",
            len(dominant_terms)
        )

        if len(dominant_terms) == 0:
            continue

        dominant_df = merged[
            merged["Term"].isin(
                dominant_terms
            )
        ].copy()

        # ==================================================
        # sort pathways
        # ==================================================

        pathway_order = (
            dominant_df[
                dominant_df["model"] == "ToyCombined"
            ]
            .sort_values(
                "NES",
                ascending=False
            )["Term"]
            .tolist()
        )

        n_pathways = len(pathway_order)

        ncols = int(
            np.ceil(n_pathways / 2)
        )

        # ==================================================
        # Figure
        # ==================================================

        fig, axes = plt.subplots(
            2,
            ncols,
            figsize=(4*ncols, 8)
        )

        axes = np.array(axes).reshape(-1)

        # ==================================================
        # loop pathways
        # ==================================================

        for ax_i, term in enumerate(pathway_order):

            ax = axes[ax_i]

            sub = dominant_df[
                dominant_df["Term"] == term
            ].copy()

            sub["model"] = pd.Categorical(
                sub["model"],
                categories=MODEL_ORDER,
                ordered=True
            )

            sub = sub.sort_values("model")

            x = np.arange(len(sub))

            bars = ax.bar(
                x,
                sub["NES"],
                color=[
                    MODEL_COLORS_NEW[m]
                    for m in sub["model"]
                ],
                alpha=0.95
            )

            # ==============================================
            # FDR text
            # ==============================================

            for i, row in enumerate(
                sub.itertuples()
            ):

                try:

                    padj = sub.iloc[i]["FDR_q_val"]

                    ax.text(
                        i,
                        row.NES + 0.08,
                        f"{padj:.1e}",
                        ha="center",
                        fontsize=7,
                        rotation=90
                    )

                except:
                    pass

            # ==============================================
            # title
            # ==============================================

            short_term = (
                term.replace(
                    "HALLMARK_",
                    ""
                )
                .replace("_", " ")
                .title()
            )

            ax.set_title(
                short_term,
                fontsize=10
            )

            ax.set_xticks(x)

            ax.set_xticklabels(
                sub["model"],
                rotation=45,
                ha="right",
                fontsize=8
            )

            ax.axhline(
                0,
                linestyle="--",
                color="gray",
                linewidth=1
            )

            ax.set_ylabel("NES")

            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

        # ==================================================
        # remove empty axes
        # ==================================================

        for j in range(
            len(pathway_order),
            len(axes)
        ):

            fig.delaxes(axes[j])

        # ==================================================
        # title
        # ==================================================

        fig.suptitle(
            f"{ENDPOINT_LABELS[endpoint]} | "
            f"{direction.upper()} pathways "
            f"preferentially enhanced in ToyCombined",
            fontsize=18,
            y=0.995
        )

        plt.tight_layout(
            rect=[0,0,1,0.97]
        )

        out_pdf = os.path.join(
            DOMINANT_DIR,
            f"{endpoint}_{direction}_toy_dominant_pathways.pdf"
        )

        plt.savefig(
            out_pdf,
            dpi=300,
            transparent=True,
            bbox_inches="tight"
        )

        plt.close()

        print("Saved:", out_pdf)

print("\nDone dominant pathway plots.")
print("Output:", DOMINANT_DIR)

In [ ]:
# ======================================================
# FINAL VERSION
# no positive / negative split
# cached GSEA
# ToyCombined dominant pathway plots
# missing pathways allowed
# ======================================================

import os
import re
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
import gseapy as gp

# ======================================================
# AI editable text
# ======================================================

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'DejaVu Sans'

# ======================================================
# Config
# ======================================================

BASE = "data/private/inputs"

OUT_DIR = os.path.join(
    BASE,
    "code_3_3_save/protein_risk_correlation_gsea_final"
)

os.makedirs(OUT_DIR, exist_ok=True)

# ======================================================
# cache
# ======================================================

CACHE_DIR = os.path.join(
    OUT_DIR,
    "cached_gsea"
)

os.makedirs(CACHE_DIR, exist_ok=True)

# ======================================================
# dominant
# ======================================================

DOMINANT_DIR = os.path.join(
    OUT_DIR,
    "toy_model_dominant_pathways"
)

os.makedirs(DOMINANT_DIR, exist_ok=True)

# ======================================================
# endpoints
# ======================================================

TARGET_ENDPOINTS = [
    "CHD",
    "istroke",
    "ICVD"
]

ENDPOINT_LABELS = {
    "CHD": "CHD",
    "istroke": "Ischemic stroke",
    "ICVD": "ASCVD"
}

# ======================================================
# models
# ======================================================

RISK_MODELS = {

    "SNP":
        "code_02_02_save/snp_20260202_1432/preds/{endpoint}_snp_TR_test_best_pred.csv",

    "Proteomics":
        "code_02_02_save/soma_20260202_2006/preds/{endpoint}_soma_TR_test_best_pred.csv",

    "ECG":
        "code_02_02_save/ecg_20260203_1003/preds/{endpoint}_ecg_transformer_h256_l3_test_best_pred.csv",

    "Fundus":
        "code_02_02_save/fundus_20251229_1519/preds/{endpoint}_fundus_vitsmallFT_test_best_pred.csv",

    "ToyCombined":
        "code_3_3_save/toy_multimodal_risk_combination/preds/{endpoint}_toy_combined_pred.csv"
}

MODEL_ORDER = [
    "SNP",
    "Proteomics",
    "ECG",
    "Fundus",
    "ToyCombined"
]

# ======================================================
# colors
# ======================================================

MODEL_COLORS = {
    "SNP": "#6EB8D2",
    "Proteomics": "#479D88",
    "ECG": "#435385",
    "Fundus": "#8790B2",
    "ToyCombined": "#E6A083"
}

# ======================================================
# protein input
# ======================================================

PROTEIN_INPUT = (
    BASE +
    "/code_3_3_save/soma_single/CHD_soma_TR_best_IG_input.csv"
)

# ======================================================
# GSEA
# ======================================================

GENESET_NAME = "MSigDB_Hallmark_2020"

# ======================================================
# Utils
# ======================================================

def clean_gene(x):

    x = str(x)

    x = re.sub(r"\.\d+$", "", x)

    if x.startswith("seq."):

        x = x.split(".")[1]

    x = x.upper()

    return x


def rank_norm(x):

    r = pd.Series(x).rank(
        method="average"
    ).values

    return (
        (r - 1)
        / (len(r) - 1 + 1e-8)
    )


# ======================================================
# protein matrix
# ======================================================

def load_protein_matrix(endpoint):

    fp = PROTEIN_INPUT.format(
        endpoint=endpoint
    )

    df = pd.read_csv(fp)

    df["pid"] = df["pid"].astype(int)

    df["feature"] = df["feature"].astype(str)

    df["gene"] = df["feature"].map(
        clean_gene
    )

    mat = df.pivot_table(
        index="gene",
        columns="pid",
        values="ig_abs",
        aggfunc="mean"
    )

    return mat


# ======================================================
# correlation
# ======================================================

def compute_gene_corr(
    protein_mat,
    risk_df
):

    common_pids = sorted(
        list(
            set(protein_mat.columns)
            &
            set(risk_df["pid"])
        )
    )

    protein_mat = protein_mat[
        common_pids
    ]

    risk_df = risk_df[
        risk_df["pid"].isin(
            common_pids
        )
    ].copy()

    risk_df = risk_df.set_index(
        "pid"
    )

    risk_vec = risk_df.loc[
        common_pids,
        "risk_norm"
    ].values

    results = []

    total = len(protein_mat.index)

    for idx, gene in enumerate(
        protein_mat.index
    ):

        if idx % 500 == 0:

            print(
                f"{idx}/{total}"
            )

        vals = protein_mat.loc[
            gene,
            common_pids
        ].values.astype(float)

        keep = ~np.isnan(vals)

        if np.sum(keep) < 10:
            continue

        try:

            rho, pval = spearmanr(
                vals[keep],
                risk_vec[keep]
            )

            results.append({
                "gene": gene,
                "rho": rho,
                "pval": pval
            })

        except:
            continue

    res = pd.DataFrame(results)

    res = res.dropna()

    res["FDR"] = multipletests(
        res["pval"],
        method="fdr_bh"
    )[1]

    res = res.sort_values(
        "rho",
        ascending=False
    )

    return res


# ======================================================
# GSEA
# ======================================================

def run_gsea(
    corr_df,
    geneset_name
):

    ranking = corr_df[
        ["gene", "rho"]
    ].drop_duplicates()

    ranking.columns = [
        "gene",
        "score"
    ]

    pre_res = gp.prerank(
        rnk=ranking,
        gene_sets=geneset_name,
        permutation_num=1000,
        outdir=None,
        seed=42,
        min_size=2,
        max_size=500,
        verbose=False
    )

    gsea_res = pre_res.res2d.copy()

    gsea_res.columns = [
        c.replace(" ", "_")
        .replace("-", "_")
        for c in gsea_res.columns
    ]

    gsea_res = gsea_res.sort_values(
        "NES",
        ascending=False
    )

    return gsea_res


# ======================================================
# Main
# ======================================================

for endpoint in TARGET_ENDPOINTS:

    print("\n====================================")
    print(endpoint)
    print("====================================")

    protein_mat = load_protein_matrix(
        endpoint
    )

    merged_all = []

    # ==================================================
    # loop models
    # ==================================================

    for model in MODEL_ORDER:

        print("\n--------------------------------")
        print(model)
        print("--------------------------------")

        cache_fp = os.path.join(
            CACHE_DIR,
            f"{endpoint}_{model}_Hallmark_gsea.csv"
        )

        # ==============================================
        # load cache
        # ==============================================

        if os.path.exists(cache_fp):

            print(
                "Loading cached:"
            )

            print(cache_fp)

            gsea_res = pd.read_csv(
                cache_fp
            )

            gsea_res["model"] = model

            merged_all.append(
                gsea_res
            )

            continue

        # ==============================================
        # risk
        # ==============================================

        risk_fp = os.path.join(
            BASE,
            RISK_MODELS[model].format(
                endpoint=endpoint
            )
        )

        risk_df = pd.read_csv(
            risk_fp
        )

        if "ID" in risk_df.columns:

            risk_df = risk_df.rename(
                columns={
                    "ID":"pid"
                }
            )

        risk_df["pid"] = risk_df[
            "pid"
        ].astype(int)

        risk_df["risk_norm"] = rank_norm(
            risk_df["risk"].values
        )

        # ==============================================
        # correlation
        # ==============================================

        corr_df = compute_gene_corr(
            protein_mat,
            risk_df
        )

        # ==============================================
        # save correlation
        # ==============================================

        corr_out = os.path.join(
            CACHE_DIR,
            f"{endpoint}_{model}_corr.csv"
        )

        corr_df.to_csv(
            corr_out,
            index=False
        )

        # ==============================================
        # GSEA
        # ==============================================

        gsea_res = run_gsea(
            corr_df,
            GENESET_NAME
        )

        # ==============================================
        # save cache
        # ==============================================

        gsea_res.to_csv(
            cache_fp,
            index=False
        )

        print(
            "Saved cache:"
        )

        print(cache_fp)

        gsea_res["model"] = model

        merged_all.append(
            gsea_res
        )

    # ==================================================
    # merge
    # ==================================================

    merged = pd.concat(
        merged_all,
        axis=0
    )

    # ==================================================
    # dominant pathways
    # ==================================================

    dominant_terms = []

    for term in merged["Term"].unique():

        sub = merged[
            merged["Term"] == term
        ].copy()

        if (
            "ToyCombined"
            not in sub["model"].values
        ):

            continue

        toy_nes = sub[
            sub["model"]
            == "ToyCombined"
        ]["NES"].values[0]

        others = sub[
            sub["model"]
            != "ToyCombined"
        ]["NES"].values

        if len(others) == 0:
            continue

        if toy_nes > np.max(others):

            dominant_terms.append(
                term
            )

    print(
        "Dominant pathways:",
        len(dominant_terms)
    )

    # ==================================================
    # save dominant list
    # ==================================================

    dominant_txt = os.path.join(
        DOMINANT_DIR,
        f"{endpoint}_dominant_pathways.txt"
    )

    with open(dominant_txt, "w") as f:

        for t in dominant_terms:

            f.write(t + "\n")

    # ==================================================
    # no dominant
    # ==================================================

    if len(dominant_terms) == 0:

        continue

    dominant_df = merged[
        merged["Term"].isin(
            dominant_terms
        )
    ].copy()

    # ==================================================
    # sort pathways
    # ==================================================

    pathway_order = (
        dominant_df[
            dominant_df["model"]
            == "ToyCombined"
        ]
        .sort_values(
            "NES",
            ascending=False
        )["Term"]
        .tolist()
    )

    n_pathways = len(
        pathway_order
    )

    ncols = int(
        np.ceil(n_pathways / 2)
    )

    # ==================================================
    # figure
    # ==================================================

    fig, axes = plt.subplots(
        2,
        ncols,
        figsize=(4*ncols, 8)
    )

    axes = np.array(
        axes
    ).reshape(-1)

    # ==================================================
    # each pathway
    # ==================================================

    for ax_i, term in enumerate(
        pathway_order
    ):

        ax = axes[ax_i]

        sub = dominant_df[
            dominant_df["Term"] == term
        ].copy()

        # ==============================================
        # build full table
        # ==============================================

        rows = []

        for model in MODEL_ORDER:

            tmp = sub[
                sub["model"] == model
            ]

            if len(tmp) > 0:

                nes = tmp[
                    "NES"
                ].values[0]

            else:

                nes = np.nan

            rows.append({
                "model": model,
                "NES": nes
            })

        plot_sub = pd.DataFrame(
            rows
        )

        x = np.arange(
            len(plot_sub)
        )

        # ==============================================
        # plot bars
        # ==============================================

        for i, row in plot_sub.iterrows():

            if np.isnan(row["NES"]):

                continue

            ax.bar(
                i,
                row["NES"],
                color=MODEL_COLORS[
                    row["model"]
                ],
                alpha=0.95,
                width=0.8
            )

        # ==============================================
        # title
        # ==============================================

        short_term = (
            term.replace(
                "HALLMARK_",
                ""
            )
            .replace("_", " ")
            .title()
        )

        ax.set_title(
            short_term,
            fontsize=10
        )

        ax.set_xticks(x)

        ax.set_xticklabels(
            MODEL_ORDER,
            rotation=45,
            ha="right",
            fontsize=8
        )

        ax.axhline(
            0,
            linestyle="--",
            color="gray",
            linewidth=1
        )

        ax.set_ylabel(
            "NES"
        )

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    # ==================================================
    # remove empty axes
    # ==================================================

    for j in range(
        len(pathway_order),
        len(axes)
    ):

        fig.delaxes(
            axes[j]
        )

    # ==================================================
    # title
    # ==================================================

    fig.suptitle(
        f"{ENDPOINT_LABELS[endpoint]} | "
        f"Pathways preferentially enhanced "
        f"in ToyCombined",
        fontsize=18,
        y=0.995
    )

    plt.tight_layout(
        rect=[0,0,1,0.97]
    )

    out_pdf = os.path.join(
        DOMINANT_DIR,
        f"{endpoint}_toy_dominant_pathways.pdf"
    )

    plt.savefig(
        out_pdf,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.close()

    print("Saved:", out_pdf)

print("\nDone.")
print("Output:", DOMINANT_DIR)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# ======================================================
# AI editable text
# ======================================================

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'DejaVu Sans'

# ======================================================
# Config
# ======================================================

BASE = "data/private/inputs"

CACHE_DIR = os.path.join(
    BASE,
    "code_3_3_save/protein_risk_correlation_gsea_final/cached_gsea"
)

OUT_DIR = os.path.join(
    BASE,
    "fig3_final/final_selected_pathway_plots"
)

os.makedirs(OUT_DIR, exist_ok=True)

# ======================================================
# Colors
# ======================================================

MODEL_COLORS = {
    "SNP": "#6EB8D2",
    "Proteomics": "#479D88",
    "ECG": "#435385",
    "Fundus": "#8790B2",
    "ToyCombined": "#E6A083"
}

MODEL_ORDER = [
    "SNP",
    "Proteomics",
    "ECG",
    "Fundus",
    "ToyCombined"
]

# ======================================================
# Selected pathways
# ======================================================

SELECTED = {

    "CHD": [

        "Fatty Acid Metabolism",
        "Angiogenesis",
        "Cholesterol Homeostasis",
        "Complement"
    ],

    "istroke": [

        "Coagulation",
        "Hypoxia",
        "IL-2/STAT5 Signaling",
        "Epithelial Mesenchymal Transition"
    ],
    "ICVDr": [

        "IL-2/STAT5 Signaling",
        "Hypoxia",
        "Epithelial Mesenchymal Transition",
        "PI3K/AKT/mTOR  Signaling"
    ]
}

# ======================================================
# Endpoint labels
# ======================================================

ENDPOINT_LABELS = {
    "CHD": "CHD",
    "istroke": "Ischemic stroke",
    "ICVDr": "ASCVD"
}

# ======================================================
# Load GSEA
# ======================================================

def load_all_gsea(endpoint):

    all_res = []

    for model in MODEL_ORDER:

        fp = os.path.join(
            CACHE_DIR,
            f"{endpoint}_{model}_Hallmark_gsea.csv"
        )

        if not os.path.exists(fp):

            print("Missing:", fp)
            continue

        df = pd.read_csv(fp)

        df["model"] = model

        all_res.append(df)

    merged = pd.concat(
        all_res,
        axis=0
    )

    return merged

# ======================================================
# Plot function
# ======================================================

def make_plot(endpoint):

    print("\n================================")
    print(endpoint)
    print("================================")

    merged = load_all_gsea(endpoint)

    pathways = SELECTED[endpoint]

    # ==================================================
    # figure
    # ==================================================

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(6, 6.5)
    )

    axes = axes.reshape(-1)

    # ==================================================
    # loop pathways
    # ==================================================

    for ax_i, pathway in enumerate(pathways):

        ax = axes[ax_i]

        sub = merged[
            merged["Term"] == pathway
        ].copy()

        # ==============================================
        # build complete table
        # ==============================================

        rows = []

        for model in MODEL_ORDER:

            tmp = sub[
                sub["model"] == model
            ]

            if len(tmp) > 0:

                nes = tmp["NES"].values[0]

            else:

                nes = np.nan

            rows.append({
                "model": model,
                "NES": nes
            })

        plot_df = pd.DataFrame(rows)

        # ==============================================
        # x
        # ==============================================

        x = np.arange(len(plot_df))

        # ==============================================
        # bars
        # ==============================================

        for i, row in plot_df.iterrows():

            if np.isnan(row["NES"]):

                continue

            ax.bar(
                i,
                row["NES"],
                color=MODEL_COLORS[
                    row["model"]
                ],
                alpha=0.95,
                width=0.8
            )

        # ==============================================
        # title
        # ==============================================

        title = (
            pathway
            .replace("HALLMARK_", "")
            .replace("_", " ")
            .title()
        )

        ax.set_title(
            title,
            fontsize=20
        )

        # ==============================================
        # x labels
        # ==============================================

        ax.set_xticks(x)

        ax.set_xticklabels(
            MODEL_ORDER,
            rotation=45,
            ha="right",
            fontsize=13
        )

        # ==============================================
        # y
        # ==============================================

        ax.axhline(
            0,
            linestyle="--",
            color="gray",
            linewidth=1
        )

        ax.set_ylabel(
            "NES",
            fontsize=15
        )

        # ==============================================
        # clean
        # ==============================================

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    # ==================================================
    # title
    # ==================================================

    fig.suptitle(
        f"{ENDPOINT_LABELS[endpoint]} pathway enrichment",
        fontsize=18,
        y=0.98
    )
    mpl.rcParams['pdf.fonttype'] = 42
    mpl.rcParams['ps.fonttype'] = 42
    mpl.rcParams['svg.fonttype'] = 'none'

    plt.tight_layout(
        rect=[0,0,1,0.96]
    )

    # ==================================================
    # save
    # ==================================================

    out_pdf = os.path.join(
        OUT_DIR,
        f"{endpoint}_selected_pathways.pdf"
    )

    out_png = os.path.join(
        OUT_DIR,
        f"{endpoint}_selected_pathways.png"
    )

    plt.savefig(
        out_pdf,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.savefig(
        out_png,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print("Saved:", out_pdf)

# ======================================================
# Run
# ======================================================

if __name__ == "__main__":

    make_plot("CHD")

    make_plot("istroke")
    make_plot("ICVDr")

    print("\nDone.")
    print("Output:", OUT_DIR)